# Scientific Motion Studio v5
## Illustration-Grade Open Visual Intent Compiler

Versi ini tidak hanya memvalidasi bahwa SVG bisa dirender. Setiap aset dan scene diuji terhadap **benchmark illustration style** dari video referensi: kepadatan outline, kompleksitas contour, flat palette, negative space, semantic layers, dan recognizability.

Pipeline tidak membatasi jenis objek yang boleh diminta LLM. Engine memilih kombinasi data geometry, licensed/user references, generated reference (opsional), layered raster-to-vector reconstruction, open visual-program synthesis, dan scene-level composition. Scene yang masih icon-like **ditolak**, bukan diam-diam dirender.

In [ ]:
#@title 1. Runtime controls
INSTALL_LOCAL_LLM = False        #@param {type:"boolean"}
ENABLE_LOCAL_IMAGE_GEN = False   #@param {type:"boolean"}
RUN_SMOKE_TESTS = True           #@param {type:"boolean"}
RUN_ILLUSTRATION_DEMO = True     #@param {type:"boolean"}
RENDER_VIDEO = True              #@param {type:"boolean"}
FORCE_REFRESH = False            #@param {type:"boolean"}
STRICT_QUALITY_GATE = True       #@param {type:"boolean"}

TOPIC_MODE = "manual"            #@param ["manual", "keyword", "auto"]
TOPIC_VALUE = "What if Earth suddenly stopped rotating?"  #@param {type:"string"}

# Upload the 10-second benchmark MP4 to Colab, then set this path.
STYLE_REFERENCE_VIDEO = "/content/style_reference.mp4"  #@param {type:"string"}

# Match the uploaded reference. Change to 1080 x 1920 for Shorts.
OUTPUT_WIDTH = 1280   #@param {type:"integer"}
OUTPUT_HEIGHT = 720   #@param {type:"integer"}
OUTPUT_FPS = 30       #@param {type:"integer"}

# Optional image-generation model identifiers. Leave blank to use licensed/user references only.
GEMINI_IMAGE_MODEL = ""          #@param {type:"string"}
OPENAI_IMAGE_MODEL = ""          #@param {type:"string"}
LOCAL_IMAGE_MODEL = "black-forest-labs/FLUX.1-Kontext-dev"  #@param {type:"string"}

print("Controls loaded.")

In [ ]:
#@title 2. Install system and Python dependencies
import os, shutil, subprocess, sys

subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run([
    "apt-get", "-qq", "install", "-y",
    "ffmpeg", "espeak-ng", "libcairo2", "libpango-1.0-0", "npm"
], check=True)

packages = [
    "pydantic>=2.9",
    "requests>=2.32",
    "tqdm>=4.66",
    "Pillow>=10",
    "numpy<3",
    "opencv-python-headless>=4.10",
    "cairosvg>=2.7",
    "shapely>=2.0",
    "basemap>=2.0.0",
    "edge-tts>=7",
    "google-genai>=1.0",
    "openai>=1.0",
]
if INSTALL_LOCAL_LLM:
    packages += ["transformers>=4.48", "accelerate>=1.2", "bitsandbytes>=0.45", "sentencepiece"]
if ENABLE_LOCAL_IMAGE_GEN:
    packages += ["diffusers>=0.32", "accelerate>=1.2", "safetensors>=0.4", "sentencepiece", "protobuf"]  # FLUX needs T5 tokenizer deps

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("Python:", sys.version.split()[0])
print("Node:", subprocess.check_output(["node", "--version"], text=True).strip())
print("FFmpeg:", subprocess.check_output(["ffmpeg", "-version"], text=True).splitlines()[0])

In [ ]:
#@title 3. Load API keys from Colab Secrets
import os

SECRET_NAMES = ["OPENAI_API_KEY", "OPENROUTER_API_KEY", "BFL_API_KEY", "GEMINI_API_KEY", "HF_TOKEN"]
SECRETS = {}
try:
    from google.colab import userdata
    for name in SECRET_NAMES:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            SECRETS[name] = value
            os.environ[name] = value
except Exception:
    for name in SECRET_NAMES:
        if os.environ.get(name):
            SECRETS[name] = os.environ[name]

print("Available providers:", {
    "OpenAI": bool(SECRETS.get("OPENAI_API_KEY")),
    "OpenRouter (free)": bool(SECRETS.get("OPENROUTER_API_KEY")),
    "BFL FLUX Kontext": bool(SECRETS.get("BFL_API_KEY")),
    "Gemini": bool(SECRETS.get("GEMINI_API_KEY")),
    "Local LLM": INSTALL_LOCAL_LLM,
    "Local image generator": ENABLE_LOCAL_IMAGE_GEN,
})

In [ ]:
#@title 4. Upload or locate the benchmark video
from pathlib import Path

reference_path = Path(STYLE_REFERENCE_VIDEO)
if not reference_path.exists():
    print("Benchmark video not found at:", reference_path)
    print("Upload the reference MP4 now, or leave it missing to use the built-in calibrated style profiles.")
    try:
        from google.colab import files
        uploaded = files.upload()
        mp4_names = [name for name in uploaded if name.lower().endswith(".mp4")]
        if mp4_names:
            source = Path(mp4_names[0])
            target = Path("/content/style_reference.mp4")
            if source.resolve() != target.resolve():
                target.write_bytes(source.read_bytes())
            STYLE_REFERENCE_VIDEO = str(target)
            print("Benchmark video:", STYLE_REFERENCE_VIDEO)
    except Exception as exc:
        print("Interactive upload skipped:", exc)
else:
    print("Benchmark video:", reference_path)

In [ ]:
#@title 5. Workspace
from pathlib import Path

ROOT = Path("/content/scientific_motion_studio_v5")
PACKAGE_ROOT = ROOT / "src"
(PACKAGE_ROOT / "scistudio_v5").mkdir(parents=True, exist_ok=True)
(ROOT / "user_references").mkdir(parents=True, exist_ok=True)
print("Workspace:", ROOT)

In [ ]:
%%writefile /content/scientific_motion_studio_v5/src/scistudio_v5/__init__.py
# Cell 6: scistudio_v5/__init__.py
from .pipeline import StudioPipeline
from .schemas import *

__all__ = ["StudioPipeline"]


In [ ]:
%%writefile /content/scientific_motion_studio_v5/src/scistudio_v5/utils.py
# Cell 7: scistudio_v5/utils.py
from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import subprocess
import time
from pathlib import Path
from typing import Any, Iterable


def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def slugify(value: str, max_len: int = 80) -> str:
    text = re.sub(r"[^a-zA-Z0-9]+", "-", str(value).strip().lower()).strip("-")
    return (text or "untitled")[:max_len].rstrip("-")


def canonical_json(value: Any) -> str:
    if hasattr(value, "model_dump"):
        value = value.model_dump(mode="json")
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), default=str)


def hash_value(value: Any, length: int = 20) -> str:
    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()[:length]


def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def save_json(path: str | Path, value: Any) -> Path:
    path = Path(path)
    ensure_dir(path.parent)
    if hasattr(value, "model_dump"):
        value = value.model_dump(mode="json")
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    return path


def load_json(path: str | Path, default: Any = None) -> Any:
    path = Path(path)
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))


def extract_json(text: str | bytes | dict | list | None, fallback: Any = None) -> Any:
    if isinstance(text, (dict, list)):
        return text
    if text is None:
        return fallback
    if isinstance(text, bytes):
        text = text.decode("utf-8", errors="replace")
    text = str(text).strip()
    if not text:
        return fallback
    try:
        return json.loads(text)
    except Exception:
        pass

    fenced = re.findall(r"```(?:json)?\s*(.*?)```", text, flags=re.S | re.I)
    for candidate in fenced:
        try:
            return json.loads(candidate.strip())
        except Exception:
            continue

    starts = [idx for idx in (text.find("{"), text.find("[")) if idx >= 0]
    if not starts:
        return fallback
    start = min(starts)
    opening = text[start]
    closing = "}" if opening == "{" else "]"
    depth = 0
    in_string = False
    escaped = False
    for index in range(start, len(text)):
        char = text[index]
        if in_string:
            if escaped:
                escaped = False
            elif char == "\\":
                escaped = True
            elif char == '"':
                in_string = False
            continue
        if char == '"':
            in_string = True
        elif char == opening:
            depth += 1
        elif char == closing:
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:index + 1])
                except Exception:
                    break
    return fallback


def run_command(
    command: Iterable[str],
    *,
    cwd: str | Path | None = None,
    env: dict[str, str] | None = None,
    timeout: int | float | None = None,
    check: bool = True,
    capture: bool = True,
) -> subprocess.CompletedProcess:
    merged_env = os.environ.copy()
    if env:
        merged_env.update({str(k): str(v) for k, v in env.items()})
    result = subprocess.run(
        [str(part) for part in command],
        cwd=str(cwd) if cwd else None,
        env=merged_env,
        timeout=timeout,
        check=False,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.PIPE if capture else None,
    )
    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): {' '.join(map(str, command))}\n"
            f"STDOUT:\n{result.stdout or ''}\nSTDERR:\n{result.stderr or ''}"
        )
    return result


def ffprobe_duration(path: str | Path) -> float:
    path = Path(path)
    if not path.exists():
        return 0.0
    try:
        result = run_command([
            "ffprobe", "-v", "error", "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1", str(path)
        ])
        return float((result.stdout or "0").strip())
    except Exception:
        return 0.0


def copy_into(src: str | Path, dst: str | Path) -> Path:
    src, dst = Path(src), Path(dst)
    ensure_dir(dst.parent)
    shutil.copy2(src, dst)
    return dst


def retry(fn, attempts: int = 3, delay: float = 1.0, exceptions: tuple[type[Exception], ...] = (Exception,)):
    last: Exception | None = None
    for index in range(attempts):
        try:
            return fn()
        except exceptions as exc:
            last = exc
            if index + 1 < attempts:
                time.sleep(delay * (index + 1))
    if last:
        raise last
    raise RuntimeError("retry() exhausted without an exception")


In [ ]:
%%writefile /content/scientific_motion_studio_v5/src/scistudio_v5/schemas.py
# Cell 8: scistudio_v5/schemas.py
from __future__ import annotations

import json
import re
from datetime import datetime, timezone
from typing import Any

import types as _types
import typing as _typing

from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator


def _coerce_to_list(value: Any) -> list:
    """Turn LLM output into a list. Models often return a dict such as
    {"notes": [...]} or {"F01": {...}}, or a bare scalar, where a list is
    expected — flatten dict values, wrap scalars, treat None as empty."""
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, dict):
        output: list = []
        for item in value.values():
            output.extend(item if isinstance(item, list) else [item])
        return output
    return [value]


def _stringify(value: Any) -> str:
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        return " ".join(_stringify(v) for v in value)
    if isinstance(value, dict):
        return " ".join(_stringify(v) for v in value.values())
    return str(value)


def _annotation_allows(annotation: Any, target: Any) -> bool:
    """True if a field's declared type is (or optionally includes) `target`."""
    origin = _typing.get_origin(annotation)
    if origin in (_typing.Union, getattr(_types, "UnionType", None)):
        return any(_annotation_allows(arg, target) for arg in _typing.get_args(annotation))
    if target is list:
        return origin in (list, _typing.List)
    if target is str:
        return annotation is str
    return False


def _is_list_field(annotation: Any) -> bool:
    # A list field, but not a bare `Any` / object field (those accept dicts fine).
    if annotation in (Any, object, None):
        return False
    return _annotation_allows(annotation, list)


def _is_str_field(annotation: Any) -> bool:
    # Exactly str or Optional[str] — never `Any`, so free-form fields stay free.
    if annotation is str:
        return True
    origin = _typing.get_origin(annotation)
    if origin in (_typing.Union, getattr(_types, "UnionType", None)):
        args = [a for a in _typing.get_args(annotation) if a is not type(None)]
        return bool(args) and all(a is str for a in args)
    return False


class OpenModel(BaseModel):
    """Permissive model for creative-agent output.

    The engine validates only invariants required for execution. Unknown creative
    fields are retained rather than discarded, so the LLM can invent new visual
    concepts without causing schema failures.

    A single generic before-validator guards *every* subclass against the most
    common LLM shape mismatches — a dict or scalar arriving where a list is
    declared, or a list/dict arriving where a plain string is declared — so the
    pipeline never dies on a container-type error, whether the model is a
    generative artifact (research/script/storyboard) or a monitoring artifact
    (quality reports, critiques). Values already of the right shape pass through
    untouched, and fields typed `Any` stay completely free.
    """

    # validate_assignment is intentionally OFF. LLM output is validated (and
    # coerced) at the boundary — model_validate / construction — where the
    # `mode="before"` validators run exactly once. Re-running them on every
    # internal attribute assignment re-fired side-effecting validators (e.g.
    # SceneRequest.normalize_scene appending a dashboard visual on each set),
    # accumulating raw dicts in typed lists. Internal code sets correct types.
    model_config = ConfigDict(extra="allow", validate_assignment=False, arbitrary_types_allowed=True)

    @model_validator(mode="before")
    @classmethod
    def _coerce_open_types(cls, data: Any) -> Any:
        if not isinstance(data, dict):
            return data
        for name, field in cls.model_fields.items():
            if name not in data:
                continue
            value = data[name]
            annotation = field.annotation
            if _is_list_field(annotation) and not isinstance(value, list):
                data[name] = _coerce_to_list(value)
            elif _is_str_field(annotation) and isinstance(value, (list, dict)):
                data[name] = _stringify(value)
        return data


class SourceDoc(OpenModel):
    source_id: str = ""
    provider: str = "unknown"
    title: str = "Untitled source"
    url: str = ""
    snippet: str = ""
    author: str = ""
    published_at: str = ""
    authority_score: float = 0.5
    relevance_score: float = 0.0
    final_score: float = 0.0
    metadata: dict[str, Any] = Field(default_factory=dict)


class Fact(OpenModel):
    fact_id: str = ""
    claim: str = ""
    source_ids: list[str] = Field(default_factory=list)
    confidence: float = 0.6
    numeric_values: list[Any] = Field(default_factory=list)
    comparison: Any = ""
    visual_hint: Any = ""

    @field_validator("source_ids", mode="before")
    @classmethod
    def normalize_source_ids(cls, value: Any) -> list[str]:
        if value is None:
            return []
        items = value if isinstance(value, (list, tuple, set)) else [value]
        output: list[str] = []
        for item in items:
            if isinstance(item, dict):
                text = str(item.get("source_id", item.get("id", item.get("value", "")))).strip()
            else:
                text = str(item).strip()
            if text and text not in output:
                output.append(text)
        return output

    @field_validator("numeric_values", mode="before")
    @classmethod
    def normalize_numeric_values(cls, value: Any) -> list[Any]:
        # Deliberately preserve structured values. Formatting happens at display time.
        if value is None:
            return []
        return list(value) if isinstance(value, (list, tuple, set)) else [value]


class ResearchPack(OpenModel):
    # List fields are auto-coerced by OpenModel._coerce_open_types.
    topic: str
    summary: str = ""
    hooks: list[Any] = Field(default_factory=list)
    facts: list[Fact] = Field(default_factory=list)
    comparisons: list[Any] = Field(default_factory=list)
    visual_ideas: list[Any] = Field(default_factory=list)
    limitations: list[Any] = Field(default_factory=list)
    sources: list[SourceDoc] = Field(default_factory=list)
    validation_score: float = 0.0
    research_hash: str = ""
    raw_llm_output: Any = None


class Beat(OpenModel):
    beat_id: str = ""
    purpose: Any = "information_gain"
    duration_s: float = 5.0
    spoken_line: str = ""
    visual_event: Any = ""
    evidence_refs: list[str] = Field(default_factory=list)
    emotion: Any = "curiosity"
    retention_function: Any = "information_gain"
    emphasis_words: list[Any] = Field(default_factory=list)
    pause_after_ms: int = 80
    sfx: Any = "none"
    raw: dict[str, Any] = Field(default_factory=dict)

    @field_validator("duration_s", mode="before")
    @classmethod
    def duration_to_float(cls, value: Any) -> float:
        if isinstance(value, dict):
            value = value.get("seconds", value.get("value", 5.0))
        try:
            return max(0.25, float(value))
        except Exception:
            return 5.0

    @field_validator("evidence_refs", mode="before")
    @classmethod
    def listify_refs(cls, value: Any) -> list[str]:
        if value is None:
            return []
        items = value if isinstance(value, list) else [value]
        return [str(item) for item in items if str(item).strip()]


class ScriptPackage(OpenModel):
    topic: str
    title: str = ""
    hook: str = ""
    beats: list[Beat] = Field(default_factory=list)
    closing: str = ""
    total_words: int = 0
    estimated_duration_s: float = 0.0
    script_hash: str = ""
    raw_llm_output: Any = None


class VisualIntent(OpenModel):
    """Free-form creative request for one visual entity or system."""

    intent_id: str = ""
    description: str = ""
    role: Any = "support"
    kind: Any = "unspecified"
    attributes: dict[str, Any] = Field(default_factory=dict)
    relationships: list[Any] = Field(default_factory=list)
    desired_layers: list[Any] = Field(default_factory=list)
    motion_intents: list[Any] = Field(default_factory=list)
    reference_requests: list[Any] = Field(default_factory=list)
    visual_constraints: dict[str, Any] = Field(default_factory=dict)
    style: Any = None
    raw_llm_output: Any = None

    @model_validator(mode="before")
    @classmethod
    def accept_any_visual(cls, value: Any) -> Any:
        if isinstance(value, cls):
            return value
        if isinstance(value, str):
            return {"description": value, "kind": value, "raw_llm_output": value}
        if not isinstance(value, dict):
            return {"description": str(value), "raw_llm_output": value}
        data = dict(value)
        data.setdefault(
            "description",
            str(data.get("prompt", data.get("asset", data.get("name", data.get("kind", "visual"))))),
        )
        data.setdefault("kind", data.get("type", data.get("asset_class", "unspecified")))
        data.setdefault("intent_id", str(data.get("id", data.get("asset_id", ""))))
        data.setdefault("desired_layers", data.get("layers", data.get("semantic_layers", [])))
        data.setdefault("motion_intents", data.get("motions", data.get("animation", [])))
        data.setdefault("reference_requests", data.get("references", data.get("search_queries", [])))
        for key in ("attributes", "visual_constraints"):
            current = data.get(key)
            if current is None:
                data[key] = {}
            elif not isinstance(current, dict):
                data[key] = {"value": current}
        for key in ("relationships", "desired_layers", "motion_intents", "reference_requests"):
            current = data.get(key)
            if current is None:
                data[key] = []
            elif not isinstance(current, list):
                data[key] = [current]
        data.setdefault("raw_llm_output", value)
        return data


class MotionIntent(OpenModel):
    target: Any = "root"
    operation: Any = "hold"
    parameters: dict[str, Any] = Field(default_factory=dict)
    timing: dict[str, Any] = Field(default_factory=dict)
    physical_behavior: dict[str, Any] = Field(default_factory=dict)
    raw_llm_output: Any = None

    @model_validator(mode="before")
    @classmethod
    def accept_any_motion(cls, value: Any) -> Any:
        if isinstance(value, cls):
            return value
        if isinstance(value, str):
            return {"operation": value, "raw_llm_output": value}
        if not isinstance(value, dict):
            return {"operation": str(value), "raw_llm_output": value}
        data = dict(value)
        data.setdefault("operation", data.get("primitive", data.get("action", data.get("motion", "hold"))))
        data.setdefault("target", data.get("target_layer_id", data.get("target_asset_id", "root")))
        data.setdefault("parameters", {})
        for key in (
            "from_value", "to_value", "easing", "loop", "amplitude", "frequency",
            "direction", "distance", "angle", "speed", "delay", "stagger",
        ):
            if key in data and key not in data["parameters"]:
                data["parameters"][key] = data[key]
        data.setdefault("timing", {
            "start_s": data.get("start_s", data.get("start", 0)),
            "end_s": data.get("end_s", data.get("end", 1)),
        })
        for key in ("parameters", "timing", "physical_behavior"):
            current = data.get(key)
            if current is None:
                data[key] = {}
            elif not isinstance(current, dict):
                data[key] = {"value": current}
        data.setdefault("raw_llm_output", value)
        return data


class SceneRequest(OpenModel):
    scene_id: str = ""
    beat_id: str = ""
    duration_s: float = 5.0
    narration: str = ""
    headline: str = ""
    visual_event: Any = ""
    background: Any = "paper"
    visuals: list[VisualIntent] = Field(default_factory=list)
    motions: list[MotionIntent] = Field(default_factory=list)
    layout: Any = Field(default_factory=dict)
    camera: Any = Field(default_factory=dict)
    dashboard: Any = None
    transition: Any = "cut"
    raw_llm_output: Any = None

    @model_validator(mode="before")
    @classmethod
    def normalize_scene(cls, value: Any) -> Any:
        if not isinstance(value, dict):
            value = {"visual_event": value}
        data = dict(value)
        if isinstance(data.get("scene"), dict):
            nested = dict(data.pop("scene"))
            nested.update({k: v for k, v in data.items() if k not in nested})
            data = nested
        aliases = {
            "duration": "duration_s",
            "voiceover": "narration",
            "voice_over": "narration",
            "spoken_line": "narration",
            "title": "headline",
            "visual": "visual_event",
            "assets": "visuals",
            "objects": "visuals",
            "animation": "motions",
        }
        for src, dst in aliases.items():
            if src in data and dst not in data:
                data[dst] = data[src]
        visuals = data.get("visuals", [])
        if not isinstance(visuals, list):
            visuals = [visuals]
        # A string dashboard is a creative visual intent, not a schema error.
        # Idempotent: never add a second dashboard visual if one already exists
        # (guards against re-validation of already-normalized data).
        def _role_of(item: Any) -> str:
            if isinstance(item, dict):
                return str(item.get("role", ""))
            return str(getattr(item, "role", ""))
        dashboard = data.get("dashboard")
        has_dashboard_visual = any(_role_of(v) == "dashboard" for v in visuals)
        if dashboard not in (None, "", {}, []) and not has_dashboard_visual:
            visuals.append({
                "description": dashboard if isinstance(dashboard, str) else json.dumps(dashboard, ensure_ascii=False),
                "kind": "scientific dashboard",
                "role": "dashboard",
                "attributes": dashboard if isinstance(dashboard, dict) else {"requested_form": dashboard},
                "raw_llm_output": dashboard,
            })
        data["visuals"] = visuals
        motions = data.get("motions", [])
        data["motions"] = motions if isinstance(motions, list) else [motions]
        data.setdefault("raw_llm_output", value)
        return data

    @field_validator("duration_s", mode="before")
    @classmethod
    def duration_to_float(cls, value: Any) -> float:
        if isinstance(value, dict):
            value = value.get("seconds", value.get("value", 5.0))
        try:
            return max(0.25, float(value))
        except Exception:
            return 5.0


class Storyboard(OpenModel):
    topic: str
    width: int = 1080
    height: int = 1920
    fps: int = 30
    scenes: list[SceneRequest] = Field(default_factory=list)
    estimated_duration_s: float = 0.0
    storyboard_hash: str = ""
    raw_llm_output: Any = None

    @model_validator(mode="before")
    @classmethod
    def normalize_container(cls, value: Any) -> Any:
        if not isinstance(value, dict):
            return {"topic": "Untitled", "scenes": [value], "raw_llm_output": value}
        original = value
        data = dict(value)
        wrapped = data.get("storyboard")
        if isinstance(wrapped, dict):
            outer = {k: v for k, v in data.items() if k != "storyboard"}
            data = {**outer, **wrapped}
        elif isinstance(wrapped, list):
            data["scenes"] = wrapped

        scenes = data.get("scenes")
        if isinstance(scenes, dict):
            def natural_key(key: Any) -> tuple[int, str]:
                match = re.search(r"(\d+)", str(key))
                return (int(match.group(1)) if match else 10**9, str(key))
            scenes = [scenes[key] for key in sorted(scenes, key=natural_key)]
        if scenes is None:
            candidates = []
            for key, item in data.items():
                if isinstance(item, dict) and re.match(r"^(scene|s)[_-]?\d+", str(key), flags=re.I):
                    candidates.append((key, item))
            if candidates:
                scenes = [item for _, item in sorted(candidates, key=lambda pair: pair[0])]
        if not isinstance(scenes, list):
            scenes = [scenes] if scenes is not None else []
        data["scenes"] = scenes
        data.setdefault("topic", str(data.get("title", "Untitled")))
        data.setdefault("raw_llm_output", original)
        return data


class BuildPlan(OpenModel):
    plan_id: str = ""
    intent_id: str = ""
    strategy: Any = "hybrid"
    rationale: Any = ""
    reference_queries: list[Any] = Field(default_factory=list)
    data_requests: list[Any] = Field(default_factory=list)
    visual_program: Any = Field(default_factory=dict)
    layer_plan: list[Any] = Field(default_factory=list)
    animation_plan: list[Any] = Field(default_factory=list)
    acceptance_criteria: Any = Field(default_factory=dict)
    raw_llm_output: Any = None

    @model_validator(mode="before")
    @classmethod
    def normalize_plan(cls, value: Any) -> Any:
        if not isinstance(value, dict):
            return {"strategy": value, "raw_llm_output": value}
        data = dict(value)
        for key in ("reference_queries", "data_requests", "layer_plan", "animation_plan"):
            current = data.get(key)
            if current is None:
                data[key] = []
            elif not isinstance(current, list):
                data[key] = [current]
        if data.get("visual_program") is None:
            data["visual_program"] = {}
        if not isinstance(data.get("acceptance_criteria", {}), dict):
            data["acceptance_criteria"] = {"value": data.get("acceptance_criteria")}
        data.setdefault("raw_llm_output", value)
        return data


class AssetRecord(OpenModel):
    asset_id: str
    intent_id: str = ""
    description: str = ""
    method: Any = ""
    svg_path: str
    preview_path: str = ""
    manifest_path: str = ""
    source_urls: list[str] = Field(default_factory=list)
    licenses: list[str] = Field(default_factory=list)
    semantic_layers: list[dict[str, Any]] = Field(default_factory=list)
    quality: dict[str, Any] = Field(default_factory=dict)
    warnings: list[str] = Field(default_factory=list)
    created_at: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    asset_hash: str = ""
    raw_build_plan: Any = None


class MotionTrack(OpenModel):
    target: str = "root"
    property: str = "opacity"
    keyframes: list[dict[str, Any]] = Field(default_factory=list)
    easing: Any = "linear"
    loop: bool = False
    transform_origin: Any = "center"
    raw_motion_intent: Any = None


class CompiledScene(OpenModel):
    scene_id: str
    beat_id: str = ""
    duration_s: float = 5.0
    narration: str = ""
    headline: str = ""
    background: Any = "paper"
    assets: list[dict[str, Any]] = Field(default_factory=list)
    motion_tracks: list[MotionTrack] = Field(default_factory=list)
    camera_tracks: list[MotionTrack] = Field(default_factory=list)
    transition: Any = "cut"
    raw_scene: Any = None


class QualityReport(OpenModel):
    passed: bool = False
    scores: dict[str, float] = Field(default_factory=dict)
    warnings: list[str] = Field(default_factory=list)
    failures: list[str] = Field(default_factory=list)
    repair_plan: list[Any] = Field(default_factory=list)
    media: dict[str, Any] = Field(default_factory=dict)


def format_numeric_value(value: Any) -> str:
    if isinstance(value, str):
        return value
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return f"{value:g}" if isinstance(value, float) else str(value)
    if isinstance(value, dict):
        amount = value.get("value", value.get("amount", value.get("number", "")))
        unit = value.get("unit", value.get("units", ""))
        context = value.get("context", value.get("label", value.get("description", "")))
        main = " ".join(str(x).strip() for x in (amount, unit) if str(x).strip())
        return f"{main} ({context})" if main and str(context).strip() else main or json.dumps(value, ensure_ascii=False)
    return str(value)


In [ ]:
%%writefile /content/scientific_motion_studio_v5/src/scistudio_v5/llm.py
# Cell 9: scistudio_v5/llm.py
from __future__ import annotations

import base64
import json
import os
from pathlib import Path
from typing import Any

from .utils import ensure_dir, extract_json, hash_value, load_json, save_json, sha256_file


class LLMRouter:
    """Lazy, provider-agnostic creative router.

    Provider order is configurable. Local checkpoints are never loaded when a
    remote provider succeeds, preventing the unnecessary 15-20 GB download that
    occurred in earlier prototypes.
    """

    def __init__(self, config: dict[str, Any], secrets: dict[str, Any], cache_root: str | Path):
        self.config = config
        self.secrets = secrets
        self.cache_root = ensure_dir(cache_root)
        self._local = None
        self._local_tokenizer = None
        self._gemini_client = None
        self._openai_client = None
        self._openrouter_client = None

    @property
    def provider_order(self) -> list[str]:
        # Default priority: OpenAI -> local Qwen -> OpenRouter (free).
        order = self.config.get("provider_order", ["openai", "local", "openrouter"])
        if isinstance(order, str):
            order = [item.strip() for item in order.split(",") if item.strip()]
        return list(order)

    def _secret(self, *names: str) -> str:
        for name in names:
            value = self.secrets.get(name) or os.environ.get(name)
            if value:
                return value
        return ""

    def available(self, provider: str) -> bool:
        provider = provider.lower()
        if provider == "gemini":
            return bool(self._secret("GEMINI_API_KEY"))
        if provider == "openai":
            return bool(self._secret("OPENAI_API_KEY"))
        if provider == "openrouter":
            return bool(self._secret("OPENROUTER_API_KEY"))
        if provider == "local":
            return bool(self.config.get("enable_local_fallback", False))
        return False

    def generate_json(
        self,
        *,
        system: str,
        prompt: str,
        namespace: str,
        fallback: Any,
        json_schema: dict[str, Any] | None = None,
        force: bool = False,
        temperature: float | None = None,
    ) -> Any:
        key = hash_value({"system": system, "prompt": prompt, "schema": json_schema, "v": 4})
        cache_path = self.cache_root / namespace / f"{key}.json"
        if cache_path.exists() and not force:
            return load_json(cache_path, fallback)

        errors: list[str] = []
        for provider in self.provider_order:
            if not self.available(provider):
                continue
            try:
                if provider == "gemini":
                    output = self._gemini_json(system, prompt, json_schema, temperature)
                elif provider == "openai":
                    output = self._openai_json(system, prompt, temperature)
                elif provider == "openrouter":
                    output = self._openrouter_json(system, prompt, temperature)
                elif provider == "local":
                    output = self._local_json(system, prompt, temperature)
                else:
                    continue
                parsed = extract_json(output, fallback=None)
                if parsed is not None:
                    save_json(cache_path, parsed)
                    return parsed
                errors.append(f"{provider}: returned non-JSON output")
            except Exception as exc:
                errors.append(f"{provider}: {type(exc).__name__}: {exc}")

        value = fallback() if callable(fallback) else fallback
        save_json(cache_path, value)
        if errors:
            print("[LLMRouter] Providers failed; using fallback. " + " | ".join(errors[-3:]))
        return value

    def critique_image(
        self,
        *,
        image_path: str | Path,
        prompt: str,
        namespace: str = "vision_critic",
        fallback: Any = None,
        force: bool = False,
    ) -> Any:
        image_path = Path(image_path)
        key = hash_value({"image": image_path.name, "size": image_path.stat().st_size, "prompt": prompt, "v": 4})
        cache_path = self.cache_root / namespace / f"{key}.json"
        if cache_path.exists() and not force:
            return load_json(cache_path, fallback)

        errors: list[str] = []
        # Vision can use a dedicated provider order (e.g. OpenAI-only vision control).
        vision_order = self.config.get("vision_provider_order") or self.provider_order
        if isinstance(vision_order, str):
            vision_order = [item.strip() for item in vision_order.split(",") if item.strip()]
        for provider in vision_order:
            if provider not in {"gemini", "openai", "openrouter"} or not self.available(provider):
                continue
            try:
                if provider == "gemini":
                    output = self._gemini_vision(image_path, prompt)
                elif provider == "openrouter":
                    output = self._openrouter_vision(image_path, prompt)
                else:
                    output = self._openai_vision(image_path, prompt)
                parsed = extract_json(output, fallback=None)
                if parsed is not None:
                    save_json(cache_path, parsed)
                    return parsed
                errors.append(f"{provider}: non-JSON vision response")
            except Exception as exc:
                errors.append(f"{provider}: {type(exc).__name__}: {exc}")
        if errors:
            print("[LLMRouter] Vision critic fallback. " + " | ".join(errors[-2:]))
        return fallback() if callable(fallback) else fallback


    def generate_reference_image(
        self,
        *,
        prompt: str,
        output_path: str | Path,
        init_image: str | Path | None = None,
        force: bool = False,
    ) -> Path | None:
        """Generate a raster construction reference when configured.

        Provider order: Black Forest Labs FLUX.1 Kontext [pro] API (fast, no
        multi-GB download) -> Gemini/Imagen -> OpenAI images -> local diffusers.
        If none is configured the engine continues with licensed/user references
        and open visual-program synthesis.
        """
        output_path = Path(output_path)
        ensure_dir(output_path.parent)
        if output_path.exists() and output_path.stat().st_size > 1024 and not force:
            return output_path
        errors = []
        # Black Forest Labs FLUX.1 Kontext [pro] via API (preferred image path).
        if self.config.get("image_provider", "bfl") == "bfl" and self.available_bfl():
            bfl = self._bfl_flux_image(prompt, output_path, init_image=init_image, force=force)
            if bfl is not None:
                return bfl
            errors.append("bfl: request failed")
        # Gemini/Imagen-style provider, only when an explicit image model is configured.
        gemini_model = self.config.get("gemini_image_model")
        if gemini_model and self.available("gemini"):
            try:
                from google.genai import types
                response = self._gemini().models.generate_images(
                    model=gemini_model,
                    prompt=prompt,
                    config=types.GenerateImagesConfig(number_of_images=1),
                )
                generated = getattr(response, "generated_images", None) or []
                if generated:
                    image = getattr(generated[0], "image", generated[0])
                    data = getattr(image, "image_bytes", None) or getattr(image, "bytes", None)
                    if data:
                        output_path.write_bytes(data)
                        return output_path
            except Exception as exc:
                errors.append(f"gemini image: {type(exc).__name__}: {exc}")
        # OpenAI image provider, only when an explicit image model is configured.
        openai_model = self.config.get("openai_image_model")
        if openai_model and self.available("openai"):
            try:
                image_kwargs = {
                    "model": openai_model, "prompt": prompt,
                    "size": self.config.get("openai_image_size", "1024x1024"),
                }
                # `response_format` is only valid for the dall-e models; gpt-image-1
                # rejects it and always returns b64_json.
                if "dall-e" in str(openai_model).lower():
                    image_kwargs["response_format"] = "b64_json"
                response = self._openai().images.generate(**image_kwargs)
                item = response.data[0]
                encoded = getattr(item, "b64_json", None)
                if encoded:
                    output_path.write_bytes(base64.b64decode(encoded))
                    return output_path
                url = getattr(item, "url", None)
                if url:
                    import requests
                    output_path.write_bytes(requests.get(url, timeout=120).content)
                    return output_path
            except Exception as exc:
                errors.append(f"openai image: {type(exc).__name__}: {exc}")
        # Optional local diffusion model for A100/other CUDA runtimes.
        local_model = self.config.get("local_image_model")
        if local_model and self.config.get("enable_local_image_generation", False):
            # FLUX.1 [Kontext] / [dev] path. Kontext is an instruction image
            # editor, so if an init image (a procedural/data render) is provided
            # it is restyled into a flat editorial illustration; otherwise the
            # text-to-image FLUX pipeline is used. Needs a large-VRAM GPU (A100).
            if "flux" in str(local_model).lower() or "kontext" in str(local_model).lower():
                flux = self._flux_image(prompt, output_path, local_model)
                if flux is not None:
                    return flux
                errors.append("flux image: pipeline unavailable or failed")
            try:
                import torch
                from diffusers import AutoPipelineForText2Image
                pipe = AutoPipelineForText2Image.from_pretrained(
                    local_model, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                    use_safetensors=True,
                )
                if torch.cuda.is_available():
                    pipe.enable_model_cpu_offload()
                image = pipe(
                    prompt=prompt,
                    num_inference_steps=int(self.config.get("local_image_steps", 24)),
                    guidance_scale=float(self.config.get("local_image_guidance", 6.5)),
                    height=int(self.config.get("local_image_height", 1024)),
                    width=int(self.config.get("local_image_width", 1024)),
                ).images[0]
                image.save(output_path)
                del pipe
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                return output_path
            except Exception as exc:
                errors.append(f"local image: {type(exc).__name__}: {exc}")
        if errors:
            print("[LLMRouter] Reference image generation unavailable. " + " | ".join(errors[-3:]))
        return None

    def available_bfl(self) -> bool:
        return bool(self._secret("BFL_API_KEY", "BFL_KEY"))

    def _bfl_flux_image(
        self,
        prompt: str,
        output_path: Path,
        *,
        init_image: str | Path | None = None,
        force: bool = False,
    ) -> Path | None:
        """FLUX.1 Kontext [pro] via the Black Forest Labs API (submit -> poll ->
        download). Content-addressed cache: identical (prompt, seed image, model,
        params) is never re-requested — the cached PNG is copied out. This keeps
        API spend and latency minimal while staying deterministic per input."""
        import shutil
        import time

        import requests

        model = self.config.get("bfl_model", "flux-kontext-pro")
        base = self.config.get("bfl_base_url", "https://api.bfl.ai/v1")
        api_key = self._secret("BFL_API_KEY", "BFL_KEY")
        init_path = Path(init_image) if init_image else None
        init_hash = sha256_file(init_path) if init_path and init_path.exists() else ""
        aspect = self.config.get("bfl_aspect_ratio", "16:9")

        # Content-addressed cache.
        cache_dir = ensure_dir(self.cache_root / "bfl_images")
        key = hash_value({"model": model, "prompt": prompt, "init": init_hash, "aspect": aspect, "v": 1})
        cached = cache_dir / f"{key}.png"
        if cached.exists() and cached.stat().st_size > 1024 and not force:
            shutil.copy2(cached, output_path)
            return output_path

        payload: dict[str, Any] = {"prompt": prompt, "output_format": "png", "aspect_ratio": aspect}
        if self.config.get("bfl_seed") is not None:
            payload["seed"] = int(self.config.get("bfl_seed"))
        # Kontext restyles a seed image when one is supplied (instruction editing);
        # otherwise it runs as text-to-image.
        if init_path and init_path.exists():
            payload["input_image"] = base64.b64encode(init_path.read_bytes()).decode("ascii")

        headers = {"x-key": api_key, "Content-Type": "application/json", "accept": "application/json"}
        try:
            submit = requests.post(f"{base}/{model}", headers=headers, json=payload, timeout=60)
            submit.raise_for_status()
            data = submit.json()
            polling_url = data.get("polling_url") or f"{base}/get_result"
            request_id = data.get("id")
            deadline = time.time() + float(self.config.get("bfl_timeout", 180))
            sample_url = None
            while time.time() < deadline:
                params = None if data.get("polling_url") else {"id": request_id}
                result = requests.get(polling_url, headers={"x-key": api_key, "accept": "application/json"}, params=params, timeout=30)
                result.raise_for_status()
                body = result.json()
                status = str(body.get("status", ""))
                if status == "Ready":
                    sample_url = (body.get("result") or {}).get("sample")
                    break
                if status in {"Error", "Failed", "Content Moderated", "Request Moderated"}:
                    print(f"[LLMRouter] BFL status: {status}")
                    return None
                time.sleep(float(self.config.get("bfl_poll_interval", 1.5)))
            if not sample_url:
                print("[LLMRouter] BFL timed out before image was ready.")
                return None
            image_bytes = requests.get(sample_url, timeout=120).content
            cached.write_bytes(image_bytes)
            shutil.copy2(cached, output_path)
            return output_path
        except Exception as exc:
            print(f"[LLMRouter] BFL FLUX Kontext request failed: {type(exc).__name__}: {exc}")
            return None

    def _flux_image(self, prompt: str, output_path: Path, model: str) -> Path | None:
        """FLUX.1 [Kontext-dev] / [dev] generator. Kontext restyles an optional
        init image (config['flux_init_image']) with the prompt as an edit
        instruction; without one it falls back to text-to-image FLUX."""
        try:
            import torch
            dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
            steps = int(self.config.get("flux_steps", self.config.get("local_image_steps", 28)))
            guidance = float(self.config.get("flux_guidance", 2.5))
            token = self.secrets.get("HF_TOKEN") or os.environ.get("HF_TOKEN") or None
            init_path = self.config.get("flux_init_image")
            is_kontext = "kontext" in str(model).lower()
            if is_kontext and init_path and Path(init_path).exists():
                from diffusers import FluxKontextPipeline
                from diffusers.utils import load_image
                pipe = FluxKontextPipeline.from_pretrained(model, torch_dtype=dtype, token=token)
                if torch.cuda.is_available():
                    pipe.enable_model_cpu_offload()
                image = pipe(
                    image=load_image(str(init_path)), prompt=prompt,
                    guidance_scale=guidance, num_inference_steps=steps,
                ).images[0]
            else:
                from diffusers import FluxPipeline
                base = "black-forest-labs/FLUX.1-dev" if is_kontext else model
                pipe = FluxPipeline.from_pretrained(base, torch_dtype=dtype, token=token)
                if torch.cuda.is_available():
                    pipe.enable_model_cpu_offload()
                image = pipe(
                    prompt=prompt, guidance_scale=guidance, num_inference_steps=steps,
                    height=int(self.config.get("local_image_height", 1024)),
                    width=int(self.config.get("local_image_width", 1024)),
                ).images[0]
            image.save(output_path)
            del pipe
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return output_path
        except Exception as exc:
            print(f"[LLMRouter] FLUX image generation failed: {type(exc).__name__}: {exc}")
            return None

    def _gemini(self):
        if self._gemini_client is None:
            from google import genai
            api_key = self.secrets.get("GEMINI_API_KEY") or os.environ.get("GEMINI_API_KEY")
            self._gemini_client = genai.Client(api_key=api_key)
        return self._gemini_client

    def _gemini_json(self, system: str, prompt: str, schema: dict[str, Any] | None, temperature: float | None):
        from google.genai import types
        model = self.config.get("gemini_model", "gemini-2.5-flash")
        kwargs: dict[str, Any] = {
            "system_instruction": system,
            "temperature": self.config.get("temperature", 0.75) if temperature is None else temperature,
            "response_mime_type": "application/json",
        }
        if schema:
            kwargs["response_json_schema"] = schema
        response = self._gemini().models.generate_content(
            model=model,
            contents=prompt,
            config=types.GenerateContentConfig(**kwargs),
        )
        return response.text

    def _gemini_vision(self, image_path: Path, prompt: str):
        from google.genai import types
        model = self.config.get("gemini_vision_model", self.config.get("gemini_model", "gemini-2.5-flash"))
        mime = "image/png" if image_path.suffix.lower() == ".png" else "image/jpeg"
        response = self._gemini().models.generate_content(
            model=model,
            contents=[
                types.Part.from_bytes(data=image_path.read_bytes(), mime_type=mime),
                prompt,
            ],
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                temperature=0.25,
            ),
        )
        return response.text

    def _openai(self):
        if self._openai_client is None:
            from openai import OpenAI
            api_key = self.secrets.get("OPENAI_API_KEY") or os.environ.get("OPENAI_API_KEY")
            self._openai_client = OpenAI(api_key=api_key)
        return self._openai_client

    def _openai_json(self, system: str, prompt: str, temperature: float | None):
        model = self.config.get("openai_model", "gpt-5-mini")
        response = self._openai().responses.create(
            model=model,
            input=[
                {"role": "system", "content": [{"type": "input_text", "text": system}]},
                {"role": "user", "content": [{"type": "input_text", "text": prompt + "\nReturn valid JSON only."}]},
            ],
        )
        return response.output_text

    def _openai_vision(self, image_path: Path, prompt: str):
        model = self.config.get("openai_vision_model", self.config.get("openai_model", "gpt-5-mini"))
        mime = "image/png" if image_path.suffix.lower() == ".png" else "image/jpeg"
        data_url = f"data:{mime};base64,{base64.b64encode(image_path.read_bytes()).decode('ascii')}"
        response = self._openai().responses.create(
            model=model,
            input=[{
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt + "\nReturn valid JSON only."},
                    {"type": "input_image", "image_url": data_url},
                ],
            }],
        )
        return response.output_text

    def _openrouter(self):
        # OpenRouter is OpenAI-API compatible; reuse the OpenAI SDK with its base_url.
        if getattr(self, "_openrouter_client", None) is None:
            from openai import OpenAI
            self._openrouter_client = OpenAI(
                api_key=self._secret("OPENROUTER_API_KEY"),
                base_url=self.config.get("openrouter_base_url", "https://openrouter.ai/api/v1"),
                default_headers={
                    "HTTP-Referer": self.config.get("openrouter_referer", "https://scientific-motion-studio.local"),
                    "X-Title": "Scientific Motion Studio v5",
                },
            )
        return self._openrouter_client

    def _openrouter_json(self, system: str, prompt: str, temperature: float | None):
        model = self.config.get("openrouter_model", "meta-llama/llama-3.3-70b-instruct:free")
        response = self._openrouter().chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": prompt + "\nReturn valid JSON only, no Markdown."},
            ],
            temperature=self.config.get("temperature", 0.7) if temperature is None else temperature,
            response_format={"type": "json_object"},
        )
        return response.choices[0].message.content

    def _openrouter_vision(self, image_path: Path, prompt: str):
        model = self.config.get("openrouter_vision_model", "meta-llama/llama-3.2-11b-vision-instruct:free")
        mime = "image/png" if image_path.suffix.lower() == ".png" else "image/jpeg"
        data_url = f"data:{mime};base64,{base64.b64encode(image_path.read_bytes()).decode('ascii')}"
        response = self._openrouter().chat.completions.create(
            model=model,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt + "\nReturn valid JSON only."},
                    {"type": "image_url", "image_url": {"url": data_url}},
                ],
            }],
        )
        return response.choices[0].message.content

    def _load_local(self):
        if self._local is not None:
            return self._local_tokenizer, self._local
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

        model_name = self.config.get("local_model", "Qwen/Qwen2.5-3B-Instruct")
        token = self.secrets.get("HF_TOKEN") or os.environ.get("HF_TOKEN") or None
        quantization_config = None
        if torch.cuda.is_available() and self.config.get("local_4bit", True):
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
            )
        tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            token=token,
            device_map="auto" if torch.cuda.is_available() else None,
            torch_dtype="auto",
            quantization_config=quantization_config,
            low_cpu_mem_usage=True,
        )
        self._local_tokenizer, self._local = tokenizer, model
        return tokenizer, model

    def _local_json(self, system: str, prompt: str, temperature: float | None):
        import torch
        tokenizer, model = self._load_local()
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": prompt + "\nReturn valid JSON only, without Markdown."},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt")
        device = next(model.parameters()).device
        inputs = {key: value.to(device) for key, value in inputs.items()}
        with torch.inference_mode():
            generated = model.generate(
                **inputs,
                max_new_tokens=int(self.config.get("local_max_new_tokens", 2600)),
                do_sample=True,
                temperature=float(self.config.get("temperature", 0.7) if temperature is None else temperature),
                top_p=0.92,
                repetition_penalty=1.04,
            )
        output = generated[0, inputs["input_ids"].shape[1]:]
        return tokenizer.decode(output, skip_special_tokens=True)


In [ ]:
%%writefile /content/scientific_motion_studio_v5/src/scistudio_v5/research.py
# Cell 10: scistudio_v5/research.py
from __future__ import annotations

import html
import json
import math
import re
from pathlib import Path
from typing import Any

import requests

from .llm import LLMRouter
from .schemas import Fact, ResearchPack, SourceDoc
from .utils import ensure_dir, hash_value, load_json, save_json


class PublicResearchSearch:
    def __init__(self, config: dict[str, Any], cache_root: str | Path):
        self.config = config
        self.cache_root = ensure_dir(cache_root)
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": "ScientificMotionStudio/4.0 research@example.invalid"})
        self.timeout = int(config.get("timeout", 25))

    def search(self, topic: str, force: bool = False) -> list[SourceDoc]:
        cache_path = self.cache_root / f"search-{hash_value(topic)}.json"
        if cache_path.exists() and not force:
            return [SourceDoc.model_validate(x) for x in load_json(cache_path, [])]
        sources: list[SourceDoc] = []
        for fn in (self._wikipedia, self._crossref, self._openalex, self._arxiv):
            try:
                sources.extend(fn(topic))
            except Exception as exc:
                print(f"[research] {fn.__name__} skipped: {exc}")
        dedup: dict[str, SourceDoc] = {}
        for item in sources:
            key = (item.url or item.title).strip().lower()
            if key and key not in dedup:
                dedup[key] = item
        ranked = list(dedup.values())
        words = set(re.findall(r"[a-z0-9]+", topic.lower()))
        for item in ranked:
            text_words = set(re.findall(r"[a-z0-9]+", f"{item.title} {item.snippet}".lower()))
            overlap = len(words & text_words) / max(1, len(words))
            item.relevance_score = min(1.0, overlap)
            item.final_score = 0.6 * item.authority_score + 0.4 * item.relevance_score
        ranked.sort(key=lambda x: x.final_score, reverse=True)
        ranked = ranked[: int(self.config.get("max_sources", 24))]
        for index, item in enumerate(ranked, 1):
            item.source_id = f"S{index:02d}"
        save_json(cache_path, ranked)
        return ranked

    def _wikipedia(self, topic: str) -> list[SourceDoc]:
        endpoint = "https://en.wikipedia.org/w/api.php"
        payload = self.session.get(endpoint, params={
            "action": "query", "generator": "search", "gsrsearch": topic,
            "gsrlimit": 5, "prop": "extracts|info", "exintro": 1,
            "explaintext": 1, "inprop": "url", "format": "json",
        }, timeout=self.timeout).json()
        output = []
        for page in payload.get("query", {}).get("pages", {}).values():
            output.append(SourceDoc(
                provider="wikipedia", title=page.get("title", "Wikipedia"),
                url=page.get("fullurl", ""), snippet=page.get("extract", "")[:1600],
                authority_score=0.72,
            ))
        return output

    def _crossref(self, topic: str) -> list[SourceDoc]:
        payload = self.session.get("https://api.crossref.org/works", params={
            "query": topic, "rows": 6, "select": "DOI,title,author,published,URL,abstract,publisher,type"
        }, timeout=self.timeout).json()
        output = []
        for item in payload.get("message", {}).get("items", []):
            title = " ".join(item.get("title") or ["Untitled"])
            authors = item.get("author") or []
            author = ", ".join(" ".join(filter(None, [a.get("given", ""), a.get("family", "")])) for a in authors[:3])
            abstract = re.sub(r"<[^>]+>", " ", item.get("abstract", ""))
            output.append(SourceDoc(
                provider="crossref", title=title, url=item.get("URL", ""), author=author,
                snippet=html.unescape(abstract)[:1800], authority_score=0.86,
                metadata={"doi": item.get("DOI", ""), "publisher": item.get("publisher", "")},
            ))
        return output

    def _openalex(self, topic: str) -> list[SourceDoc]:
        payload = self.session.get("https://api.openalex.org/works", params={
            "search": topic, "per-page": 6, "mailto": "research@example.invalid"
        }, timeout=self.timeout).json()
        output = []
        for item in payload.get("results", []):
            inverted = item.get("abstract_inverted_index") or {}
            words = sorted(((pos, word) for word, positions in inverted.items() for pos in positions), key=lambda x: x[0])
            abstract = " ".join(word for _, word in words)
            source = ((item.get("primary_location") or {}).get("source") or {}).get("display_name", "")
            output.append(SourceDoc(
                provider="openalex", title=item.get("display_name", "Untitled"),
                url=item.get("doi") or item.get("id", ""), snippet=abstract[:1800],
                authority_score=0.88, published_at=str(item.get("publication_year", "")),
                metadata={"cited_by_count": item.get("cited_by_count", 0), "venue": source},
            ))
        return output

    def _arxiv(self, topic: str) -> list[SourceDoc]:
        import xml.etree.ElementTree as ET
        response = self.session.get("https://export.arxiv.org/api/query", params={
            "search_query": f"all:{topic}", "start": 0, "max_results": 5,
            "sortBy": "relevance", "sortOrder": "descending",
        }, timeout=self.timeout)
        root = ET.fromstring(response.content)
        ns = {"a": "http://www.w3.org/2005/Atom"}
        output = []
        for entry in root.findall("a:entry", ns):
            output.append(SourceDoc(
                provider="arxiv", title=" ".join((entry.findtext("a:title", default="", namespaces=ns)).split()),
                url=entry.findtext("a:id", default="", namespaces=ns),
                snippet=" ".join((entry.findtext("a:summary", default="", namespaces=ns)).split())[:1800],
                author=", ".join(a.findtext("a:name", default="", namespaces=ns) for a in entry.findall("a:author", ns)[:3]),
                published_at=entry.findtext("a:published", default="", namespaces=ns),
                authority_score=0.82,
            ))
        return output


class ResearchEngine:
    SYSTEM = """You are a scientific research editor for a short-form motion graphics studio.
Preserve uncertainty. Do not invent citations. Return rich JSON, but you may use nested objects when useful.
The downstream engine is permissive: prioritize factual quality over matching a brittle schema."""

    def __init__(self, llm: LLMRouter, cache_root: str | Path):
        self.llm = llm
        self.cache_root = ensure_dir(cache_root)

    def build(self, topic: str, sources: list[SourceDoc], force: bool = False) -> ResearchPack:
        cache_path = self.cache_root / f"research-{hash_value([topic, [s.url for s in sources]])}.json"
        if cache_path.exists() and not force:
            return ResearchPack.model_validate(load_json(cache_path))
        source_payload = [s.model_dump(mode="json") for s in sources]
        fallback = self._fallback(topic, sources)
        raw = self.llm.generate_json(
            system=self.SYSTEM,
            prompt=f"""Build an evidence pack for this topic: {topic}

Sources:
{json.dumps(source_payload, ensure_ascii=False, indent=2)}

Return an object with topic, summary, hooks, facts, comparisons, visual_ideas, limitations.
Each fact should include claim, source_ids, confidence, numeric_values, comparison, visual_hint.
Structured numeric_values are allowed. Never cite a source ID that is not in the supplied list.""",
            namespace="research", fallback=fallback.model_dump(mode="json"), force=force,
        )
        if not isinstance(raw, dict):
            raw = fallback.model_dump(mode="json")
        raw["topic"] = topic
        raw["sources"] = source_payload
        raw["raw_llm_output"] = raw.copy()
        pack = ResearchPack.model_validate(raw)
        valid_ids = {s.source_id for s in sources}
        for index, fact in enumerate(pack.facts, 1):
            fact.fact_id = fact.fact_id or f"F{index:02d}"
            fact.source_ids = [sid for sid in fact.source_ids if sid in valid_ids]
            fact.confidence = max(0.0, min(1.0, float(fact.confidence or 0.0)))
        supported = sum(1 for f in pack.facts if f.source_ids)
        pack.validation_score = round(supported / max(1, len(pack.facts)), 3)
        pack.research_hash = hash_value(pack.model_dump(exclude={"research_hash", "raw_llm_output"}))
        save_json(cache_path, pack)
        return pack

    def _fallback(self, topic: str, sources: list[SourceDoc]) -> ResearchPack:
        facts = []
        for index, source in enumerate(sources[:8], 1):
            claim = source.snippet.strip().split(". ")[0].strip()
            if claim:
                facts.append(Fact(
                    fact_id=f"F{index:02d}", claim=claim[:500], source_ids=[source.source_id],
                    confidence=0.55, visual_hint=source.title,
                ))
        return ResearchPack(
            topic=topic,
            summary=f"Evidence pack assembled from {len(sources)} public sources for {topic}.",
            hooks=[f"What changes first if {topic.lower()}?"],
            facts=facts,
            comparisons=[], visual_ideas=[topic], limitations=["Fallback extraction used; review claims before publication."],
            sources=sources,
        )


In [ ]:
%%writefile /content/scientific_motion_studio_v5/src/scistudio_v5/director.py
# Cell 11: scistudio_v5/director.py
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

from .llm import LLMRouter
from .schemas import Beat, ResearchPack, SceneRequest, ScriptPackage, Storyboard, VisualIntent
from .utils import ensure_dir, hash_value, load_json, save_json


class CreativeDirector:
    SCRIPT_SYSTEM = """You are an award-winning science short-form writer.
Write vivid, accurate, retention-first narration under 60 seconds. Do not flatten scientific nuance.
Return JSON. The engine accepts rich nested structures, so use detail when it helps."""

    VISUAL_SYSTEM = """You are a senior scientific motion director and visual systems designer.
You may request any visual object, diagram, cutaway, field, interface, material, organism, machine,
physical effect, or abstraction. Do not limit yourself to a fixed asset catalog.
Describe the intended visual precisely, including important parts, relationships, reference needs,
layering, physical behavior, and what must remain scientifically recognizable.
Return JSON. The downstream visual compiler is capability-driven and accepts open-ended intents.
Think in complete editorial illustrations, not isolated icons: establish silhouette, foreground/midground/background,
internal parts, causal arrows/fields only when meaningful, and enough semantic groups for object-level motion."""

    def __init__(self, llm: LLMRouter, config: dict[str, Any], cache_root: str | Path):
        self.llm = llm
        self.config = config
        self.cache_root = ensure_dir(cache_root)

    def script(self, research: ResearchPack, force: bool = False) -> ScriptPackage:
        cache_path = self.cache_root / f"script-{research.research_hash}.json"
        if cache_path.exists() and not force:
            return ScriptPackage.model_validate(load_json(cache_path))
        fallback = self._fallback_script(research)
        raw = self.llm.generate_json(
            system=self.SCRIPT_SYSTEM,
            prompt=f"""Create a science short script from this evidence pack:
{json.dumps(research.model_dump(mode='json', exclude={'raw_llm_output'}), ensure_ascii=False, indent=2)}

Requirements:
- 7 to 10 beats.
- Spoken language should feel natural, not like a paper.
- Every beat must add information or change stakes.
- Keep the total target duration between 42 and 58 seconds.
- Include evidence_refs using the supplied fact IDs when a claim depends on research.
- Return: topic, title, hook, beats, closing.
- Each beat may contain any additional creative metadata you find useful.""",
            namespace="script", fallback=fallback.model_dump(mode="json"), force=force,
        )
        package = self._normalize_script(raw, fallback, research.topic)
        save_json(cache_path, package)
        return package

    def storyboard(self, script: ScriptPackage, research: ResearchPack, force: bool = False) -> Storyboard:
        cache_path = self.cache_root / f"storyboard-{script.script_hash}.json"
        if cache_path.exists() and not force:
            return Storyboard.model_validate(load_json(cache_path))
        fallback = self._fallback_storyboard(script)
        raw = self.llm.generate_json(
            system=self.VISUAL_SYSTEM,
            prompt=f"""Direct a vertical scientific motion-graphics video for this script:
{json.dumps(script.model_dump(mode='json', exclude={'raw_llm_output'}), ensure_ascii=False, indent=2)}

Evidence context:
{json.dumps(research.model_dump(mode='json', include={'topic','facts','visual_ideas','limitations'}), ensure_ascii=False, indent=2)}

The result may use any creative vocabulary. For each scene describe:
- narration, headline, duration
- visuals: arbitrary visual intents; each can specify description, important parts, relationships,
  desired layers, reference requests, constraints, and motion intentions
- motions: natural-language physical behavior is allowed
- dashboard: a string or rich object is both acceptable
- layout, camera, transition

Important rules:
- Hero visuals must be scientifically recognizable, not generic icons.
- Build each scene as a complete editorial scientific illustration with deliberate negative space, depth, and 6-20 semantic parts.
- For real objects, request multiple references or a generated isolated reference when needed.
- Use flat polygonal shading, strong outline hierarchy, and a restrained grayscale/blue palette with sparse warning accents.
- Prefer reference/data-driven reconstruction for real objects.
- Use semantic layers for anything that moves independently.
- Explain causality visually rather than through decorative motion.
- No random camera shake.
- Keep the original creative intent; do not reduce it to a fixed asset taxonomy.

Return a JSON object. Scenes may be a list or a scene_0/scene_1 mapping; both are accepted.""",
            namespace="storyboard", fallback=fallback.model_dump(mode="json"), force=force,
        )
        storyboard = self._normalize_storyboard(raw, fallback, script)
        save_json(cache_path, storyboard)
        return storyboard

    def _normalize_script(self, raw: Any, fallback: ScriptPackage, topic: str) -> ScriptPackage:
        if not isinstance(raw, dict):
            raw = fallback.model_dump(mode="json")
        if isinstance(raw.get("script"), dict):
            raw = {**raw, **raw["script"]}
        beats = raw.get("beats")
        if isinstance(beats, dict):
            def key_fn(item):
                match = re.search(r"\d+", str(item[0]))
                return int(match.group()) if match else 10**9
            beats = [value for _, value in sorted(beats.items(), key=key_fn)]
        if not isinstance(beats, list) or not beats:
            beats = fallback.model_dump(mode="json")["beats"]
        normalized = []
        for index, value in enumerate(beats, 1):
            if isinstance(value, str):
                value = {"spoken_line": value}
            if not isinstance(value, dict):
                value = {"spoken_line": str(value)}
            item = dict(value)
            item.setdefault("beat_id", str(item.get("id", f"B{index:02d}")))
            item.setdefault("spoken_line", str(item.get("narration", item.get("voiceover", ""))))
            item.setdefault("visual_event", item.get("visual", item.get("scene", item["spoken_line"])))
            item.setdefault("duration_s", item.get("duration", 5.0))
            item.setdefault("raw", value)
            normalized.append(Beat.model_validate(item))
        package = ScriptPackage(
            topic=topic,
            title=str(raw.get("title", fallback.title)),
            hook=str(raw.get("hook", normalized[0].spoken_line if normalized else fallback.hook)),
            beats=normalized,
            closing=str(raw.get("closing", fallback.closing)),
            raw_llm_output=raw,
        )
        package.total_words = sum(len(re.findall(r"\b\w+\b", beat.spoken_line)) for beat in package.beats)
        default_wps = float(self.config.get("words_per_second", 2.65))
        for beat in package.beats:
            words = len(re.findall(r"\b\w+\b", beat.spoken_line))
            beat.duration_s = max(2.4, beat.duration_s, words / max(1.0, default_wps) + 0.2)
        package.estimated_duration_s = round(sum(beat.duration_s for beat in package.beats), 3)
        package.script_hash = hash_value(package.model_dump(exclude={"script_hash", "raw_llm_output"}))
        return package

    def _normalize_storyboard(self, raw: Any, fallback: Storyboard, script: ScriptPackage) -> Storyboard:
        try:
            storyboard = Storyboard.model_validate(raw if isinstance(raw, dict) else {"topic": script.topic, "scenes": raw})
        except Exception as exc:
            print(f"[director] Open storyboard normalization fallback: {exc}")
            storyboard = fallback
        scenes = list(storyboard.scenes)
        fallback_scenes = fallback.scenes
        # Preserve every LLM scene, but fill only execution invariants that are absent.
        for index, beat in enumerate(script.beats):
            if index >= len(scenes):
                scenes.append(fallback_scenes[min(index, len(fallback_scenes) - 1)])
            scene = scenes[index]
            scene.scene_id = scene.scene_id or f"SC{index + 1:02d}"
            scene.beat_id = scene.beat_id or beat.beat_id
            scene.narration = scene.narration or beat.spoken_line
            scene.headline = scene.headline or self._headline(beat.spoken_line)
            scene.duration_s = max(scene.duration_s, beat.duration_s)
            if not scene.visuals:
                scene.visuals = [VisualIntent(
                    intent_id=f"{scene.scene_id}-hero",
                    description=str(scene.visual_event or beat.visual_event or beat.spoken_line),
                    role="hero",
                    raw_llm_output=scene.visual_event,
                )]
            for visual_index, visual in enumerate(scene.visuals):
                visual.intent_id = visual.intent_id or f"{scene.scene_id}-V{visual_index + 1:02d}"
                if visual.raw_llm_output is None:
                    visual.raw_llm_output = visual.model_dump(mode="json", exclude={"raw_llm_output"})
        storyboard.topic = script.topic
        storyboard.scenes = scenes
        storyboard.estimated_duration_s = round(sum(scene.duration_s for scene in scenes), 3)
        storyboard.storyboard_hash = hash_value(storyboard.model_dump(exclude={"storyboard_hash", "raw_llm_output"}))
        return storyboard

    def _fallback_script(self, research: ResearchPack) -> ScriptPackage:
        claims = [fact.claim for fact in research.facts if fact.claim][:7]
        if not claims:
            claims = [research.summary or research.topic]
        # A retention-shaped fallback (cold-open question -> escalating stakes ->
        # payoff) used only when no LLM is reachable. Still not a substitute for
        # the model-authored script, but far less flat than a claim dump.
        topic_q = research.topic.strip().rstrip("?")
        lines = [
            f"{topic_q}? Here is what actually happens.",
            *claims[:5],
            "And it does not stop there — each change forces the next.",
            "The real lesson: this system is connected, so one shift cascades through every layer.",
        ]
        beats = [Beat(
            beat_id=f"B{index:02d}", purpose="information_gain", duration_s=5.0,
            spoken_line=line, visual_event=line, evidence_refs=[], emotion="curiosity",
            retention_function="escalation", sfx="whoosh" if index == 1 else "none",
        ) for index, line in enumerate(lines, 1)]
        package = ScriptPackage(topic=research.topic, title=research.topic, hook=lines[0], beats=beats, closing=lines[-1])
        package.total_words = sum(len(line.split()) for line in lines)
        package.estimated_duration_s = sum(beat.duration_s for beat in beats)
        package.script_hash = hash_value(package.model_dump(exclude={"script_hash"}))
        return package

    def _fallback_storyboard(self, script: ScriptPackage) -> Storyboard:
        scenes = []
        for index, beat in enumerate(script.beats, 1):
            scenes.append(SceneRequest(
                scene_id=f"SC{index:02d}", beat_id=beat.beat_id, duration_s=beat.duration_s,
                narration=beat.spoken_line, headline=self._headline(beat.spoken_line),
                visual_event=beat.visual_event, background="paper",
                visuals=[VisualIntent(
                    intent_id=f"SC{index:02d}-hero", description=str(beat.visual_event),
                    role="hero", kind="open scientific visual",
                    desired_layers=["primary structure", "causal overlay", "annotation"],
                    motion_intents=["reveal the causal change clearly"],
                )],
                motions=[], layout={"composition": "hero_center", "safe_margin": 0.06},
                dashboard={"request": "scientific status interface derived from the narration"},
                transition="cut",
            ))
        board = Storyboard(topic=script.topic, scenes=scenes)
        board.estimated_duration_s = sum(scene.duration_s for scene in scenes)
        board.storyboard_hash = hash_value(board.model_dump(exclude={"storyboard_hash"}))
        return board

    @staticmethod
    def _headline(text: str) -> str:
        words = re.findall(r"\b[\w'-]+\b", text)
        return " ".join(words[:7]).upper() if words else "SCIENCE EVENT"


In [ ]:
%%writefile /content/scientific_motion_studio_v5/src/scistudio_v5/references.py
# Cell 12: scistudio_v5/references.py
from __future__ import annotations

import json
import mimetypes
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import requests

from .schemas import VisualIntent
from .utils import ensure_dir, hash_value, load_json, save_json


@dataclass
class ReferenceCandidate:
    title: str
    url: str
    mime: str
    license: str = ""
    attribution: str = ""
    source_page: str = ""
    score: float = 0.0


class ReferenceCollector:
    """Reference acquisition with licensing metadata.

    Priority:
    1. User-supplied reference folder.
    2. Wikimedia Commons files with license metadata.
    3. No reference (the compiler uses data or procedural synthesis).

    General web-image search is intentionally not auto-traced because search
    availability does not grant reuse rights.
    """

    COMMONS_API = "https://commons.wikimedia.org/w/api.php"

    def __init__(self, config: dict[str, Any], root: str | Path):
        self.config = config
        self.root = ensure_dir(root)
        self.user_root = ensure_dir(config.get("user_reference_dir", self.root / "user"))
        self.cache_root = ensure_dir(self.root / "cache")
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": "ScientificMotionStudio/5.0 vector-reference"})
        self.timeout = int(config.get("timeout", 30))

    def collect(self, intent: VisualIntent, limit: int = 8, force: bool = False) -> list[ReferenceCandidate]:
        cache_path = self.cache_root / f"refs-{hash_value(intent.model_dump(mode='json'))}.json"
        if cache_path.exists() and not force:
            return [ReferenceCandidate(**item) for item in load_json(cache_path, [])]
        candidates = self._user_candidates(intent)
        queries = self._queries(intent)
        for query in queries[:5]:
            try:
                candidates.extend(self._commons(query, limit=limit))
            except Exception as exc:
                print(f"[references] Wikimedia query skipped ({query}): {exc}")
        dedup: dict[str, ReferenceCandidate] = {}
        tokens = set(re.findall(r"[a-z0-9]+", intent.description.lower()))
        for candidate in candidates:
            key = candidate.url
            title_tokens = set(re.findall(r"[a-z0-9]+", candidate.title.lower()))
            candidate.score = len(tokens & title_tokens) / max(1, len(tokens))
            if key and (key not in dedup or candidate.score > dedup[key].score):
                dedup[key] = candidate
        ranked = sorted(dedup.values(), key=lambda x: (x.score, x.mime == "image/svg+xml"), reverse=True)[:limit]
        save_json(cache_path, [candidate.__dict__ for candidate in ranked])
        return ranked

    def download(self, candidate: ReferenceCandidate, target: str | Path) -> Path:
        target = Path(target)
        ensure_dir(target.parent)
        if target.exists() and target.stat().st_size > 128:
            return target
        response = self.session.get(candidate.url, timeout=self.timeout)
        response.raise_for_status()
        target.write_bytes(response.content)
        return target

    def _queries(self, intent: VisualIntent) -> list[str]:
        queries: list[str] = []
        for item in intent.reference_requests:
            if isinstance(item, str):
                queries.append(item)
            elif isinstance(item, dict):
                text = item.get("query", item.get("description", item.get("subject", "")))
                if text:
                    queries.append(str(text))
        description = intent.description.strip()
        if description:
            queries.extend([
                f"{description} illustration",
                f"{description} diagram",
                f"{description} silhouette",
            ])
        output = []
        for query in queries:
            query = re.sub(r"\s+", " ", query).strip()
            if query and query not in output:
                output.append(query)
        return output

    def _user_candidates(self, intent: VisualIntent) -> list[ReferenceCandidate]:
        tokens = set(re.findall(r"[a-z0-9]+", intent.description.lower()))
        output = []
        for path in self.user_root.glob("**/*"):
            if not path.is_file() or path.suffix.lower() not in {".svg", ".png", ".jpg", ".jpeg", ".webp"}:
                continue
            stem_tokens = set(re.findall(r"[a-z0-9]+", path.stem.lower()))
            if tokens and not (tokens & stem_tokens):
                continue
            mime = mimetypes.guess_type(path.name)[0] or "application/octet-stream"
            output.append(ReferenceCandidate(
                title=path.stem, url=path.resolve().as_uri(), mime=mime,
                license="user-supplied", attribution="user-supplied", source_page=str(path), score=1.0,
            ))
        return output

    def _commons(self, query: str, limit: int = 8) -> list[ReferenceCandidate]:
        payload = self.session.get(self.COMMONS_API, params={
            "action": "query", "generator": "search", "gsrsearch": query,
            "gsrnamespace": 6, "gsrlimit": limit, "prop": "imageinfo|info",
            "iiprop": "url|mime|extmetadata", "inprop": "url", "format": "json",
        }, timeout=self.timeout).json()
        output = []
        for page in payload.get("query", {}).get("pages", {}).values():
            info = (page.get("imageinfo") or [{}])[0]
            metadata = info.get("extmetadata") or {}
            license_name = (metadata.get("LicenseShortName") or {}).get("value", "")
            artist = re.sub(r"<[^>]+>", " ", (metadata.get("Artist") or {}).get("value", ""))
            mime = info.get("mime", "")
            url = info.get("url", "")
            if not url or mime not in {"image/svg+xml", "image/png", "image/jpeg", "image/webp"}:
                continue
            output.append(ReferenceCandidate(
                title=page.get("title", "Wikimedia file"), url=url, mime=mime,
                license=license_name, attribution=artist.strip(), source_page=page.get("fullurl", ""),
            ))
        return output


In [ ]:
%%writefile /content/scientific_motion_studio_v5/src/scistudio_v5/benchmark.py
# Cell 13: scistudio_v5/benchmark.py
from __future__ import annotations

import json
import math
import subprocess
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Iterable
from xml.etree import ElementTree as ET

import cv2
import numpy as np
from PIL import Image

from .utils import ensure_dir, save_json


@dataclass
class StyleFingerprint:
    width: int
    height: int
    foreground_coverage: float
    edge_density: float
    contour_count: float
    connected_components: float
    palette_size: float
    white_fraction: float
    dark_fraction: float
    blue_fraction: float
    gray_fraction: float
    red_fraction: float
    mean_saturation: float
    mean_value: float
    dominant_palette: list[list[int]]

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


class BenchmarkAnalyzer:
    """Extracts a visual-style target from a reference video or images.

    The benchmark intentionally measures illustration grammar rather than
    copying any specific frame: white-space balance, outline density, contour
    complexity, palette discipline, object density, and flat-color usage.
    """

    def __init__(self, root: str | Path):
        self.root = ensure_dir(root)


    def from_video_set(self, video_path: str | Path, sample_count: int = 10) -> list[StyleFingerprint]:
        """Return per-frame style fingerprints so each generated scene is compared
        against the closest reference archetype instead of an averaged video frame."""
        video_path = Path(video_path)
        frames_dir = ensure_dir(self.root / "frames_set")
        duration = self._duration(video_path)
        times = np.linspace(max(0.15, duration * 0.04), max(0.25, duration * 0.96), sample_count)
        frame_paths = []
        profiles = []
        for index, timestamp in enumerate(times):
            target = frames_dir / f"frame_{index:02d}.png"
            subprocess.run([
                "ffmpeg", "-y", "-loglevel", "error", "-ss", f"{timestamp:.4f}",
                "-i", str(video_path), "-frames:v", "1", str(target),
            ], check=True)
            frame_paths.append(target)
            profiles.append(self.from_images([target]))
        save_json(self.root / "reference_style_fingerprint_set.json", [p.to_dict() for p in profiles])
        self.contact_sheet(frame_paths, self.root / "reference_contact_sheet_set.png")
        return profiles

    def compare_to_best(self, generated: StyleFingerprint, references: list[StyleFingerprint]) -> dict[str, Any]:
        if not references:
            raise ValueError("No reference fingerprints supplied")
        comparisons = [self.compare(generated, reference) for reference in references]
        best_index = max(range(len(comparisons)), key=lambda i: comparisons[i]["score"])
        return {"best_index": best_index, **comparisons[best_index], "all_scores": [item["score"] for item in comparisons]}
    def from_video(self, video_path: str | Path, sample_count: int = 8) -> StyleFingerprint:
        video_path = Path(video_path)
        frames_dir = ensure_dir(self.root / "frames")
        # Use a deterministic evenly spaced extraction.
        duration = self._duration(video_path)
        times = np.linspace(max(0.2, duration * 0.08), max(0.3, duration * 0.92), sample_count)
        frame_paths = []
        for index, timestamp in enumerate(times):
            target = frames_dir / f"frame_{index:02d}.png"
            subprocess.run([
                "ffmpeg", "-y", "-loglevel", "error", "-ss", f"{timestamp:.4f}",
                "-i", str(video_path), "-frames:v", "1", str(target),
            ], check=True)
            frame_paths.append(target)
        profile = self.from_images(frame_paths)
        save_json(self.root / "reference_style_fingerprint.json", profile.to_dict())
        self.contact_sheet(frame_paths, self.root / "reference_contact_sheet.png")
        return profile

    def from_images(self, images: Iterable[str | Path]) -> StyleFingerprint:
        metrics = [self.image_metrics(path) for path in images]
        if not metrics:
            raise ValueError("No images supplied for style fingerprinting")
        keys = [
            "foreground_coverage", "edge_density", "contour_count", "connected_components",
            "palette_size", "white_fraction", "dark_fraction", "blue_fraction", "gray_fraction",
            "red_fraction", "mean_saturation", "mean_value",
        ]
        averaged = {key: float(np.mean([m[key] for m in metrics])) for key in keys}
        palettes = np.concatenate([np.asarray(m["dominant_palette"], dtype=np.float32) for m in metrics], axis=0)
        palette = self._cluster_pixels(palettes.astype(np.uint8), k=min(8, len(palettes)))
        first = metrics[0]
        return StyleFingerprint(
            width=int(first["width"]), height=int(first["height"]),
            dominant_palette=[list(map(int, row)) for row in palette], **averaged,
        )

    def image_metrics(self, path: str | Path) -> dict[str, Any]:
        image = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if image is None:
            raise ValueError(f"Could not read image: {path}")
        h, w = image.shape[:2]
        small = cv2.resize(image, (384, max(1, round(h * 384 / w))), interpolation=cv2.INTER_AREA)
        rgb = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)
        hsv = cv2.cvtColor(small, cv2.COLOR_BGR2HSV)
        gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)

        # Estimate background from the four corners and define foreground by color distance.
        corner = np.vstack([
            rgb[:18, :18].reshape(-1, 3), rgb[:18, -18:].reshape(-1, 3),
            rgb[-18:, :18].reshape(-1, 3), rgb[-18:, -18:].reshape(-1, 3),
        ])
        background = np.median(corner, axis=0)
        distance = np.linalg.norm(rgb.astype(np.float32) - background.astype(np.float32), axis=2)
        foreground = distance > 22

        edges = cv2.Canny(gray, 70, 170)
        contours, _ = cv2.findContours((foreground.astype(np.uint8) * 255), cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        components = cv2.connectedComponentsWithStats(foreground.astype(np.uint8), 8)[0] - 1

        # Palette and semantic color families.
        sample = rgb.reshape(-1, 3)
        if len(sample) > 40000:
            rng = np.random.default_rng(42)
            sample = sample[rng.choice(len(sample), 40000, replace=False)]
        palette = self._cluster_pixels(sample, k=8)
        sat = hsv[..., 1] / 255.0
        val = hsv[..., 2] / 255.0
        hue = hsv[..., 0]
        dark = gray < 75
        white = gray > 235
        blue = ((hue >= 85) & (hue <= 125) & (sat > 0.22) & (val > 0.25))
        red = (((hue <= 10) | (hue >= 170)) & (sat > 0.34) & (val > 0.35))
        gray_family = (sat < 0.13) & (gray >= 60) & (gray <= 220)

        return {
            "width": w, "height": h,
            "foreground_coverage": float(np.mean(foreground)),
            "edge_density": float(np.mean(edges > 0)),
            "contour_count": float(len([c for c in contours if cv2.contourArea(c) > 8])),
            "connected_components": float(max(0, components)),
            "palette_size": float(len(palette)),
            "white_fraction": float(np.mean(white)),
            "dark_fraction": float(np.mean(dark)),
            "blue_fraction": float(np.mean(blue)),
            "gray_fraction": float(np.mean(gray_family)),
            "red_fraction": float(np.mean(red)),
            "mean_saturation": float(np.mean(sat)),
            "mean_value": float(np.mean(val)),
            "dominant_palette": [list(map(int, row)) for row in palette],
        }

    def compare(self, generated: StyleFingerprint, reference: StyleFingerprint) -> dict[str, Any]:
        # Metric tolerances are intentionally broad enough to allow original composition,
        # while rejecting icon-like or nearly empty SVGs.
        tolerances = {
            "foreground_coverage": 0.19, "edge_density": 0.035, "contour_count": 95.0,
            "connected_components": 35.0, "palette_size": 3.0, "white_fraction": 0.20,
            "dark_fraction": 0.12, "blue_fraction": 0.10, "gray_fraction": 0.18,
            "red_fraction": 0.035, "mean_saturation": 0.16, "mean_value": 0.16,
        }
        scores: dict[str, float] = {}
        for key, tol in tolerances.items():
            a = float(getattr(generated, key)); b = float(getattr(reference, key))
            scores[key] = float(math.exp(-abs(a - b) / max(tol, 1e-6)))
        palette_score = self._palette_similarity(generated.dominant_palette, reference.dominant_palette)
        scores["palette_similarity"] = palette_score
        weighted = {
            "foreground_coverage": 0.10, "edge_density": 0.13, "contour_count": 0.11,
            "connected_components": 0.08, "palette_size": 0.05, "white_fraction": 0.07,
            "dark_fraction": 0.08, "blue_fraction": 0.08, "gray_fraction": 0.06,
            "red_fraction": 0.03, "mean_saturation": 0.04, "mean_value": 0.04,
            "palette_similarity": 0.13,
        }
        total = sum(scores[key] * weight for key, weight in weighted.items()) / sum(weighted.values())
        return {"score": round(float(total), 4), "scores": {k: round(v, 4) for k, v in scores.items()}}

    def svg_complexity(self, svg_path: str | Path) -> dict[str, Any]:
        text = Path(svg_path).read_text(encoding="utf-8", errors="ignore")
        root = ET.fromstring(text)
        tags = [node.tag.split("}")[-1] for node in root.iter()]
        paths = [node for node in root.iter() if node.tag.split("}")[-1] == "path"]
        groups = [node for node in root.iter() if node.tag.split("}")[-1] == "g"]
        simple = sum(tag in {"circle", "ellipse", "rect", "line"} for tag in tags)
        drawable = sum(tag in {"path", "polygon", "polyline", "circle", "ellipse", "rect", "line"} for tag in tags)
        path_data = [node.attrib.get("d", "") for node in paths]
        path_chars = sum(len(value) for value in path_data)
        curved = sum(any(command in value for command in ("C", "Q", "A")) for value in path_data)
        long_paths = sum(len(value) > 80 for value in path_data)
        return {
            "path_count": len(paths), "group_count": len(groups), "drawable_count": drawable,
            "path_character_count": path_chars,
            "mean_path_character_count": round(path_chars / max(1, len(paths)), 3),
            "curved_path_fraction": round(curved / max(1, len(paths)), 4),
            "long_path_fraction": round(long_paths / max(1, len(paths)), 4),
            "simple_shape_ratio": round(simple / max(1, drawable), 4),
            "has_raster_embed": "<image" in text.lower() or "data:image" in text.lower(),
        }

    @staticmethod
    def contact_sheet(images: Iterable[str | Path], output: str | Path, columns: int = 4) -> Path:
        opened = [Image.open(path).convert("RGB") for path in images]
        if not opened:
            raise ValueError("No images for contact sheet")
        thumb_w = 480
        thumbs = []
        for image in opened:
            thumb_h = round(image.height * thumb_w / image.width)
            thumbs.append(image.resize((thumb_w, thumb_h)))
        row_h = max(image.height for image in thumbs)
        rows = math.ceil(len(thumbs) / columns)
        canvas = Image.new("RGB", (thumb_w * columns, row_h * rows), (240, 240, 238))
        for index, image in enumerate(thumbs):
            canvas.paste(image, ((index % columns) * thumb_w, (index // columns) * row_h))
        output = Path(output)
        ensure_dir(output.parent)
        canvas.save(output)
        return output

    @staticmethod
    def _cluster_pixels(pixels: np.ndarray, k: int = 8) -> np.ndarray:
        pixels = np.asarray(pixels, dtype=np.float32).reshape(-1, 3)
        if len(pixels) == 0:
            return np.zeros((1, 3), dtype=np.uint8)
        k = max(1, min(k, len(pixels)))
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.5)
        _, labels, centers = cv2.kmeans(pixels, k, None, criteria, 4, cv2.KMEANS_PP_CENTERS)
        counts = np.bincount(labels.ravel(), minlength=k)
        order = np.argsort(counts)[::-1]
        return np.clip(centers[order], 0, 255).astype(np.uint8)

    @staticmethod
    def _palette_similarity(a: list[list[int]], b: list[list[int]]) -> float:
        aa = np.asarray(a, dtype=np.float32)
        bb = np.asarray(b, dtype=np.float32)
        if aa.size == 0 or bb.size == 0:
            return 0.0
        distances = []
        for color in aa:
            distances.append(float(np.min(np.linalg.norm(bb - color, axis=1))))
        for color in bb:
            distances.append(float(np.min(np.linalg.norm(aa - color, axis=1))))
        mean = float(np.mean(distances))
        return float(math.exp(-mean / 62.0))

    @staticmethod
    def _duration(video_path: Path) -> float:
        value = subprocess.check_output([
            "ffprobe", "-v", "error", "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1", str(video_path),
        ], text=True).strip()
        return max(0.5, float(value))


In [ ]:
%%writefile /content/scientific_motion_studio_v5/src/scistudio_v5/illustration.py
# Cell 14: scistudio_v5/illustration.py
from __future__ import annotations

import base64
import html
import json
import math
import mimetypes
import re
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Any
from urllib.parse import urlparse
from xml.etree import ElementTree as ET

import cv2
import numpy as np
from PIL import Image

from .benchmark import BenchmarkAnalyzer, StyleFingerprint
from .schemas import AssetRecord, SceneRequest, VisualIntent
from .utils import ensure_dir, hash_value, save_json, sha256_file

# Emit un-prefixed SVG so inlined markup renders in the browser (HTML innerHTML),
# not only in XML parsers. ElementTree otherwise adds ns0: prefixes on tostring.
ET.register_namespace("", "http://www.w3.org/2000/svg")


DEFAULT_PALETTE = {
    "paper": "#F7F7F4",
    "ink": "#17212B",
    "dark": "#4B555D",
    "gray": "#858B90",
    "muted": "#B9BDC2",
    "blue_dark": "#2E6C98",
    "blue": "#4390C4",
    "blue_light": "#9BC9E1",
    "red": "#D73232",
    "warning": "#F3A53B",
    "white": "#FFFFFF",
}


class IllustrationQualityError(RuntimeError):
    pass


class AdvancedRasterVectorizer:
    """Converts a raster illustration into layered editable SVG.

    Unlike a single black/white trace, this implementation:
    - estimates and removes the background;
    - posterizes the image into a disciplined flat palette;
    - preserves separate connected regions;
    - creates semantic color layers and a dedicated exterior outline;
    - simplifies contours while retaining recognizability.
    """

    def __init__(self, root: str | Path, palette: dict[str, str] | None = None):
        self.root = ensure_dir(root)
        self.palette = {**DEFAULT_PALETTE, **(palette or {})}

    def vectorize(
        self,
        image_path: str | Path,
        *,
        asset_id: str,
        desired_layers: list[Any] | None = None,
        output_size: int = 1000,
        color_count: int = 8,
        style_reference: StyleFingerprint | None = None,
    ) -> tuple[str, list[dict[str, Any]], dict[str, Any]]:
        image_path = Path(image_path)
        rgba = cv2.imread(str(image_path), cv2.IMREAD_UNCHANGED)
        if rgba is None:
            raise ValueError(f"Unable to read raster reference: {image_path}")
        if rgba.ndim == 2:
            rgba = cv2.cvtColor(rgba, cv2.COLOR_GRAY2BGRA)
        if rgba.shape[2] == 3:
            alpha = self._background_alpha(rgba)
            rgba = np.dstack([rgba, alpha])
        bgr = rgba[..., :3]
        alpha = rgba[..., 3]

        # Normalize size without stretching.
        h, w = bgr.shape[:2]
        scale = min(output_size / max(1, w), output_size / max(1, h))
        nw, nh = max(1, round(w * scale)), max(1, round(h * scale))
        bgr = cv2.resize(bgr, (nw, nh), interpolation=cv2.INTER_AREA)
        alpha = cv2.resize(alpha, (nw, nh), interpolation=cv2.INTER_AREA)
        canvas = np.full((output_size, output_size, 3), 247, dtype=np.uint8)
        mask_canvas = np.zeros((output_size, output_size), dtype=np.uint8)
        ox, oy = (output_size - nw) // 2, (output_size - nh) // 2
        canvas[oy:oy+nh, ox:ox+nw] = bgr
        mask_canvas[oy:oy+nh, ox:ox+nw] = alpha

        # Bilateral smoothing reduces photographic micro-noise before flat vectorization.
        smooth = cv2.bilateralFilter(canvas, 9, 52, 52)
        rgb = cv2.cvtColor(smooth, cv2.COLOR_BGR2RGB)
        foreground = mask_canvas > 18
        pixels = rgb[foreground]
        if len(pixels) < 50:
            raise ValueError("Reference has too little foreground after background removal")
        if len(pixels) > 100000:
            rng = np.random.default_rng(42)
            pixels_for_cluster = pixels[rng.choice(len(pixels), 100000, replace=False)]
        else:
            pixels_for_cluster = pixels

        target_palette = self._target_palette(style_reference)
        labels, centers = self._quantize(rgb, foreground, pixels_for_cluster, color_count)
        mapped_centers = np.asarray([self._nearest_color(center, target_palette) for center in centers], dtype=np.uint8)

        # Build largest regions first, then details.
        cluster_sizes = [(index, int(np.sum((labels == index) & foreground))) for index in range(len(centers))]
        cluster_sizes.sort(key=lambda item: item[1], reverse=True)
        requested_names = self._layer_names(desired_layers or [])
        groups: list[str] = []
        layer_records: list[dict[str, Any]] = []
        total_paths = 0

        for order, (cluster_index, cluster_size) in enumerate(cluster_sizes):
            if cluster_size < 20:
                continue
            region = ((labels == cluster_index) & foreground).astype(np.uint8) * 255
            kernel = np.ones((3, 3), np.uint8)
            region = cv2.morphologyEx(region, cv2.MORPH_CLOSE, kernel, iterations=1)
            region = cv2.morphologyEx(region, cv2.MORPH_OPEN, kernel, iterations=1)
            contours, hierarchy = cv2.findContours(region, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)
            if hierarchy is None:
                continue
            hierarchy = hierarchy[0]
            paths: list[str] = []
            for contour_index, contour in enumerate(contours):
                area = abs(cv2.contourArea(contour))
                if area < 24:
                    continue
                epsilon = max(0.7, 0.0018 * cv2.arcLength(contour, True))
                approx = cv2.approxPolyDP(contour, epsilon, True)
                if len(approx) < 3:
                    continue
                # Keep only top-level contours here; append direct children as holes.
                if hierarchy[contour_index][3] != -1:
                    continue
                d = self._contour_path(approx)
                child = hierarchy[contour_index][2]
                while child != -1:
                    hole = contours[child]
                    if abs(cv2.contourArea(hole)) > 20:
                        h_approx = cv2.approxPolyDP(hole, max(0.7, 0.0018 * cv2.arcLength(hole, True)), True)
                        if len(h_approx) >= 3:
                            d += " " + self._contour_path(h_approx)
                    child = hierarchy[child][0]
                paths.append(f'<path d="{d}"/>')
            if not paths:
                continue
            color = self._hex(mapped_centers[cluster_index])
            role = self._semantic_color_role(mapped_centers[cluster_index], order)
            name = self._safe_id(requested_names[order] if order < len(requested_names) else role)
            groups.append(
                f'<g id="{name}" fill="{color}" stroke="none" fill-rule="evenodd">{"".join(paths)}</g>'
            )
            layer_records.append({"layer_id": f"{asset_id}--{name}", "role": role, "bbox": [0, 0, output_size, output_size]})
            total_paths += len(paths)

        # Exterior outline is generated from the alpha silhouette, preserving the hand-inked look.
        silhouette = (foreground.astype(np.uint8) * 255)
        contours, _ = cv2.findContours(silhouette, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        outline_paths = []
        for contour in contours:
            if abs(cv2.contourArea(contour)) < 40:
                continue
            approx = cv2.approxPolyDP(contour, max(0.7, 0.0015 * cv2.arcLength(contour, True)), True)
            if len(approx) >= 3:
                outline_paths.append(f'<path d="{self._contour_path(approx)}"/>')
        outline_width = 4.5 if style_reference is None else max(3.2, min(8.0, 3.0 + style_reference.edge_density * 70))
        groups.append(
            f'<g id="exterior-outline" fill="none" stroke="{self.palette["ink"]}" '
            f'stroke-width="{outline_width:.2f}" stroke-linejoin="round" stroke-linecap="round">'
            f'{"".join(outline_paths)}</g>'
        )
        layer_records.append({"layer_id": f"{asset_id}--exterior-outline", "role": "exterior outline", "bbox": [0, 0, output_size, output_size]})

        svg = (
            f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {output_size} {output_size}">'
            f'<g id="{asset_id}">{"".join(groups)}</g></svg>'
        )
        ET.fromstring(svg)
        metadata = {
            "source": str(image_path), "cluster_count": len(cluster_sizes), "path_count": total_paths,
            "outline_paths": len(outline_paths), "construction": "flat-color clustered contour vectorization",
        }
        return svg, layer_records, metadata

    @staticmethod
    def _background_alpha(bgr: np.ndarray) -> np.ndarray:
        h, w = bgr.shape[:2]
        corners = np.vstack([
            bgr[:max(2, h//20), :max(2, w//20)].reshape(-1, 3),
            bgr[:max(2, h//20), -max(2, w//20):].reshape(-1, 3),
            bgr[-max(2, h//20):, :max(2, w//20)].reshape(-1, 3),
            bgr[-max(2, h//20):, -max(2, w//20):].reshape(-1, 3),
        ])
        background = np.median(corners.astype(np.float32), axis=0)
        distance = np.linalg.norm(bgr.astype(np.float32) - background, axis=2)
        alpha = np.clip((distance - 12) * 8, 0, 255).astype(np.uint8)
        alpha = cv2.GaussianBlur(alpha, (3, 3), 0)
        return alpha

    @staticmethod
    def _quantize(rgb: np.ndarray, foreground: np.ndarray, pixels: np.ndarray, k: int):
        pixels = pixels.astype(np.float32)
        k = max(3, min(int(k), len(pixels)))
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 35, 0.45)
        _, _, centers = cv2.kmeans(pixels, k, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
        flat = rgb.reshape(-1, 3).astype(np.float32)
        # Chunked nearest center assignment keeps memory bounded.
        all_labels = np.zeros(len(flat), dtype=np.int16)
        chunk = 100000
        for start in range(0, len(flat), chunk):
            part = flat[start:start+chunk]
            dist = np.sum((part[:, None, :] - centers[None, :, :]) ** 2, axis=2)
            all_labels[start:start+chunk] = np.argmin(dist, axis=1)
        labels = all_labels.reshape(rgb.shape[:2])
        labels[~foreground] = -1
        return labels, np.clip(centers, 0, 255).astype(np.uint8)

    def _target_palette(self, style_reference: StyleFingerprint | None) -> np.ndarray:
        colors = []
        if style_reference is not None:
            colors.extend(style_reference.dominant_palette)
        colors.extend([self._rgb(value) for value in self.palette.values()])
        # Deduplicate approximately.
        unique = []
        for color in colors:
            arr = np.asarray(color, dtype=np.uint8)
            if not any(np.linalg.norm(arr.astype(float) - np.asarray(existing).astype(float)) < 12 for existing in unique):
                unique.append(arr)
        return np.asarray(unique, dtype=np.uint8)

    @staticmethod
    def _nearest_color(color: np.ndarray, palette: np.ndarray) -> np.ndarray:
        # Euclidean RGB is sufficient after posterization and a compact palette.
        index = int(np.argmin(np.sum((palette.astype(np.float32) - color.astype(np.float32)) ** 2, axis=1)))
        return palette[index]

    @staticmethod
    def _contour_path(contour: np.ndarray) -> str:
        pts = contour.reshape(-1, 2)
        if len(pts) < 3:
            return ""
        return "M " + " L ".join(f"{int(x)},{int(y)}" for x, y in pts) + " Z"

    @staticmethod
    def _layer_names(items: list[Any]) -> list[str]:
        output = []
        for item in items:
            if isinstance(item, dict):
                output.append(str(item.get("name", item.get("id", item.get("role", "layer")))))
            else:
                output.append(str(item))
        return output

    def _semantic_color_role(self, rgb: np.ndarray, order: int) -> str:
        r, g, b = [int(x) for x in rgb]
        mx, mn = max(r, g, b), min(r, g, b)
        if mx < 70:
            return "ink detail"
        if mx - mn < 18:
            if mx < 130:
                return "dark structure"
            if mx < 205:
                return "mid gray structure"
            return "light surface"
        if b > r * 1.15 and b > g * 1.05:
            return "blue accent" if mx < 190 else "light blue highlight"
        if r > g * 1.25 and r > b * 1.25:
            return "warning accent"
        return f"color region {order + 1}"

    @staticmethod
    def _safe_id(value: str) -> str:
        return re.sub(r"[^a-zA-Z0-9_-]", "-", value).strip("-") or "layer"

    @staticmethod
    def _hex(rgb: np.ndarray) -> str:
        return "#" + "".join(f"{int(value):02X}" for value in rgb)

    @staticmethod
    def _rgb(value: str) -> list[int]:
        value = value.lstrip("#")
        if len(value) == 3:
            value = "".join(ch * 2 for ch in value)
        return [int(value[i:i+2], 16) for i in (0, 2, 4)]


class OpenSceneIllustrationCompiler:
    """Builds one integrated semantic SVG per scene.

    The compiler does not restrict object classes. It accepts any asset SVGs
    produced by data geometry, reference vectorization, generated references,
    or open visual programs, then composes them into an editorial scientific
    illustration with explicit z-order, stable IDs, typography, dashboards,
    and a benchmark-calibrated palette.
    """

    def __init__(self, root: str | Path, palette: dict[str, str] | None = None):
        self.root = ensure_dir(root)
        self.palette = {**DEFAULT_PALETTE, **(palette or {})}

    def compose(
        self,
        scene: SceneRequest,
        records: list[AssetRecord],
        *,
        width: int,
        height: int,
        layout: dict[str, Any] | None = None,
        include_headline: bool = True,
    ) -> AssetRecord:
        layout = layout if isinstance(layout, dict) else {}
        scene_id = self._safe_id(scene.scene_id or f"scene-{hash_value(scene.narration, 8)}")
        placements = self._placements(scene, records, width, height, layout)
        body = []
        semantic_layers: list[dict[str, Any]] = []

        body.append(self._paper_texture(width, height))
        body.append(f'<g id="{scene_id}--illustration-field">')
        for placement in sorted(placements, key=lambda item: item["z"]):
            record = placement["record"]
            svg_text = Path(record.svg_path).read_text(encoding="utf-8", errors="ignore")
            root = ET.fromstring(svg_text)
            viewbox = root.attrib.get("viewBox", "0 0 1000 1000").replace(",", " ").split()
            try:
                vx, vy, vw, vh = map(float, viewbox[:4])
            except Exception:
                vx, vy, vw, vh = 0.0, 0.0, 1000.0, 1000.0
            x, y, w, h = placement["x"], placement["y"], placement["width"], placement["height"]
            sx, sy = w / max(vw, 1), h / max(vh, 1)
            transform = f"translate({x - vx*sx:.4f} {y - vy*sy:.4f}) scale({sx:.8f} {sy:.8f})"
            group_id = f"{scene_id}--{self._safe_id(record.asset_id)}"
            children = []
            for child in list(root):
                if child.attrib.get("id", "").endswith("background"):
                    continue
                children.append(ET.tostring(child, encoding="unicode"))
            body.append(f'<g id="{group_id}" transform="{transform}">{"".join(children)}</g>')
            semantic_layers.append({"layer_id": group_id, "role": record.description or record.asset_id, "bbox": [x, y, w, h]})
            semantic_layers.extend(record.semantic_layers)
        body.append("</g>")

        if include_headline and scene.headline:
            body.append(self._headline(scene.headline, width, height))
        body.append(self._experiment_panel(scene, width, height))
        if scene.dashboard not in (None, "", {}, []):
            body.append(self._dashboard(scene.dashboard, width, height))
        if scene.narration:
            body.append(self._caption(scene.narration, width, height))

        svg = (
            f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {width} {height}">'
            f'<rect id="{scene_id}--background" width="{width}" height="{height}" fill="{self.palette["paper"]}"/>'
            f'{"".join(body)}</svg>'
        )
        ET.fromstring(svg)
        svg_path = self.root / f"{scene_id}.svg"
        svg_path.write_text(svg, encoding="utf-8")
        preview_path = self.root / f"{scene_id}.png"
        try:
            import cairosvg
            cairosvg.svg2png(bytestring=svg.encode(), write_to=str(preview_path), output_width=width, output_height=height)
        except Exception:
            preview_path = Path("")
        manifest = self.root / f"{scene_id}.json"
        record = AssetRecord(
            asset_id=scene_id,
            intent_id=scene_id,
            description=f"Integrated scene illustration: {scene.headline or scene.narration}",
            method="scene-level editorial composition of open visual intents",
            svg_path=str(svg_path),
            preview_path=str(preview_path) if preview_path and preview_path.exists() else "",
            manifest_path=str(manifest),
            semantic_layers=semantic_layers,
            quality={}, warnings=[], asset_hash=sha256_file(svg_path),
            raw_build_plan={"placements": [{k: v for k, v in item.items() if k != "record"} for item in placements]},
        )
        save_json(manifest, record)
        return record

    def _placements(self, scene: SceneRequest, records: list[AssetRecord], width: int, height: int, layout: dict[str, Any]) -> list[dict[str, Any]]:
        supplied = layout.get("placements", []) if isinstance(layout, dict) else []
        supplied_by_id = {}
        if isinstance(supplied, list):
            for item in supplied:
                if isinstance(item, dict):
                    key = str(item.get("intent_id", item.get("asset_id", item.get("id", ""))))
                    if key:
                        supplied_by_id[key] = item
        output = []
        count = len(records)
        for index, record in enumerate(records):
            custom = supplied_by_id.get(record.intent_id) or supplied_by_id.get(record.asset_id) or {}
            if custom:
                x = self._coord(custom.get("x", 0.06), width)
                y = self._coord(custom.get("y", 0.20), height)
                w = self._coord(custom.get("width", 0.88), width)
                h = self._coord(custom.get("height", 0.62), height)
                z = float(custom.get("z", custom.get("z_index", index + 1)))
            elif count == 1:
                x, y, w, h, z = width * 0.04, height * 0.17, width * 0.92, height * 0.66, 2
            else:
                # Open visual grid with the hero occupying the majority of the canvas.
                if index == 0:
                    x, y, w, h, z = width * 0.03, height * 0.18, width * 0.68, height * 0.65, 2
                else:
                    slot = index - 1
                    x = width * 0.70
                    y = height * (0.20 + slot * 0.19)
                    w = width * 0.27
                    h = height * 0.17
                    z = 3 + slot
            output.append({"record": record, "x": x, "y": y, "width": w, "height": h, "z": z})
        return output

    def _headline(self, text: str, width: int, height: int) -> str:
        # The experiment/dashboard panels occupy the top corners down to ~0.155*h.
        # Keep the headline centred in the gap between them and start it below the
        # panels so the title never collides with the status cards.
        lines = self._wrap(text.upper(), max_chars=24 if width > height else 18)[:2]
        font_size = min(width, height) * (0.060 if width > height else 0.046)
        line_height = font_size * 0.95
        y0 = height * 0.205
        cx = width * 0.5
        tspans = "".join(
            f'<tspan x="{cx:.1f}" y="{y0 + i*line_height:.1f}">{html.escape(line)}</tspan>'
            for i, line in enumerate(lines)
        )
        return (
            f'<g id="headline"><text text-anchor="middle" font-family="Arial Narrow,Arial,sans-serif" '
            f'font-size="{font_size:.1f}" font-weight="900" letter-spacing="-1.5" fill="{self.palette["ink"]}">{tspans}</text></g>'
        )

    def _experiment_panel(self, scene: SceneRequest, width: int, height: int) -> str:
        x, y, w, h = width * 0.018, height * 0.025, width * 0.255, height * 0.13
        size = min(width, height) * 0.026
        status = "RUNNING" if not scene.transition else str(scene.transition).upper()[:18]
        return (
            f'<g id="experiment-panel"><rect x="{x:.1f}" y="{y:.1f}" width="{w:.1f}" height="{h:.1f}" fill="{self.palette["paper"]}" '
            f'stroke="{self.palette["ink"]}" stroke-width="2.7"/>'
            f'<text x="{x+12:.1f}" y="{y+size+4:.1f}" font-family="Arial Narrow,Arial,sans-serif" font-size="{size:.1f}" font-weight="900" fill="{self.palette["ink"]}">EXPERIMENT {html.escape(scene.scene_id or "#001")}</text>'
            f'<line x1="{x+10:.1f}" y1="{y+size+11:.1f}" x2="{x+w-10:.1f}" y2="{y+size+11:.1f}" stroke="{self.palette["ink"]}" stroke-width="1.4"/>'
            f'<text x="{x+12:.1f}" y="{y+size*2.08:.1f}" font-family="Arial Narrow,Arial,sans-serif" font-size="{size*0.83:.1f}" font-weight="700" fill="{self.palette["ink"]}">SIMULATION STATUS:</text>'
            f'<text x="{x+12:.1f}" y="{y+size*2.82:.1f}" font-family="Arial Narrow,Arial,sans-serif" font-size="{size*0.83:.1f}" font-weight="900" fill="{self.palette["ink"]}">{html.escape(status)}</text></g>'
        )

    def _dashboard(self, dashboard: Any, width: int, height: int) -> str:
        if isinstance(dashboard, dict):
            title = str(dashboard.get("title", dashboard.get("label", dashboard.get("type", "SYSTEM STATE"))))
            value = str(dashboard.get("value", dashboard.get("metric", dashboard.get("status", "ACTIVE"))))
            unit = str(dashboard.get("unit", ""))
            critical = bool(dashboard.get("critical", False)) or "critical" in value.lower()
        else:
            title, value, unit = str(dashboard), "ACTIVE", ""
            critical = "critical" in str(dashboard).lower()
        x, y, w, h = width * 0.79, height * 0.025, width * 0.19, height * 0.13
        color = self.palette["red"] if critical else self.palette["ink"]
        value_text = f"{value} {unit}".strip()
        size = min(width, height) * 0.028
        return (
            f'<g id="dashboard"><rect x="{x:.1f}" y="{y:.1f}" width="{w:.1f}" height="{h:.1f}" fill="{self.palette["paper"]}" stroke="{self.palette["ink"]}" stroke-width="2.7"/>'
            f'<text x="{x+12:.1f}" y="{y+size+4:.1f}" font-family="Arial Narrow,Arial,sans-serif" font-size="{size:.1f}" font-weight="900" fill="{self.palette["ink"]}">{html.escape(title.upper()[:24])}</text>'
            f'<line x1="{x+10:.1f}" y1="{y+size+10:.1f}" x2="{x+w-10:.1f}" y2="{y+size+10:.1f}" stroke="{self.palette["ink"]}" stroke-width="1.4"/>'
            f'<text x="{x+w-12:.1f}" y="{y+h-15:.1f}" text-anchor="end" font-family="Arial Narrow,Arial,sans-serif" font-size="{size*1.35:.1f}" font-weight="900" fill="{color}">{html.escape(value_text[:20])}</text></g>'
        )

    def _caption(self, narration: str, width: int, height: int) -> str:
        # Reference style usually uses an in-scene statement rather than a heavy subtitle box.
        line = self._wrap(narration, max_chars=52 if width > height else 34)[0]
        size = min(width, height) * 0.026
        return (
            f'<text id="narration-caption" x="{width*0.975:.1f}" y="{height*0.955:.1f}" text-anchor="end" '
            f'font-family="Arial Narrow,Arial,sans-serif" font-size="{size:.1f}" font-weight="800" fill="{self.palette["ink"]}" opacity="0.68">{html.escape(line.upper())}</text>'
        )

    def _paper_texture(self, width: int, height: int) -> str:
        # Deterministic sparse texture, fully vector and animation-safe.
        rng = np.random.default_rng(20260713)
        marks = []
        for index in range(130):
            x = float(rng.uniform(0, width)); y = float(rng.uniform(0, height))
            r = float(rng.uniform(0.35, 1.15)); opacity = float(rng.uniform(0.018, 0.055))
            marks.append(f'<circle cx="{x:.2f}" cy="{y:.2f}" r="{r:.2f}" fill="{self.palette["ink"]}" opacity="{opacity:.3f}"/>')
        return f'<g id="paper-texture">{"".join(marks)}</g>'

    @staticmethod
    def _coord(value: Any, total: float) -> float:
        try:
            number = float(value)
        except Exception:
            return 0.0
        return number * total if -1.0 <= number <= 1.0 else number

    @staticmethod
    def _wrap(text: str, max_chars: int) -> list[str]:
        words = text.split()
        lines, current = [], []
        for word in words:
            trial = " ".join([*current, word])
            if current and len(trial) > max_chars:
                lines.append(" ".join(current)); current = [word]
            else:
                current.append(word)
        if current:
            lines.append(" ".join(current))
        return lines or [text]

    @staticmethod
    def _safe_id(value: str) -> str:
        return re.sub(r"[^a-zA-Z0-9_-]", "-", value).strip("-") or "scene"


class BenchmarkQualityGate:
    """Rejects technically valid but illustration-poor SVGs.

    The gate never silently accepts a generic icon for a concrete scene. It can
    be used after each critic round; if the budget is exhausted, the pipeline
    raises an explicit quality error instead of publishing the asset.
    """

    def __init__(self, analyzer: BenchmarkAnalyzer, reference: StyleFingerprint | list[StyleFingerprint], config: dict[str, Any] | None = None):
        self.analyzer = analyzer
        self.references = reference if isinstance(reference, list) else [reference]
        self.config = config or {}

    def evaluate(self, record: AssetRecord) -> dict[str, Any]:
        if not record.preview_path or not Path(record.preview_path).exists():
            return {"passed": False, "score": 0.0, "failures": ["preview missing"]}
        generated = self.analyzer.from_images([record.preview_path])
        comparison = self.analyzer.compare_to_best(generated, self.references)
        complexity = self.analyzer.svg_complexity(record.svg_path)
        min_score = float(self.config.get("style_score_threshold", 0.72))
        min_paths = int(self.config.get("minimum_path_count", 16))
        min_groups = int(self.config.get("minimum_group_count", 7))
        max_simple = float(self.config.get("maximum_simple_shape_ratio", 0.72))
        min_curved = float(self.config.get("minimum_curved_path_fraction", 0.045))
        min_long = float(self.config.get("minimum_long_path_fraction", 0.16))
        failures = []
        if comparison["score"] < min_score:
            failures.append(f"benchmark style score {comparison['score']:.3f} < {min_score:.3f}")
        if complexity["path_count"] < min_paths:
            failures.append(f"path count {complexity['path_count']} < {min_paths}")
        if complexity["group_count"] < min_groups:
            failures.append(f"semantic group count {complexity['group_count']} < {min_groups}")
        if complexity["simple_shape_ratio"] > max_simple and complexity["path_count"] < min_paths * 2:
            failures.append("asset remains too icon-like / primitive-heavy")
        if complexity.get("curved_path_fraction", 0.0) < min_curved and complexity.get("long_path_fraction", 0.0) < min_long:
            failures.append("geometry is dominated by repeated short primitives rather than illustration contours")
        if complexity["has_raster_embed"]:
            failures.append("raster embed detected")
        return {
            "passed": not failures,
            "score": comparison["score"],
            "style_comparison": comparison,
            "complexity": complexity,
            "failures": failures,
            "generated_fingerprint": generated.to_dict(),
        }


class EditorialBenchmarkDemo:
    """A deterministic open-program scene used only for regression benchmarking.

    It demonstrates the quality level expected from the compiler without using
    the supplied reference frame as geometry. The paths and composition are
    original and remain fully layered for animation.
    """

    def __init__(self, root: str | Path, palette: dict[str, str] | None = None):
        self.root = ensure_dir(root)
        self.palette = {**DEFAULT_PALETTE, **(palette or {})}

    def build(self) -> AssetRecord:
        p = self.palette
        svg = f'''<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1280 720">
<rect id="background" width="1280" height="720" fill="{p['paper']}"/>
<g id="paper-texture" opacity=".055" fill="{p['ink']}">
  <circle cx="85" cy="104" r="1"/><circle cx="225" cy="606" r=".8"/><circle cx="548" cy="67" r=".7"/><circle cx="988" cy="632" r="1"/><circle cx="1177" cy="192" r=".8"/>
</g>
<g id="status-panel" font-family="Arial Narrow,Arial,sans-serif" fill="{p['ink']}">
 <rect x="20" y="20" width="300" height="104" fill="{p['paper']}" stroke="{p['ink']}" stroke-width="2.5"/>
 <text x="32" y="48" font-size="23" font-weight="900">EXPERIMENT #071</text><line x1="30" y1="58" x2="309" y2="58" stroke="{p['ink']}"/>
 <text x="32" y="83" font-size="20" font-weight="800">SIMULATION STATUS:</text><text x="32" y="108" font-size="20" font-weight="900">CRITICAL</text>
</g>
<g id="metric-panel" font-family="Arial Narrow,Arial,sans-serif">
 <rect x="1015" y="20" width="245" height="102" fill="{p['paper']}" stroke="{p['ink']}" stroke-width="2.5"/>
 <text x="1030" y="49" font-size="23" font-weight="900" fill="{p['ink']}">WATER WALL</text><line x1="1028" y1="59" x2="1248" y2="59" stroke="{p['ink']}"/>
 <path d="M1035 104 L1052 70 L1069 104 Z" fill="{p['red']}"/><text x="1238" y="104" text-anchor="end" font-size="34" font-weight="900" fill="{p['red']}">18 m</text>
</g>
<g id="sky-debris" fill="{p['dark']}" stroke="{p['ink']}" stroke-width="3">
 <path d="M808 102 L852 82 L886 110 L880 151 L842 168 L809 140 Z"/><path d="M916 178 L941 167 L958 185 L951 211 L928 216 L910 197 Z"/>
 <path d="M1002 121 L1021 111 L1038 128 L1029 149 L1008 151 L995 135 Z"/>
</g>
<g id="storm-cloud" fill="{p['dark']}" stroke="{p['ink']}" stroke-width="3.2">
 <path d="M783 0 L1280 0 L1280 202 L1204 207 L1158 185 L1110 204 L1053 181 L1012 194 L955 164 L909 177 L864 147 L818 154 L790 120 Z"/>
</g>
<g id="wave-base" stroke="{p['ink']}" stroke-width="4" stroke-linejoin="round">
 <path d="M0 520 L120 478 L240 505 L356 466 L472 507 L588 474 L704 508 L820 475 L944 510 L1070 484 L1280 520 L1280 720 L0 720 Z" fill="{p['blue_dark']}"/>
 <path d="M0 574 L142 536 L254 566 L389 524 L514 570 L646 532 L780 575 L930 536 L1082 571 L1280 548 L1280 720 L0 720 Z" fill="{p['blue']}"/>
 <path d="M0 625 L132 593 L265 616 L391 579 L522 621 L657 586 L796 622 L951 588 L1096 615 L1280 590 L1280 720 L0 720 Z" fill="{p['blue_light']}"/>
</g>
<g id="wave-crest" stroke="{p['ink']}" stroke-width="4.2" stroke-linejoin="round">
 <path d="M0 474 C74 393 121 298 181 231 C229 176 294 141 371 143 C448 145 513 190 544 253 C566 299 553 343 521 374 C487 406 441 416 410 397 C452 377 471 348 462 321 C447 276 390 257 342 276 C278 301 255 371 266 441 L224 493 L129 510 Z" fill="{p['blue_light']}"/>
 <path d="M0 505 C69 433 113 344 171 281 C223 224 286 196 350 206 C404 214 445 246 460 289 C469 315 458 340 438 358 C401 390 354 386 324 358 C344 340 348 320 338 304 C320 275 276 278 246 305 C194 351 185 424 199 493 L134 536 Z" fill="{p['blue']}"/>
 <path d="M0 545 C64 478 102 398 153 342 C189 303 225 286 261 293 C225 323 209 363 215 408 C220 449 244 480 278 493 L225 548 L126 574 Z" fill="{p['blue_dark']}"/>
 <path d="M126 272 C172 224 229 180 302 163 C366 148 426 159 469 194 L445 205 L413 201 L389 223 L353 206 L327 234 L293 214 L268 243 L231 229 L209 258 L174 247 L153 279 Z" fill="{p['white']}"/>
 <path d="M322 258 L350 239 L372 263 L397 244 L421 273 L447 258 L461 291 L438 310 L409 294 L389 319 L359 300 L341 323 L315 305 Z" fill="{p['white']}"/>
</g>
<g id="coastal-city" stroke="{p['ink']}" stroke-width="3" stroke-linejoin="round">
 <path d="M761 579 L842 504 L910 506 L955 470 L1014 471 L1068 436 L1135 438 L1181 399 L1280 404 L1280 720 L728 720 Z" fill="{p['dark']}"/>
 <g id="industrial-buildings" fill="{p['muted']}">
  <path d="M861 487 L910 487 L910 558 L861 558 Z"/><path d="M919 450 L970 450 L970 558 L919 558 Z"/>
  <path d="M981 415 L1048 415 L1048 558 L981 558 Z"/><path d="M1063 452 L1134 452 L1134 558 L1063 558 Z"/>
  <path d="M1146 404 L1251 404 L1251 558 L1146 558 Z"/>
 </g>
 <g id="windows" fill="{p['dark']}" stroke="none">
  <rect x="874" y="501" width="10" height="10"/><rect x="892" y="501" width="10" height="10"/><rect x="934" y="466" width="10" height="10"/><rect x="951" y="466" width="10" height="10"/>
  <rect x="997" y="434" width="11" height="11"/><rect x="1018" y="434" width="11" height="11"/><rect x="997" y="455" width="11" height="11"/><rect x="1018" y="455" width="11" height="11"/>
 </g>
 <g id="smokestacks">
  <path d="M1168 404 L1177 322 L1190 322 L1197 404 Z" fill="{p['muted']}"/><path d="M1210 404 L1218 296 L1232 296 L1239 404 Z" fill="{p['muted']}"/>
  <path d="M1173 352 L1193 352 L1192 365 L1171 365 Z" fill="{p['red']}" stroke="none"/><path d="M1214 334 L1235 334 L1234 348 L1213 348 Z" fill="{p['red']}" stroke="none"/>
 </g>
</g>
<g id="smoke" fill="{p['gray']}" stroke="{p['ink']}" stroke-width="2.5" opacity=".92">
 <path d="M1168 314 C1142 293 1148 267 1171 259 C1153 228 1181 209 1207 225 C1221 197 1262 201 1266 232 C1294 224 1309 257 1287 276 C1300 303 1269 323 1245 309 C1222 332 1191 327 1168 314 Z"/>
</g>
<g id="headline" font-family="Arial Narrow,Arial,sans-serif" fill="{p['ink']}" text-anchor="middle">
 <text x="666" y="320" font-size="54" font-weight="900" letter-spacing="-1.5">COASTAL IMPACT</text>
 <text x="666" y="370" font-size="54" font-weight="900" letter-spacing="-1.5">CASCADE</text>
</g>
<text id="footer" x="1245" y="691" text-anchor="end" font-family="Arial Narrow,Arial,sans-serif" font-size="19" font-weight="800" fill="{p['ink']}" opacity=".68">SCIENTIFIC INCIDENT SIMULATOR</text>
</svg>'''
        svg_path = self.root / "editorial_benchmark_demo.svg"
        svg_path.write_text(svg, encoding="utf-8")
        preview_path = self.root / "editorial_benchmark_demo.png"
        import cairosvg
        cairosvg.svg2png(bytestring=svg.encode(), write_to=str(preview_path), output_width=1280, output_height=720)
        groups = [node.attrib.get("id", "") for node in ET.fromstring(svg).iter() if node.tag.split("}")[-1] == "g" and node.attrib.get("id")]
        semantic_layers = [{"layer_id": group, "role": group.replace("-", " "), "bbox": [0, 0, 1280, 720]} for group in groups]
        manifest = self.root / "editorial_benchmark_demo.json"
        record = AssetRecord(
            asset_id="editorial-benchmark-demo", intent_id="benchmark-demo",
            description="Original benchmark-level scientific editorial illustration",
            method="open layered SVG scene program", svg_path=str(svg_path), preview_path=str(preview_path),
            manifest_path=str(manifest), semantic_layers=semantic_layers, quality={}, warnings=[],
            asset_hash=sha256_file(svg_path), raw_build_plan={"purpose": "quality regression benchmark"},
        )
        save_json(manifest, record)
        return record


In [ ]:
%%writefile /content/scientific_motion_studio_v5/src/scistudio_v5/svg_compiler.py
# Cell 15: scistudio_v5/svg_compiler.py
from __future__ import annotations

import html
import json
import math
import re
import shutil
from pathlib import Path
from typing import Any, Callable
from urllib.parse import urlparse
from xml.etree import ElementTree as ET

ET.register_namespace("", "http://www.w3.org/2000/svg")

import numpy as np

from .llm import LLMRouter
from .benchmark import StyleFingerprint
from .illustration import AdvancedRasterVectorizer, DEFAULT_PALETTE
from .references import ReferenceCandidate, ReferenceCollector
from .schemas import AssetRecord, BuildPlan, VisualIntent
from .utils import ensure_dir, hash_value, load_json, save_json, sha256_file


FORBIDDEN = ("<script", "<image", "data:image", "<foreignobject", "javascript:", "onload=", "onclick=")
DRAWABLE = {"path", "polygon", "polyline", "circle", "ellipse", "rect", "line", "text", "g"}


class SVGValidator:
    @staticmethod
    def sanitize(text: str) -> str:
        text = re.sub(r"<script.*?</script>", "", text, flags=re.I | re.S)
        text = re.sub(r"<foreignObject.*?</foreignObject>", "", text, flags=re.I | re.S)
        text = re.sub(r"<image\b[^>]*/?>", "", text, flags=re.I | re.S)
        text = re.sub(r"\son\w+\s*=\s*(['\"]).*?\1", "", text, flags=re.I | re.S)
        text = re.sub(r"javascript\s*:", "", text, flags=re.I)
        return text

    @staticmethod
    def validate(text: str) -> None:
        low = text.lower()
        violations = [token for token in FORBIDDEN if token in low]
        if violations:
            raise ValueError(f"Forbidden SVG content: {violations}")
        root = ET.fromstring(text)
        if root.tag.split("}")[-1] != "svg":
            raise ValueError("Root element is not SVG")
        if "viewBox" not in root.attrib:
            raise ValueError("SVG must define a viewBox")
        if not any(element.tag.split("}")[-1] in DRAWABLE for element in root.iter()):
            raise ValueError("SVG contains no drawable elements")

    @staticmethod
    def prefix_ids(text: str, prefix: str) -> str:
        safe = re.sub(r"[^a-zA-Z0-9_-]", "-", prefix)
        ids = re.findall(r'\bid=["\']([^"\']+)["\']', text)
        for old in sorted(set(ids), key=len, reverse=True):
            new = f"{safe}--{old}"
            text = re.sub(rf'\bid=(["\']){re.escape(old)}\1', f'id="{new}"', text)
            text = text.replace(f"url(#{old})", f"url(#{new})")
            text = text.replace(f'href="#{old}"', f'href="#{new}"')
            text = text.replace(f"href='#{old}'", f"href='#{new}'")
        return text


class ProgramCompiler:
    """Compiles an extensible visual program into valid SVG.

    `op` is intentionally a free string. Known atomic operations compile
    directly; unknown compound operations are recursively decomposed by an
    injected resolver instead of being replaced with a generic blob.
    """

    ATOMIC_OPS = {
        "group", "circle", "ellipse", "rect", "rounded_rect", "line", "polyline",
        "polygon", "path", "text", "label", "arrow", "arc", "ring", "gauge", "bar",
        "grid", "wave", "vector_field", "radial_field", "particles", "cutaway",
        "callout", "bracket", "axis", "spark", "flow_band", "raw_svg_fragment",
    }

    def __init__(self, style: dict[str, Any], resolver: Callable[[dict[str, Any]], list[dict[str, Any]]] | None = None):
        self.style = style
        self.resolver = resolver
        self.warnings: list[str] = []

    def compile(self, program: dict[str, Any], prefix: str = "asset") -> tuple[str, list[dict[str, Any]]]:
        canvas = program.get("canvas", [1000, 1000])
        if isinstance(canvas, dict):
            width = float(canvas.get("width", 1000)); height = float(canvas.get("height", 1000))
        else:
            width, height = (list(canvas) + [1000, 1000])[:2]
            width, height = float(width), float(height)
        nodes = program.get("nodes", program.get("components", program.get("layers", [])))
        if isinstance(nodes, dict):
            nodes = [dict(value, id=key) if isinstance(value, dict) else {"id": key, "op": "label", "text": value} for key, value in nodes.items()]
        if not isinstance(nodes, list):
            nodes = [nodes]
        markup: list[str] = []
        layers: list[dict[str, Any]] = []
        for index, node in enumerate(nodes):
            compiled, records = self._node(node, depth=0, fallback_id=f"layer_{index+1}")
            markup.append(compiled)
            layers.extend(records)
        background = program.get("background", self.style.get("paper", "#F7F7F4"))
        svg = (
            f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {width:g} {height:g}">'
            f'<rect id="background" width="{width:g}" height="{height:g}" fill="{html.escape(str(background))}"/>'
            f'<g id="content">{"".join(markup)}</g></svg>'
        )
        svg = SVGValidator.prefix_ids(SVGValidator.sanitize(svg), prefix)
        SVGValidator.validate(svg)
        # Update layer IDs after prefixing.
        for layer in layers:
            layer["layer_id"] = f"{prefix}--{layer['layer_id']}"
        return svg, layers

    def _node(self, value: Any, depth: int, fallback_id: str) -> tuple[str, list[dict[str, Any]]]:
        if isinstance(value, str):
            value = {"op": "label", "id": fallback_id, "text": value}
        if not isinstance(value, dict):
            value = {"op": "label", "id": fallback_id, "text": str(value)}
        node = dict(value)
        op = str(node.get("op", node.get("shape", node.get("type", "group")))).strip().lower().replace(" ", "_")
        node_id = self._safe_id(str(node.get("id", node.get("name", fallback_id))))
        if op not in self.ATOMIC_OPS:
            if depth < 4 and self.resolver:
                expanded = self.resolver(node)
                if expanded:
                    return self._node({"op": "group", "id": node_id, "children": expanded}, depth + 1, node_id)
            raise ValueError(f"Unsupported visual operation could not be decomposed: {op}")
        params = dict(node.get("params", {}))
        for key, val in node.items():
            if key not in {"op", "shape", "type", "id", "name", "params", "children", "components", "layers"} and key not in params:
                params[key] = val
        children = node.get("children", node.get("components", node.get("layers", [])))
        if not isinstance(children, list):
            children = [children] if children else []
        fill = self._color(params.get("fill", "none"))
        stroke = self._color(params.get("stroke", self.style.get("ink", "#17212B")))
        sw = self._num(params.get("stroke_width", 5))
        common = f'id="{node_id}" fill="{fill}" stroke="{stroke}" stroke-width="{sw:g}" stroke-linecap="round" stroke-linejoin="round"'

        if op == "group":
            inner, records = [], []
            for index, child in enumerate(children):
                part, rec = self._node(child, depth + 1, f"{node_id}_{index+1}")
                inner.append(part); records.extend(rec)
            transform = html.escape(str(params.get("transform", "")))
            markup = f'<g {common} transform="{transform}">{"".join(inner)}</g>'
            return markup, [{"layer_id": node_id, "role": str(params.get("role", node_id)), "bbox": params.get("bbox", [0,0,1000,1000])}, *records]

        if op in {"circle", "ellipse", "rect", "rounded_rect", "line", "polyline", "polygon", "path"}:
            markup = self._basic_shape(op, common, params)
        elif op in {"text", "label"}:
            markup = self._text(common, params, label=(op == "label"))
        elif op == "arrow":
            markup = self._arrow(node_id, params)
        elif op in {"arc", "ring"}:
            markup = self._arc(common, params, ring=(op == "ring"))
        elif op == "gauge":
            markup = self._gauge(node_id, params)
        elif op == "bar":
            markup = self._bar(node_id, params)
        elif op == "grid":
            markup = self._grid(node_id, params)
        elif op == "wave":
            markup = self._wave(node_id, params)
        elif op == "vector_field":
            markup = self._vector_field(node_id, params)
        elif op == "radial_field":
            markup = self._radial_field(node_id, params)
        elif op == "particles":
            markup = self._particles(node_id, params)
        elif op == "cutaway":
            markup = self._cutaway(node_id, params, children, depth)
        elif op == "callout":
            markup = self._callout(node_id, params)
        elif op == "bracket":
            markup = self._bracket(node_id, params)
        elif op == "axis":
            markup = self._axis(node_id, params)
        elif op == "spark":
            markup = self._spark(node_id, params)
        elif op == "flow_band":
            markup = self._flow_band(node_id, params)
        elif op == "raw_svg_fragment":
            fragment = SVGValidator.sanitize(str(params.get("markup", params.get("svg", ""))))
            # Wrap and parse to reject malformed or executable fragments.
            ET.fromstring(f'<svg xmlns="http://www.w3.org/2000/svg">{fragment}</svg>')
            markup = f'<g id="{node_id}">{fragment}</g>'
        else:
            raise ValueError(op)
        return markup, [{"layer_id": node_id, "role": str(params.get("role", node_id)), "bbox": params.get("bbox", [0,0,1000,1000])}]

    def _basic_shape(self, op: str, common: str, p: dict[str, Any]) -> str:
        if op == "circle":
            return f'<circle {common} cx="{self._num(p.get("cx",500)):g}" cy="{self._num(p.get("cy",500)):g}" r="{self._num(p.get("r",100)):g}"/>'
        if op == "ellipse":
            return f'<ellipse {common} cx="{self._num(p.get("cx",500)):g}" cy="{self._num(p.get("cy",500)):g}" rx="{self._num(p.get("rx",160)):g}" ry="{self._num(p.get("ry",100)):g}"/>'
        if op in {"rect", "rounded_rect"}:
            rx = self._num(p.get("rx", 22 if op == "rounded_rect" else 0))
            return f'<rect {common} x="{self._num(p.get("x",100)):g}" y="{self._num(p.get("y",100)):g}" width="{self._num(p.get("width",800)):g}" height="{self._num(p.get("height",300)):g}" rx="{rx:g}"/>'
        if op == "line":
            return f'<line {common} x1="{self._num(p.get("x1",100)):g}" y1="{self._num(p.get("y1",500)):g}" x2="{self._num(p.get("x2",900)):g}" y2="{self._num(p.get("y2",500)):g}"/>'
        if op in {"polyline", "polygon"}:
            points = p.get("points", [[100,500],[500,100],[900,500]])
            points_text = " ".join(f"{self._num(q[0]):g},{self._num(q[1]):g}" for q in points if isinstance(q,(list,tuple)) and len(q)>=2)
            return f'<{op} {common} points="{points_text}"/>'
        d = html.escape(str(p.get("d", "M100 500 C300 100 700 900 900 500")), quote=True)
        return f'<path {common} d="{d}"/>'

    def _text(self, common: str, p: dict[str, Any], label: bool) -> str:
        x, y = self._num(p.get("x",500)), self._num(p.get("y",500))
        size = self._num(p.get("font_size", 32 if label else 48))
        weight = html.escape(str(p.get("font_weight", 800 if label else 700)))
        anchor = html.escape(str(p.get("anchor", "middle")))
        text = html.escape(str(p.get("text", p.get("value", "LABEL"))))
        fill = self._color(p.get("text_fill", p.get("fill", self.style.get("ink", "#17212B"))))
        # `common` already carries fill/stroke/stroke-width; strip them so <text>
        # does not emit duplicate attributes (which is invalid XML).
        base = re.sub(r'\s+(?:fill|stroke|stroke-width|stroke-linecap|stroke-linejoin)="[^"]*"', "", common)
        return f'<text {base} stroke="none" fill="{fill}" x="{x:g}" y="{y:g}" font-family="Arial, sans-serif" font-size="{size:g}" font-weight="{weight}" text-anchor="{anchor}">{text}</text>'

    def _arrow(self, node_id: str, p: dict[str, Any]) -> str:
        x1,y1,x2,y2 = [self._num(p.get(k,d)) for k,d in (("x1",160),("y1",500),("x2",840),("y2",500))]
        color=self._color(p.get("stroke",self.style.get("blue","#2D9CDB"))); sw=self._num(p.get("stroke_width",14)); head=self._num(p.get("head",34))
        angle=math.atan2(y2-y1,x2-x1); a1=angle+2.65; a2=angle-2.65
        hx1,hy1=x2+head*math.cos(a1),y2+head*math.sin(a1); hx2,hy2=x2+head*math.cos(a2),y2+head*math.sin(a2)
        return f'<g id="{node_id}" fill="none" stroke="{color}" stroke-width="{sw:g}" stroke-linecap="round" stroke-linejoin="round"><path d="M{x1:g},{y1:g} L{x2:g},{y2:g}"/><path d="M{hx1:g},{hy1:g} L{x2:g},{y2:g} L{hx2:g},{hy2:g}"/></g>'

    def _arc(self, common: str, p: dict[str, Any], ring: bool) -> str:
        cx,cy,r=self._num(p.get("cx",500)),self._num(p.get("cy",500)),self._num(p.get("r",250)); start=math.radians(self._num(p.get("start_deg",-140))); end=math.radians(self._num(p.get("end_deg",140)))
        x1,y1=cx+r*math.cos(start),cy+r*math.sin(start); x2,y2=cx+r*math.cos(end),cy+r*math.sin(end); large=1 if abs(end-start)>math.pi else 0
        if ring: return f'<circle {common} cx="{cx:g}" cy="{cy:g}" r="{r:g}"/>'
        return f'<path {common} d="M{x1:g},{y1:g} A{r:g},{r:g} 0 {large} 1 {x2:g},{y2:g}"/>'

    def _gauge(self, i: str, p: dict[str, Any]) -> str:
        cx,cy,r=self._num(p.get("cx",500)),self._num(p.get("cy",550)),self._num(p.get("r",240)); value=max(0,min(1,self._num(p.get("value",.65)))); angle=math.radians(-140+280*value); nx,ny=cx+r*.75*math.cos(angle),cy+r*.75*math.sin(angle)
        return f'<g id="{i}"><path d="M{cx-r*math.cos(math.radians(40)):g},{cy+r*math.sin(math.radians(40)):g} A{r:g},{r:g} 0 1 1 {cx+r*math.cos(math.radians(40)):g},{cy+r*math.sin(math.radians(40)):g}" fill="none" stroke="{self.style.get("gray","#59636D")}" stroke-width="24" stroke-linecap="round"/><line id="{i}_needle" x1="{cx:g}" y1="{cy:g}" x2="{nx:g}" y2="{ny:g}" stroke="{self.style.get("danger","#EB5757")}" stroke-width="14" stroke-linecap="round"/><circle cx="{cx:g}" cy="{cy:g}" r="24" fill="{self.style.get("ink","#17212B")}"/><text x="{cx:g}" y="{cy+80:g}" text-anchor="middle" font-family="Arial" font-size="34" font-weight="800" fill="{self.style.get("ink","#17212B")}">{html.escape(str(p.get("label","STATUS")))}</text></g>'

    def _bar(self, i: str, p: dict[str, Any]) -> str:
        x,y,w,h=[self._num(p.get(k,d)) for k,d in (("x",150),("y",480),("width",700),("height",50))]; v=max(0,min(1,self._num(p.get("value",.7))))
        return f'<g id="{i}"><rect x="{x:g}" y="{y:g}" width="{w:g}" height="{h:g}" rx="{h/2:g}" fill="{self.style.get("muted","#D8DEE3")}"/><rect id="{i}_fill" x="{x:g}" y="{y:g}" width="{w*v:g}" height="{h:g}" rx="{h/2:g}" fill="{self._color(p.get("color",self.style.get("blue","#2D9CDB")))}"/></g>'

    def _grid(self, i: str, p: dict[str, Any]) -> str:
        x,y,w,h,step=[self._num(p.get(k,d)) for k,d in (("x",100),("y",100),("width",800),("height",800),("step",80))]; color=self._color(p.get("stroke",self.style.get("ink","#17212B"))); parts=[]
        for xx in np.arange(x,x+w+.1,step): parts.append(f'<line x1="{xx:g}" y1="{y:g}" x2="{xx:g}" y2="{y+h:g}"/>')
        for yy in np.arange(y,y+h+.1,step): parts.append(f'<line x1="{x:g}" y1="{yy:g}" x2="{x+w:g}" y2="{yy:g}"/>')
        return f'<g id="{i}" fill="none" stroke="{color}" stroke-width="2" opacity="{self._num(p.get("opacity",.18)):g}">{"".join(parts)}</g>'

    def _wave(self, i: str, p: dict[str, Any]) -> str:
        y=self._num(p.get("y",520)); amp=self._num(p.get("amplitude",70)); cycles=max(1,int(self._num(p.get("cycles",3)))); fill=self._color(p.get("fill",self.style.get("blue","#2D9CDB"))); pts=[]
        for n in range(81):
            x=100+800*n/80; yy=y+amp*math.sin(2*math.pi*cycles*n/80); pts.append((x,yy))
        d="M"+" L".join(f"{x:.2f},{yy:.2f}" for x,yy in pts)+" L900,900 L100,900 Z"
        return f'<path id="{i}" d="{d}" fill="{fill}" stroke="{self.style.get("ink","#17212B")}" stroke-width="6"/>'

    def _vector_field(self, i: str, p: dict[str, Any]) -> str:
        rows=max(2,int(self._num(p.get("rows",6)))); cols=max(2,int(self._num(p.get("cols",8)))); x0,y0,w,h=[self._num(p.get(k,d)) for k,d in (("x",120),("y",200),("width",760),("height",600))]; direction=str(p.get("direction","east")); parts=[]
        for row in range(rows):
            for col in range(cols):
                x=x0+w*col/max(1,cols-1); y=y0+h*row/max(1,rows-1); phase=(row*13+col*7)%17; length=45+phase*2; angle={"east":0,"west":math.pi,"north":-math.pi/2,"south":math.pi/2}.get(direction, math.sin((row+col)*.4)*.35); x2=x+length*math.cos(angle); y2=y+length*math.sin(angle)
                parts.append(self._arrow(f"{i}_{row}_{col}",{"x1":x,"y1":y,"x2":x2,"y2":y2,"head":12,"stroke_width":5,"stroke":p.get("stroke",self.style.get("blue","#2D9CDB"))}))
        return f'<g id="{i}">{"".join(parts)}</g>'

    def _radial_field(self, i: str, p: dict[str, Any]) -> str:
        cx,cy=self._num(p.get("cx",500)),self._num(p.get("cy",500)); rings=max(2,int(self._num(p.get("rings",7)))); parts=[]
        for n in range(1,rings+1):
            r=self._num(p.get("max_r",330))*n/rings; parts.append(f'<circle id="{i}_ring_{n}" cx="{cx:g}" cy="{cy:g}" r="{r:g}" fill="none" stroke="{self._color(p.get("stroke",self.style.get("blue","#2D9CDB")))}" stroke-width="{max(2,10-n):g}" opacity="{.85*n/rings:g}"/>')
        return f'<g id="{i}">{"".join(parts)}</g>'

    def _particles(self, i: str, p: dict[str, Any]) -> str:
        count=max(1,min(300,int(self._num(p.get("count",60))))); seed=int(self._num(p.get("seed",17))); rng=np.random.default_rng(seed); x,y,w,h=[self._num(p.get(k,d)) for k,d in (("x",80),("y",80),("width",840),("height",840))]; parts=[]
        for n in range(count):
            px=x+rng.random()*w; py=y+rng.random()*h; r=2+rng.random()*self._num(p.get("max_r",6)); parts.append(f'<circle id="{i}_{n}" cx="{px:.2f}" cy="{py:.2f}" r="{r:.2f}" fill="{self._color(p.get("fill",self.style.get("blue","#2D9CDB")))}" opacity="{.25+.7*rng.random():.3f}"/>')
        return f'<g id="{i}">{"".join(parts)}</g>'

    def _cutaway(self, i: str, p: dict[str, Any], children: list[Any], depth: int) -> str:
        cx,cy,r=self._num(p.get("cx",500)),self._num(p.get("cy",500)),self._num(p.get("r",330)); layers=p.get("layers", children or ["outer shell","middle layer","core"]); layers=layers if isinstance(layers,list) else [layers]; colors=p.get("colors",[self.style.get("cyan","#56CCF2"),self.style.get("blue","#2D9CDB"),self.style.get("warning","#F2994A"),self.style.get("danger","#EB5757")]); parts=[]
        for index, layer in enumerate(layers):
            radius=r*(len(layers)-index)/len(layers); color=self._color(colors[index%len(colors)]); name=layer.get("name",f"layer_{index+1}") if isinstance(layer,dict) else str(layer); parts.append(f'<circle id="{self._safe_id(i+"_"+name)}" cx="{cx:g}" cy="{cy:g}" r="{radius:g}" fill="{color}" stroke="{self.style.get("ink","#17212B")}" stroke-width="5"/>')
        # Remove one quadrant to make the internal layers visible.
        parts.append(f'<path d="M{cx:g},{cy:g} L{cx+r+20:g},{cy-r-20:g} L{cx+r+20:g},{cy+r+20:g} Z" fill="{self.style.get("paper","#F7F7F4")}" stroke="{self.style.get("ink","#17212B")}" stroke-width="5"/>')
        return f'<g id="{i}">{"".join(parts)}</g>'

    def _callout(self, i: str, p: dict[str, Any]) -> str:
        x,y,tx,ty=[self._num(p.get(k,d)) for k,d in (("x",650),("y",300),("target_x",500),("target_y",500))]; text=html.escape(str(p.get("text","ANNOTATION"))); return f'<g id="{i}"><path d="M{tx:g},{ty:g} L{x:g},{y:g}" fill="none" stroke="{self.style.get("ink","#17212B")}" stroke-width="5"/><circle cx="{tx:g}" cy="{ty:g}" r="9" fill="{self.style.get("blue","#2D9CDB")}"/><rect x="{x:g}" y="{y-50:g}" width="280" height="74" rx="16" fill="{self.style.get("paper","#F7F7F4")}" stroke="{self.style.get("ink","#17212B")}" stroke-width="4"/><text x="{x+140:g}" y="{y-3:g}" text-anchor="middle" font-family="Arial" font-size="26" font-weight="800" fill="{self.style.get("ink","#17212B")}">{text}</text></g>'

    def _bracket(self, i: str, p: dict[str, Any]) -> str:
        x,y1,y2=self._num(p.get("x",160)),self._num(p.get("y1",250)),self._num(p.get("y2",750)); w=self._num(p.get("width",35)); return f'<path id="{i}" d="M{x+w:g},{y1:g} H{x:g} V{y2:g} H{x+w:g}" fill="none" stroke="{self.style.get("ink","#17212B")}" stroke-width="7"/>'

    def _axis(self, i: str, p: dict[str, Any]) -> str:
        x,y,w,h=[self._num(p.get(k,d)) for k,d in (("x",160),("y",820),("width",700),("height",600))]; return f'<g id="{i}" fill="none" stroke="{self.style.get("ink","#17212B")}" stroke-width="6"><path d="M{x:g},{y-h:g} V{y:g} H{x+w:g}"/>{self._arrow(i+"_x",{"x1":x+w-40,"y1":y,"x2":x+w,"y2":y,"head":18,"stroke_width":6,"stroke":self.style.get("ink","#17212B")})}{self._arrow(i+"_y",{"x1":x,"y1":y-h+40,"x2":x,"y2":y-h,"head":18,"stroke_width":6,"stroke":self.style.get("ink","#17212B")})}</g>'

    def _spark(self, i: str, p: dict[str, Any]) -> str:
        cx,cy,r=self._num(p.get("cx",500)),self._num(p.get("cy",500)),self._num(p.get("r",130)); points=[]
        for n in range(16):
            angle=math.pi*2*n/16; rr=r if n%2==0 else r*.35; points.append(f"{cx+rr*math.cos(angle):.2f},{cy+rr*math.sin(angle):.2f}")
        return f'<polygon id="{i}" points="{" ".join(points)}" fill="{self._color(p.get("fill",self.style.get("warning","#F2994A")))}" stroke="{self.style.get("ink","#17212B")}" stroke-width="5"/>'

    def _flow_band(self, i: str, p: dict[str, Any]) -> str:
        y=self._num(p.get("y",500)); width=self._num(p.get("band_width",70)); curvature=self._num(p.get("curvature",90)); d=f"M80,{y:g} C300,{y-curvature:g} 700,{y+curvature:g} 920,{y:g} L920,{y+width:g} C700,{y+width+curvature:g} 300,{y+width-curvature:g} 80,{y+width:g} Z"; return f'<path id="{i}" d="{d}" fill="{self._color(p.get("fill",self.style.get("blue","#2D9CDB")))}" opacity="{self._num(p.get("opacity",.65)):g}"/>'

    def _color(self, value: Any) -> str:
        if value is None: return "none"
        text=str(value); return str(self.style.get(text,text))
    @staticmethod
    def _num(value: Any) -> float:
        try: return float(value)
        except Exception: return 0.0
    @staticmethod
    def _safe_id(value: str) -> str:
        return re.sub(r"[^a-zA-Z0-9_-]", "-", value).strip("-") or "layer"


class ReferenceVectorizer:
    """Advanced reference reconstruction for both SVG and raster sources."""

    def __init__(self, root: str | Path, style: dict[str, Any] | None = None, style_reference: StyleFingerprint | None = None):
        self.root = ensure_dir(root)
        self.style = {**DEFAULT_PALETTE, **(style or {})}
        self.style_reference = style_reference
        self.raster = AdvancedRasterVectorizer(self.root / "layered", self.style)

    def vectorize(self, candidate: ReferenceCandidate, collector: ReferenceCollector, asset_id: str, desired_layers: list[Any]) -> tuple[str, list[dict[str, Any]], dict[str, Any]]:
        parsed = urlparse(candidate.url)
        ext = ".svg" if candidate.mime == "image/svg+xml" else (Path(parsed.path).suffix or ".png")
        source = self.root / f"{asset_id}-reference{ext}"
        if parsed.scheme == "file":
            shutil.copy2(Path(parsed.path), source)
        else:
            collector.download(candidate, source)
        if candidate.mime == "image/svg+xml" or source.suffix.lower() == ".svg":
            text = SVGValidator.sanitize(source.read_text(encoding="utf-8", errors="ignore"))
            svg, layers = self._normalize_svg_layers(text, asset_id, desired_layers)
            metadata = {"construction": "licensed native SVG normalization"}
        else:
            svg, layers, metadata = self.raster.vectorize(
                source, asset_id=asset_id, desired_layers=desired_layers,
                color_count=8, style_reference=self.style_reference,
            )
            svg = SVGValidator.prefix_ids(svg, asset_id) if f'id="{asset_id}--' not in svg else svg
            # Prefix records exactly once.
            for layer in layers:
                if not str(layer.get("layer_id", "")).startswith(asset_id + "--"):
                    layer["layer_id"] = f"{asset_id}--{layer['layer_id']}"
        SVGValidator.validate(svg)
        metadata.update({
            "source_url": candidate.url, "source_page": candidate.source_page,
            "license": candidate.license, "attribution": candidate.attribution,
        })
        return (svg, layers, metadata)

    def _normalize_svg_layers(self, text: str, asset_id: str, desired_layers: list[Any]) -> tuple[str, list[dict[str, Any]]]:
        root = ET.fromstring(text)
        viewbox = root.attrib.get("viewBox", "")
        if not viewbox:
            width = re.sub(r"[^0-9.]", "", root.attrib.get("width", "1000")) or "1000"
            height = re.sub(r"[^0-9.]", "", root.attrib.get("height", "1000")) or "1000"
            viewbox = f"0 0 {width} {height}"
        try:
            vx, vy, vw, vh = [float(x) for x in viewbox.replace(",", " ").split()[:4]]
        except Exception:
            vx, vy, vw, vh = 0, 0, 1000, 1000
        scale = min(1000 / max(vw, 1), 1000 / max(vh, 1))
        ox = (1000 - vw * scale) / 2 - vx * scale
        oy = (1000 - vh * scale) / 2 - vy * scale
        drawable = [element for element in root.iter() if element.tag.split("}")[-1] in DRAWABLE - {"g"}]
        if not drawable:
            raise ValueError("Reference SVG has no drawable vector elements")
        # Group by visual signature, but retain meaningful source IDs when present.
        groups: dict[str, list[str]] = {}
        for element in drawable:
            source_id = element.attrib.get("id", "")
            signature = source_id or "|".join([element.attrib.get("fill", ""), element.attrib.get("stroke", ""), element.attrib.get("style", "")])
            groups.setdefault(signature, []).append(ET.tostring(element, encoding="unicode"))
        ordered = sorted(groups.items(), key=lambda item: len("".join(item[1])), reverse=True)
        requested = []
        for item in desired_layers:
            requested.append(str(item.get("name", item.get("id", "layer"))) if isinstance(item, dict) else str(item))
        markup, layers = [], []
        for index, (signature, items) in enumerate(ordered):
            raw_name = requested[index] if index < len(requested) else (signature or f"detail_{index+1}")
            name = ProgramCompiler._safe_id(raw_name)
            inner = f'<g transform="translate({ox:.6f} {oy:.6f}) scale({scale:.8f})">{"".join(items)}</g>'
            markup.append(f'<g id="{name}">{inner}</g>')
            layers.append({"layer_id": f"{asset_id}--{name}", "role": raw_name, "bbox": [0, 0, 1000, 1000]})
        svg = f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1000 1000">{"".join(markup)}</svg>'
        svg = SVGValidator.prefix_ids(svg, asset_id)
        return svg, layers


class GeographicBuilder:
    def __init__(self, style: dict[str, Any]): self.style=style
    def build(self, asset_id: str, intent: VisualIntent) -> tuple[str,list[dict[str,Any]],dict[str,Any]]:
        from mpl_toolkits.basemap import Basemap
        from shapely.geometry import Polygon
        from shapely.validation import make_valid
        lon=float(intent.attributes.get("center_longitude",intent.attributes.get("lon_0",15))); lat=float(intent.attributes.get("center_latitude",intent.attributes.get("lat_0",12)))
        bm=Basemap(projection="ortho",lon_0=lon,lat_0=lat,resolution="l"); center=500.; radius=342.
        def project(x,y): return center-radius+(x/bm.xmax)*radius*2, center+radius-(y/bm.ymax)*radius*2
        def path_geom(geom):
            parts=[]; polys=[geom] if geom.geom_type=="Polygon" else list(geom.geoms)
            for poly in polys:
                pts=[project(x,y) for x,y in poly.exterior.coords]
                if len(pts)>=4: parts.append("M "+" L ".join(f"{x:.2f},{y:.2f}" for x,y in pts)+" Z")
            return " ".join(parts)
        land=[]; lakes=[]
        for (xs,ys),kind in zip(bm.coastpolygons,bm.coastpolygontypes):
            try:
                geom=Polygon(zip(xs,ys)); geom=make_valid(geom) if not geom.is_valid else geom; geom=geom.simplify(11000,preserve_topology=True); d=path_geom(geom)
                if d: (land if kind==1 else lakes).append(d)
            except Exception: continue
        ink=self.style.get("ink","#17212B"); blue=self.style.get("cyan","#56CCF2"); gray=self.style.get("gray","#59636D"); white=self.style.get("white","#FFFFFF")
        land_paths="".join('<path d="%s"/>' % d for d in land)
        lake_paths="".join('<path d="%s"/>' % d for d in lakes)
        atmo_blue=self.style.get("blue","#2D9CDB")
        layers=[
            ("ocean",f'<circle cx="500" cy="500" r="342" fill="{blue}" stroke="{ink}" stroke-width="8"/>'),
            ("landmasses",f'<g fill="{gray}" stroke="{ink}" stroke-width="3">{land_paths}</g>'),
            ("lakes",f'<g fill="{white}" stroke="{ink}" stroke-width="2">{lake_paths}</g>'),
            ("atmosphere",f'<circle cx="500" cy="500" r="365" fill="none" stroke="{atmo_blue}" stroke-width="12" opacity=".35"/>'),
            ("clouds",f'<g fill="{white}" stroke="{ink}" stroke-width="3" opacity=".86"><path d="M247 414 C274 384 318 384 340 412 C365 392 405 399 418 429 C430 458 397 474 366 470 L281 470 C248 470 229 443 247 414 Z"/><path d="M589 615 C619 584 664 588 686 620 C713 599 753 611 763 642 C773 672 740 688 707 681 L622 681 C590 681 570 649 589 615 Z"/></g>'),
        ]
        paper=self.style.get("paper","#F7F7F4")
        _layer_svg="".join('<g id="%s">%s</g>' % (name,content) for name,content in layers)
        svg=f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1000 1000"><rect width="1000" height="1000" fill="{paper}"/>{_layer_svg}</svg>'
        svg=SVGValidator.prefix_ids(svg,asset_id); SVGValidator.validate(svg)
        records=[{"layer_id":f"{asset_id}--{name}","role":name,"bbox":[0,0,1000,1000]} for name,_ in layers]
        return svg,records,{"projection":"orthographic","center_longitude":lon,"center_latitude":lat,"land_paths":len(land)}


class DynamicSVGEngine:
    SYSTEM = """You are a vector construction compiler planner. Preserve the visual director's creative intent.
Do not force an object into a fixed taxonomy. Choose any combination of data geometry, licensed reference
reconstruction, visual-program synthesis, and hybrid overlays. Real objects must remain recognizable.
Return JSON. Compound visual operations may be invented; the compiler will recursively decompose them."""

    def __init__(self, llm: LLMRouter, references: ReferenceCollector, config: dict[str,Any], root: str|Path, style_reference: StyleFingerprint | None = None):
        self.llm=llm; self.references=references; self.config=config; self.root=ensure_dir(root); self.assets_root=ensure_dir(self.root/"assets"); self.cache_root=ensure_dir(self.root/"cache")
        self.style={
            "paper":"#F7F7F4","ink":"#17212B","blue":"#4390C4","cyan":"#9BC9E1",
            "gray":"#858B90","muted":"#B9BDC2","white":"#FFFFFF","warning":"#F3A53B","danger":"#D73232","dark":"#4B555D",
            **config.get("style",{}),
        }
        self.style_reference = style_reference
        self.vectorizer=ReferenceVectorizer(self.cache_root/"traces", self.style, style_reference); self.geo=GeographicBuilder(self.style)

    def build(self, intent: VisualIntent, force: bool=False) -> AssetRecord:
        asset_id=ProgramCompiler._safe_id(intent.intent_id or f"asset-{hash_value(intent.description,10)}")
        record_path=self.cache_root/f"{hash_value(intent.model_dump(mode='json'))}.json"
        if record_path.exists() and not force:
            record=AssetRecord.model_validate(load_json(record_path));
            if Path(record.svg_path).exists(): return record
        plan=self.plan(intent,force=force)
        warnings=[]; sources=[]; licenses=[]; method=[]; svg=None; layers=[]; metadata={}
        text=(intent.description+" "+str(intent.kind)+" "+json.dumps(plan.data_requests,ensure_ascii=False)).lower()
        wants_geo=any(token in text for token in ("earth","globe","world map","coastline","geographic","orthographic projection"))
        if wants_geo:
            try:
                svg,layers,metadata=self.geo.build(asset_id,intent); method.append("data-driven-geographic-projection")
            except Exception as exc: warnings.append(f"geographic builder failed: {exc}")
        if svg is None and self.config.get("enable_reference_trace", True):
            refs = self.references.collect(intent, limit=int(self.config.get("reference_limit", 10)), force=force)
            candidates_built = []
            for candidate in refs[:int(self.config.get("reference_attempts", 5))]:
                try:
                    candidate_svg, candidate_layers, refmeta = self.vectorizer.vectorize(
                        candidate, self.references, asset_id, intent.desired_layers
                    )
                    # Prefer detailed, layered, licensed reconstructions instead of the first search result.
                    path_count = candidate_svg.lower().count("<path")
                    group_count = candidate_svg.lower().count("<g")
                    source_score = float(getattr(candidate, "score", 0.0))
                    quality_score = source_score * 2.0 + min(2.5, math.log1p(path_count) / 2.0) + min(1.5, group_count / 10.0)
                    candidates_built.append((quality_score, candidate_svg, candidate_layers, refmeta, candidate))
                except Exception as exc:
                    warnings.append(f"reference rejected ({candidate.title}): {exc}")
            if candidates_built:
                _, svg, layers, refmeta, selected = max(candidates_built, key=lambda item: item[0])
                sources.append(refmeta.get("source_page") or refmeta.get("source_url"))
                licenses.append(refmeta.get("license", ""))
                metadata.update(refmeta)
                metadata["selected_reference_title"] = selected.title
                method.append("best-of-multiple licensed reference vector reconstruction")
        if svg is None and self.config.get("enable_generated_reference", False):
            generated_path = self.cache_root / "generated" / f"{asset_id}.png"
            prompt = self._reference_generation_prompt(intent)
            generated = self.llm.generate_reference_image(prompt=prompt, output_path=generated_path, force=force)
            if generated:
                try:
                    from .references import ReferenceCandidate
                    candidate = ReferenceCandidate(
                        title=f"generated reference for {intent.description}",
                        url=generated.resolve().as_uri(), mime="image/png",
                        license="generated for this pipeline", attribution="configured image provider",
                        source_page="",
                        score=1.0,
                    )
                    svg, layers, refmeta = self.vectorizer.vectorize(
                        candidate, self.references, asset_id, intent.desired_layers
                    )
                    metadata.update(refmeta)
                    method.append("generated-reference layered vector reconstruction")
                except Exception as exc:
                    warnings.append(f"generated reference vectorization failed: {exc}")
        overlay_program=plan.visual_program if isinstance(plan.visual_program, dict) else {}
        if svg is None or overlay_program:
            if not overlay_program:
                overlay_program=self.synthesize_program(intent,plan,force=force)
            compiler=ProgramCompiler(self.style,resolver=lambda node:self.decompose_unknown(node,intent,force=force))
            program_svg,program_layers=compiler.compile(overlay_program,prefix=asset_id+("-overlay" if svg else ""))
            warnings.extend(compiler.warnings)
            if svg is None:
                svg,layers=program_svg,program_layers; method.append("open-visual-program-synthesis")
            else:
                svg=self._compose(svg,program_svg,asset_id); layers.extend(program_layers); method.append("hybrid-semantic-overlay")
        if svg is None: raise RuntimeError(f"No construction strategy succeeded for: {intent.description}")
        svg=SVGValidator.sanitize(svg); SVGValidator.validate(svg)
        svg_path=self.assets_root/f"{asset_id}.svg"; svg_path.write_text(svg,encoding="utf-8")
        preview_path=self.assets_root/f"{asset_id}.png"
        try:
            import cairosvg; cairosvg.svg2png(bytestring=svg.encode(),write_to=str(preview_path),output_width=1000,output_height=1000)
        except Exception as exc: warnings.append(f"preview render skipped: {exc}")
        manifest_path=self.assets_root/f"{asset_id}.json"
        record=AssetRecord(asset_id=asset_id,intent_id=intent.intent_id,description=intent.description,method=" + ".join(method),svg_path=str(svg_path),preview_path=str(preview_path) if preview_path.exists() else "",manifest_path=str(manifest_path),source_urls=[s for s in sources if s],licenses=[s for s in licenses if s],semantic_layers=layers,quality={},warnings=warnings,asset_hash=sha256_file(svg_path),raw_build_plan=plan.model_dump(mode="json"))
        save_json(manifest_path,record); save_json(record_path,record); return record

    def plan(self,intent:VisualIntent,force:bool=False)->BuildPlan:
        fallback={"plan_id":f"plan-{hash_value(intent.description,10)}","intent_id":intent.intent_id,"strategy":"hybrid","rationale":"Try data/reference reconstruction before program synthesis.","reference_queries":intent.reference_requests or [intent.description],"data_requests":[],"visual_program":{},"layer_plan":intent.desired_layers,"animation_plan":intent.motion_intents,"acceptance_criteria":{"recognizable":True,"semantic_layers":True,"no_raster_embed":True}}
        raw=self.llm.generate_json(system=self.SYSTEM,prompt=f"""Plan how to construct this visual intent as an advanced, editable SVG:\n{json.dumps(intent.model_dump(mode='json'),ensure_ascii=False,indent=2)}\n\nAvailable capabilities include geographic/coordinate data, licensed reference tracing, raster segmentation, color-layer vectorization, procedural geometry, diagrams, fields, particles, raw path fragments, and hybrid composition. You may invent compound operations. Return BuildPlan JSON.""",namespace="svg_plan",fallback=fallback,force=force)
        if not isinstance(raw,dict): raw=fallback
        raw.setdefault("intent_id",intent.intent_id); raw.setdefault("plan_id",fallback["plan_id"]); raw.setdefault("raw_llm_output",raw.copy())
        return BuildPlan.model_validate(raw)

    def synthesize_program(self,intent:VisualIntent,plan:BuildPlan,force:bool=False)->dict[str,Any]:
        fallback=self._fallback_program(intent)
        raw=self.llm.generate_json(system=self.SYSTEM,prompt=f"""Write an open visual program for a 1000x1000 SVG.\nIntent:\n{json.dumps(intent.model_dump(mode='json'),ensure_ascii=False,indent=2)}\nBuild plan:\n{json.dumps(plan.model_dump(mode='json'),ensure_ascii=False,indent=2)}\n\nAtomic operations currently understood: {sorted(ProgramCompiler.ATOMIC_OPS)}. You may use compound operations not in this list; the engine will ask you to decompose them. Every important independently moving part must have a stable id. Prefer multiple meaningful layers over one monolithic path. Return {{canvas, background, nodes}} JSON.""",namespace="visual_program",fallback=fallback,force=force)
        if not isinstance(raw,dict): raw=fallback
        if not any(key in raw for key in ("nodes","components","layers")): raw=fallback
        return raw

    def decompose_unknown(self,node:dict[str,Any],intent:VisualIntent,force:bool=False)->list[dict[str,Any]]:
        fallback=[{"op":"cutaway","id":str(node.get("id","structure")),"params":{"layers":["outer structure","functional layer","core"],"role":str(node.get("op","compound structure"))}},{"op":"callout","id":str(node.get("id","structure"))+"_label","params":{"text":str(node.get("op","STRUCTURE")).replace("_"," ").upper()[:28],"x":640,"y":240,"target_x":560,"target_y":430}},{"op":"vector_field","id":str(node.get("id","structure"))+"_causal_field","params":{"rows":5,"cols":7,"direction":"east","stroke":"blue"}}]
        raw=self.llm.generate_json(system=self.SYSTEM,prompt=f"""Decompose this unsupported compound visual operation into supported atomic vector operations without losing its meaning.\nCompound node: {json.dumps(node,ensure_ascii=False,indent=2)}\nOverall intent: {intent.description}\nSupported atomic operations: {sorted(ProgramCompiler.ATOMIC_OPS)}\nReturn a JSON array of nodes. Use raw_svg_fragment or path only when geometry cannot be expressed otherwise.""",namespace="op_decomposition",fallback=fallback,force=force)
        if isinstance(raw,dict): raw=raw.get("nodes",raw.get("components",raw.get("layers",fallback)))
        return raw if isinstance(raw,list) and raw else fallback

    def _reference_generation_prompt(self, intent: VisualIntent) -> str:
        return f"""Create one original, isolated scientific editorial illustration for vector tracing.
Subject: {intent.description}
Important parts: {json.dumps(intent.desired_layers, ensure_ascii=False)}
Relationships: {json.dumps(intent.relationships, ensure_ascii=False)}
Style: off-white background or transparent background, strong dark hand-inked outline, flat geometric
shapes, limited grayscale plus blue accents and sparse red warning accents, polygonal internal shading,
clear silhouette, no gradients, no photorealism, no text, no watermark, no border. Keep each important
part visually separable by color so it can become an independent SVG layer. The object must be accurate
and immediately recognizable, with illustration detail comparable to professional scientific explainer art."""

    def _fallback_program(self,intent:VisualIntent)->dict[str,Any]:
        # A meaningful scientific cutaway + causal field, never a silent rectangle/blob.
        layers=[]
        for item in intent.desired_layers[:5]: layers.append(str(item.get("name",item.get("id","layer"))) if isinstance(item,dict) else str(item))
        if not layers: layers=["outer structure","active mechanism","core state"]
        return {"canvas":[1000,1000],"background":"paper","nodes":[{"op":"grid","id":"scientific_grid","params":{"opacity":.12,"step":100}},{"op":"cutaway","id":"hero_structure","params":{"layers":layers,"cx":450,"cy":500,"r":300}},{"op":"vector_field","id":"causal_field","params":{"x":110,"y":210,"width":760,"height":580,"rows":6,"cols":8,"direction":"east"}},{"op":"callout","id":"intent_label","params":{"text":intent.description[:32].upper(),"x":620,"y":205,"target_x":570,"target_y":390}},{"op":"gauge","id":"state_dashboard","params":{"cx":790,"cy":795,"r":120,"value":.68,"label":"SYSTEM STATE"}}]}

    @staticmethod
    def _compose(base_svg:str,overlay_svg:str,asset_id:str)->str:
        base_root=ET.fromstring(base_svg); overlay_root=ET.fromstring(overlay_svg)
        overlay_group=ET.Element("g",{"id":f"{asset_id}--semantic-overlays"})
        for child in list(overlay_root):
            if child.attrib.get("id","").endswith("--background"): continue
            overlay_group.append(child)
        base_root.append(overlay_group)
        return ET.tostring(base_root,encoding="unicode")


In [ ]:
%%writefile /content/scientific_motion_studio_v5/src/scistudio_v5/critic.py
# Cell 16: scistudio_v5/critic.py
from __future__ import annotations

import json
from pathlib import Path
from typing import Any
from xml.etree import ElementTree as ET

import numpy as np
from PIL import Image, ImageOps, ImageDraw

from .benchmark import BenchmarkAnalyzer, StyleFingerprint
from .illustration import BenchmarkQualityGate, IllustrationQualityError
from .llm import LLMRouter
from .schemas import AssetRecord, QualityReport, VisualIntent
from .utils import ensure_dir, save_json


class VisualCritic:
    """Multimodal + benchmark critic for illustration-grade SVGs.

    The critic does not accept a file merely because it is valid SVG. Concrete
    visuals must be recognizable, layered, compositionally dense enough, and
    stylistically comparable to the selected reference archetype.
    """

    def __init__(
        self,
        llm: LLMRouter,
        config: dict[str, Any],
        root: str | Path,
        benchmark_analyzer: BenchmarkAnalyzer | None = None,
        benchmark_profiles: list[StyleFingerprint] | None = None,
        benchmark_contact_sheet: str | Path | None = None,
    ):
        self.llm = llm
        self.config = config
        self.root = ensure_dir(root)
        self.analyzer = benchmark_analyzer
        self.profiles = benchmark_profiles or []
        self.contact_sheet = Path(benchmark_contact_sheet) if benchmark_contact_sheet else None
        self.gate = BenchmarkQualityGate(
            benchmark_analyzer, self.profiles, config
        ) if benchmark_analyzer and self.profiles else None

    def review(self, intent: VisualIntent, record: AssetRecord, force: bool = False) -> QualityReport:
        local = self._local_metrics(record)
        benchmark = self.gate.evaluate(record) if self.gate else {
            "passed": local["structural"] >= 0.9 and local["visual_density"] >= 0.16,
            "score": 0.0,
            "failures": [],
        }
        fallback_scores = {
            "recognizability": 0.60 if "reference" not in str(record.method) else 0.79,
            "scientific_accuracy": 0.64,
            "semantic_layer_quality": local["semantic_layers"],
            "composition": local["composition"],
            "style_consistency": float(benchmark.get("score", 0.0)),
            "illustration_detail": local["illustration_detail"],
            "animation_readiness": local["structural"],
        }
        fallback = {
            "passed": bool(benchmark.get("passed") and min(fallback_scores.values()) >= 0.60),
            "scores": fallback_scores,
            "warnings": [],
            "failures": list(benchmark.get("failures", [])),
            "repair_plan": [],
            "media": {**local, "benchmark": benchmark},
        }
        if not record.preview_path or not Path(record.preview_path).exists():
            report = QualityReport.model_validate(fallback)
            report.passed = False
            report.failures.append("No preview image was available for visual critique.")
            save_json(self.root / f"{record.asset_id}-quality.json", report)
            return report

        critique_path = self._comparison_board(record)
        prompt = f"""Evaluate the generated SVG illustration against the intended visual and the reference-style examples shown beside it.
Intent: {json.dumps(intent.model_dump(mode='json'), ensure_ascii=False)}
Construction method: {record.method}
Semantic layers: {json.dumps(record.semantic_layers, ensure_ascii=False)}
Local and benchmark metrics: {json.dumps(fallback['media'], ensure_ascii=False)}

The first/large panel is the generated illustration. Other panels, if present, are style-level references only.
Do not require copied geometry. Evaluate whether the generated illustration reaches the same professional level:
- immediately recognizable subject and scientifically plausible structure
- rich editorial illustration, not a generic icon or dashboard-only diagram
- meaningful internal parts and flat polygonal shading
- coherent black/gray/blue palette with controlled accent colors
- strong silhouette and outline hierarchy
- semantic groups suitable for independent animation
- balanced composition and deliberate negative space

Return JSON with passed, scores from 0 to 1 for recognizability, scientific_accuracy,
reference_fidelity, semantic_layer_quality, composition, style_consistency,
illustration_detail, animation_readiness, plus warnings, failures, and an actionable repair_plan.
Be strict. Scores below 0.78 should identify exact geometric or compositional changes."""
        raw = self.llm.critique_image(
            image_path=critique_path,
            prompt=prompt,
            namespace="illustration_critic",
            fallback=fallback,
            force=force,
        )
        if not isinstance(raw, dict):
            raw = fallback
        raw.setdefault("media", {}).update(fallback["media"])
        raw.setdefault("scores", {}).update({k: v for k, v in fallback_scores.items() if k not in raw.get("scores", {})})
        report = QualityReport.model_validate(raw)
        threshold = float(self.config.get("acceptance_threshold", 0.78))
        required = [
            "recognizability", "scientific_accuracy", "semantic_layer_quality",
            "composition", "style_consistency", "illustration_detail", "animation_readiness",
        ]
        score_values = [float(report.scores.get(key, 0.0)) for key in required]
        vision_available = self.llm.available("gemini") or self.llm.available("openai")
        if not vision_available:
            # Honest degraded mode: with no vision model we cannot score
            # recognizability/accuracy, so fall back to the deterministic
            # structural + style benchmark gate instead of the hardcoded low
            # placeholder scores. This lets a run still produce a video, and
            # only rejects assets the benchmark gate itself flags as icon-like.
            report.passed = bool(benchmark.get("passed", False))
            report.warnings.append("Vision critic unavailable; accepted on structural/style benchmark only.")
        else:
            report.passed = bool(
                report.passed
                and benchmark.get("passed", True)
                and score_values
                and min(score_values) >= threshold
            )
        if not benchmark.get("passed", True):
            for failure in benchmark.get("failures", []):
                if failure not in report.failures:
                    report.failures.append(failure)
        save_json(self.root / f"{record.asset_id}-quality.json", report)
        return report

    def refine(self, intent: VisualIntent, record: AssetRecord, engine, force: bool = False) -> tuple[AssetRecord, QualityReport]:
        max_rounds = int(self.config.get("max_refinement_rounds", 5))
        strict = bool(self.config.get("strict_quality_gate", True))
        report = self.review(intent, record, force=force)
        current = record
        strategy_ladder = [
            "add more semantic layers and correct proportions",
            "replace weak procedural geometry with a stronger licensed reference reconstruction",
            "compose multiple references and add internal polygonal shading",
            "use a generated isolated illustration reference, then perform layered vector reconstruction",
            "rebuild the full asset with a more detailed open visual program and explicit silhouette repair",
        ]
        for round_index in range(max_rounds):
            if report.passed:
                break
            revised = intent.model_copy(deep=True)
            repairs = report.repair_plan or report.failures or report.warnings
            revised.visual_constraints = {
                **revised.visual_constraints,
                "critic_round": round_index + 1,
                "critic_scores": report.scores,
                "required_repairs": repairs,
                "quality_escalation": strategy_ladder[min(round_index, len(strategy_ladder) - 1)],
                "reject_generic_abstraction": True,
                "minimum_semantic_parts": int(self.config.get("minimum_semantic_parts", 7)),
                "minimum_internal_shading_regions": int(self.config.get("minimum_internal_shading_regions", 3)),
            }
            revised.reference_requests = list(revised.reference_requests) + [
                {"query": revised.description, "purpose": "improve recognizability"},
                {"query": f"{revised.description} flat scientific editorial illustration", "purpose": "style and detail"},
                {"query": f"{revised.description} labeled diagram silhouette", "purpose": "accurate parts"},
            ]
            current = engine.build(revised, force=True)
            report = self.review(revised, current, force=True)
        current.quality = report.model_dump(mode="json")
        save_json(current.manifest_path, current)
        if strict and not report.passed:
            raise IllustrationQualityError(
                f"Illustration quality gate rejected '{intent.description}' after {max_rounds} refinements. "
                f"Failures: {report.failures}"
            )
        return current, report

    def _comparison_board(self, record: AssetRecord) -> Path:
        generated = Image.open(record.preview_path).convert("RGB")
        max_w = 1100
        generated.thumbnail((max_w, 800))
        refs = None
        if self.contact_sheet and self.contact_sheet.exists():
            refs = Image.open(self.contact_sheet).convert("RGB")
            refs.thumbnail((max_w, 620))
        width = max(generated.width, refs.width if refs else 0, 900)
        height = generated.height + (refs.height if refs else 0) + 100
        canvas = Image.new("RGB", (width, height), (245, 245, 242))
        draw = ImageDraw.Draw(canvas)
        canvas.paste(generated, ((width - generated.width) // 2, 42))
        draw.text((18, 12), "GENERATED ILLUSTRATION", fill=(23, 33, 43))
        if refs:
            y = generated.height + 82
            canvas.paste(refs, ((width - refs.width) // 2, y))
            draw.text((18, generated.height + 52), "REFERENCE STYLE ARCHETYPES", fill=(23, 33, 43))
        output = self.root / f"{record.asset_id}-comparison.png"
        canvas.save(output)
        return output

    @staticmethod
    def _local_metrics(record: AssetRecord) -> dict[str, float]:
        svg_path = Path(record.svg_path)
        text = svg_path.read_text(encoding="utf-8", errors="ignore")
        try:
            root = ET.fromstring(text)
            drawable = [e for e in root.iter() if e.tag.split("}")[-1] in {"path","circle","ellipse","rect","line","polygon","polyline","text"}]
            paths = [e for e in root.iter() if e.tag.split("}")[-1] == "path"]
            groups = [e for e in root.iter() if e.tag.split("}")[-1] == "g"]
            structural = 1.0 if drawable and "<image" not in text.lower() else 0.0
            path_chars = sum(len(e.attrib.get("d", "")) for e in paths)
        except Exception:
            drawable, paths, groups = [], [], []
            structural, path_chars = 0.0, 0
        visual_density = 0.0
        composition = 0.5
        if record.preview_path and Path(record.preview_path).exists():
            image = np.asarray(Image.open(record.preview_path).convert("RGB").resize((256,256)), dtype=np.float32)
            corners = np.vstack([image[:12,:12].reshape(-1,3), image[-12:,-12:].reshape(-1,3)])
            background = np.median(corners, axis=0)
            diff = np.linalg.norm(image - background, axis=2)
            visual_density = float(np.mean(diff > 18))
            ys, xs = np.where(diff > 18)
            if len(xs):
                coverage_x = (xs.max() - xs.min() + 1) / 256
                coverage_y = (ys.max() - ys.min() + 1) / 256
                composition = float(min(1.0, (coverage_x + coverage_y) / 1.35))
        illustration_detail = min(1.0, (
            min(1.0, len(paths) / 24.0) * 0.35
            + min(1.0, len(groups) / 10.0) * 0.25
            + min(1.0, path_chars / 3500.0) * 0.25
            + min(1.0, visual_density / 0.42) * 0.15
        ))
        return {
            "structural": structural,
            "drawable_count": float(len(drawable)),
            "path_count": float(len(paths)),
            "group_count": float(len(groups)),
            "path_character_count": float(path_chars),
            "visual_density": visual_density,
            "composition": composition,
            "semantic_layers": min(1.0, len(record.semantic_layers) / 8.0),
            "illustration_detail": illustration_detail,
        }


In [ ]:
%%writefile /content/scientific_motion_studio_v5/src/scistudio_v5/script_critic.py
# Cell: scistudio_v5/script_critic.py
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

from .llm import LLMRouter
from .schemas import ResearchPack, ScriptPackage
from .utils import ensure_dir, save_json


STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "if", "then", "of", "to", "in", "on", "at", "for",
    "is", "are", "was", "were", "it", "its", "this", "that", "these", "those", "as", "by",
    "with", "from", "into", "would", "will", "can", "could", "so", "you", "your", "we", "our",
    "they", "their", "he", "she", "his", "her", "not", "no", "yes", "here", "there", "what",
}


class ScriptCritic:
    """Retention-first script quality gate — the narration counterpart to the
    visual critic. It is deterministic first (hook, information gain, pacing,
    curiosity, payoff), then optionally sharpened by an LLM. Weak scripts are
    rewritten instead of silently narrated.
    """

    SYSTEM = """You are a ruthless retention editor for science shorts (Kurzgesagt-level).
You judge whether a script hooks in the first 3 seconds, never stalls, escalates stakes,
and pays off. You return strict JSON only and propose concrete beat-level rewrites."""

    def __init__(self, llm: LLMRouter, config: dict[str, Any], root: str | Path):
        self.llm = llm
        self.config = config
        self.root = ensure_dir(root)

    # -- deterministic metrics ------------------------------------------------
    @staticmethod
    def _content_words(text: str) -> list[str]:
        return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOPWORDS and len(w) > 2]

    def _metrics(self, script: ScriptPackage) -> dict[str, float]:
        beats = script.beats or []
        lines = [b.spoken_line for b in beats]
        first = lines[0] if lines else ""
        # Hook: question, number, or tension word in the first beat, and not too long.
        hook_signals = bool(re.search(r"\?|\d", first)) or bool(
            re.search(r"\b(imagine|what|suddenly|never|stop|vanish|collapse|instant|impossible)\b", first.lower())
        )
        hook_len_ok = 3 <= len(first.split()) <= 22
        hook = float(0.6 * hook_signals + 0.4 * hook_len_ok)
        # Information gain: fraction of beats that introduce new content words.
        seen: set[str] = set()
        novel = 0
        for line in lines:
            words = set(self._content_words(line))
            if len(words - seen) >= max(2, int(0.4 * max(1, len(words)))):
                novel += 1
            seen |= words
        info_gain = novel / max(1, len(lines))
        # Pacing: total duration within target window and no over-long single beat.
        lo = float(self.config.get("target_duration_min", 40.0))
        hi = float(self.config.get("target_duration_max", 60.0))
        dur = script.estimated_duration_s or sum(b.duration_s for b in beats)
        pacing = 1.0 if lo <= dur <= hi else max(0.0, 1.0 - abs(dur - (lo + hi) / 2) / (hi))
        longest = max((len(l.split()) for l in lines), default=0)
        no_stall = 1.0 if longest <= int(self.config.get("max_beat_words", 26)) else 0.6
        # Escalation: later beats reference consequence/scale language.
        esc_terms = ("then", "next", "worse", "more", "every", "entire", "cascade", "chain", "finally", "result")
        escalation = min(1.0, sum(any(t in l.lower() for t in esc_terms) for l in lines[len(lines)//2:]) / max(1, len(lines) - len(lines)//2) + 0.2)
        # Payoff: closing resolves / states the lesson.
        closing = (script.closing or (lines[-1] if lines else "")).lower()
        payoff = float(bool(re.search(r"\b(lesson|because|connected|why|matters|truth|reason|result|is that)\b", closing)))
        beat_count_ok = 1.0 if 6 <= len(beats) <= 11 else 0.6
        return {
            "hook_strength": round(hook, 3),
            "information_gain": round(info_gain, 3),
            "pacing": round(pacing, 3),
            "no_stall": round(no_stall, 3),
            "escalation": round(escalation, 3),
            "payoff": round(payoff, 3),
            "structure": round(beat_count_ok, 3),
            "estimated_duration_s": round(dur, 2),
            "beat_count": len(beats),
        }

    # -- review ---------------------------------------------------------------
    def review(self, script: ScriptPackage, research: ResearchPack, force: bool = False) -> dict[str, Any]:
        metrics = self._metrics(script)
        keys = ["hook_strength", "information_gain", "pacing", "no_stall", "escalation", "payoff", "structure"]
        structural_min = min(metrics[k] for k in keys)
        threshold = float(self.config.get("acceptance_threshold", 0.6))
        vision_or_text_llm = self.llm.available("gemini") or self.llm.available("openai") or self.llm.available("local")

        report: dict[str, Any] = {"metrics": metrics, "scores": {k: metrics[k] for k in keys}, "failures": [], "repair_plan": []}
        for k in keys:
            if metrics[k] < threshold:
                report["failures"].append(f"{k}={metrics[k]:.2f} < {threshold:.2f}")

        if not vision_or_text_llm:
            report["passed"] = structural_min >= threshold
            report["mode"] = "structural-only (no LLM)"
            save_json(self.root / f"{script.script_hash or 'script'}-quality.json", report)
            return report

        prompt = f"""Score this science-short script for retention and propose rewrites.
Deterministic metrics already computed: {json.dumps(metrics, ensure_ascii=False)}
Script: {json.dumps(script.model_dump(mode='json', exclude={'raw_llm_output'}), ensure_ascii=False)}

Return JSON: {{"passed": bool, "scores": {{"hook_strength":0-1,"information_gain":0-1,"curiosity":0-1,
"escalation":0-1,"payoff":0-1,"clarity":0-1,"naturalness":0-1}}, "failures": [..],
"repair_plan": [{{"beat_id":"B0x","rewrite":"stronger spoken line"}}]}}
Be strict: a first line that does not create an open loop in 3 seconds fails hook_strength."""
        raw = self.llm.generate_json(
            system=self.SYSTEM, prompt=prompt, namespace="script_critic",
            fallback={"passed": structural_min >= threshold, "scores": report["scores"], "failures": report["failures"], "repair_plan": []},
            force=force,
        )
        if not isinstance(raw, dict):
            raw = {"passed": structural_min >= threshold, "scores": report["scores"], "failures": report["failures"], "repair_plan": []}
        raw.setdefault("metrics", metrics)
        scores = {**{k: metrics[k] for k in keys}, **(raw.get("scores") or {})}
        llm_min = min([float(v) for v in scores.values()] or [0.0])
        raw["passed"] = bool(raw.get("passed", True) and structural_min >= threshold and llm_min >= threshold)
        raw["scores"] = scores
        save_json(self.root / f"{script.script_hash or 'script'}-quality.json", raw)
        return raw

    # -- refine ---------------------------------------------------------------
    def refine(self, script: ScriptPackage, research: ResearchPack, director, force: bool = False) -> tuple[ScriptPackage, dict[str, Any]]:
        max_rounds = int(self.config.get("max_rounds", 2))
        strict = bool(self.config.get("strict", False))
        report = self.review(script, research, force=force)
        for _ in range(max_rounds):
            if report.get("passed"):
                break
            repairs = report.get("repair_plan") or report.get("failures")
            if self.llm.available("gemini") or self.llm.available("openai") or self.llm.available("local"):
                prompt = f"""Rewrite this script to fix these problems while keeping the facts and evidence refs.
Problems: {json.dumps(repairs, ensure_ascii=False)}
Current script: {json.dumps(script.model_dump(mode='json', exclude={'raw_llm_output'}), ensure_ascii=False)}
Return the same JSON shape (topic, title, hook, beats[], closing). Keep 7-10 beats, {self.config.get('target_duration_min',40)}-{self.config.get('target_duration_max',60)}s."""
                raw = self.llm.generate_json(system=director.SCRIPT_SYSTEM, prompt=prompt, namespace="script_refine",
                                             fallback=script.model_dump(mode="json"), force=True)
                script = director._normalize_script(raw, script, script.topic)
            else:
                break
            report = self.review(script, research, force=True)
        if strict and not report.get("passed"):
            report["note"] = "script did not reach retention threshold after refinement"
        return script, report


In [ ]:
%%writefile /content/scientific_motion_studio_v5/src/scistudio_v5/motion.py
# Cell 17: scistudio_v5/motion.py
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

from .llm import LLMRouter
from .schemas import AssetRecord, CompiledScene, MotionIntent, MotionTrack, SceneRequest, Storyboard
from .utils import ensure_dir, save_json


class MotionCompiler:
    """Compiles free-form physical behavior into generic property tracks.

    Motion operations are open-ended. The compiler asks the LLM to decompose a
    physical description into target/property/keyframe tracks understood by the
    renderer. It never validates against a fixed animation enum.
    """

    SYSTEM = """You are a motion compiler for scientific SVG animation.
Translate natural physical behavior into deterministic keyframe tracks. Preserve causality.
Targets must refer to supplied semantic layer IDs when possible. Return JSON only."""

    def __init__(self, llm: LLMRouter, config: dict[str, Any], root: str | Path):
        self.llm = llm
        self.config = config
        self.root = ensure_dir(root)

    def compile(self, storyboard: Storyboard, assets_by_intent: dict[str, AssetRecord], force: bool = False) -> list[CompiledScene]:
        scenes: list[CompiledScene] = []
        for scene_index, scene in enumerate(storyboard.scenes):
            placements = self._placements(scene, assets_by_intent)
            tracks = self._tracks(scene, placements, force=force)
            camera_tracks = self._camera_tracks(scene)
            compiled = CompiledScene(
                scene_id=scene.scene_id or f"SC{scene_index+1:02d}", beat_id=scene.beat_id,
                duration_s=scene.duration_s, narration=scene.narration, headline=scene.headline,
                background=scene.background, assets=placements, motion_tracks=tracks,
                camera_tracks=camera_tracks, transition=scene.transition, raw_scene=scene.model_dump(mode="json"),
            )
            save_json(self.root / f"{compiled.scene_id}.json", compiled)
            scenes.append(compiled)
        save_json(self.root / "compiled_scenes.json", scenes)
        return scenes

    def _placements(self, scene: SceneRequest, assets_by_intent: dict[str, AssetRecord]) -> list[dict[str, Any]]:
        output = []
        hero_index = 0
        support_index = 0
        for visual in scene.visuals:
            record = assets_by_intent.get(visual.intent_id)
            if not record:
                continue
            role = str(visual.role).lower()
            provided = visual.attributes.get("layout", {}) if isinstance(visual.attributes, dict) else {}
            if "integrated_scene" in role or "integrated scene" in role:
                default = {"x": 0.0, "y": 0.0, "width": 1.0, "height": 1.0, "z_index": 1}
            elif role == "hero" or ("hero" in role and hero_index == 0):
                default = {"x": 0.06, "y": 0.25, "width": 0.88, "height": 0.56, "z_index": 2}
                hero_index += 1
            elif "dashboard" in role or "meter" in str(visual.kind).lower():
                default = {"x": 0.08, "y": 0.72, "width": 0.84, "height": 0.22, "z_index": 5}
            else:
                col = support_index % 2; row = support_index // 2
                default = {"x": 0.06 + col * 0.47, "y": 0.62 + row * 0.15, "width": 0.41, "height": 0.14, "z_index": 4}
                support_index += 1
            placement = {**default, **(provided if isinstance(provided, dict) else {})}
            output.append({
                "intent_id": visual.intent_id, "asset_id": record.asset_id,
                "svg_path": record.svg_path, "semantic_layers": record.semantic_layers,
                "description": visual.description, "role": visual.role, **placement,
            })
        return output

    def _tracks(self, scene: SceneRequest, placements: list[dict[str, Any]], force: bool) -> list[MotionTrack]:
        semantic_ids = []
        for asset in placements:
            semantic_ids.append(asset["asset_id"])
            semantic_ids.extend(layer.get("layer_id", "") for layer in asset.get("semantic_layers", []))
        semantic_ids = [x for x in semantic_ids if x]
        intents = [m.model_dump(mode="json") for m in scene.motions]
        for visual in scene.visuals:
            for motion in visual.motion_intents:
                intents.append(MotionIntent.model_validate(motion).model_dump(mode="json"))
        if not intents:
            intents = [{"target": placements[0]["asset_id"] if placements else "root", "operation": "controlled reveal with a subtle stable push-in", "timing": {"start_s": 0, "end_s": scene.duration_s}}]
        fallback = self._fallback_tracks(scene, placements, intents)
        raw = self.llm.generate_json(
            system=self.SYSTEM,
            prompt=f"""Compile these motion intentions into renderer tracks.
Scene duration: {scene.duration_s} seconds
Available targets: {json.dumps(semantic_ids, ensure_ascii=False)}
Motion intentions: {json.dumps(intents, ensure_ascii=False, indent=2)}
Visual event: {scene.visual_event}

Return {{"tracks": [...]}}. Each track has:
- target: one supplied semantic ID, asset ID, or "camera"
- property: any of translateX, translateY, rotate, scale, opacity, strokeDashoffset,
  fillOpacity, blur, brightness, counter, clipProgress, pathProgress, or a CSS/SVG numeric property
- keyframes: list of {{time_s, value}}
- easing: string
- loop: boolean
- transform_origin
Use separate tracks for independent physical layers. No random camera shake.""",
            namespace="motion_compile", fallback={"tracks": [t.model_dump(mode="json") for t in fallback]}, force=force,
        )
        track_values = raw.get("tracks", raw) if isinstance(raw, dict) else raw
        if not isinstance(track_values, list):
            track_values = [t.model_dump(mode="json") for t in fallback]
        tracks = []
        for value in track_values:
            if isinstance(value, str):
                value = {"target": semantic_ids[0] if semantic_ids else "root", "property": value, "keyframes": [{"time_s": 0, "value": 0}, {"time_s": scene.duration_s, "value": 1}]}
            if not isinstance(value, dict):
                continue
            data = dict(value)
            data.setdefault("target", semantic_ids[0] if semantic_ids else "root")
            data["target"] = self._resolve_target(str(data["target"]), semantic_ids)
            data.setdefault("property", data.get("operation", "opacity"))
            data.setdefault("keyframes", [{"time_s": 0, "value": 0}, {"time_s": min(0.6, scene.duration_s), "value": 1}])
            if isinstance(data["keyframes"], dict):
                data["keyframes"] = [{"time_s": k, "value": v} for k, v in data["keyframes"].items()]
            tracks.append(MotionTrack.model_validate(data))
        return tracks or fallback

    def _fallback_tracks(self, scene: SceneRequest, placements: list[dict[str, Any]], intents: list[Any]) -> list[MotionTrack]:
        tracks = []
        for index, asset in enumerate(placements):
            target = asset["asset_id"]
            delay = min(0.8, index * 0.12)
            tracks.extend([
                MotionTrack(target=target, property="opacity", keyframes=[{"time_s": delay, "value": 0}, {"time_s": delay + 0.45, "value": 1}], easing="easeOut"),
                MotionTrack(target=target, property="scale", keyframes=[{"time_s": delay, "value": 0.94}, {"time_s": delay + 0.65, "value": 1}], easing="spring"),
            ])
        combined = " ".join(str(x) for x in intents).lower()
        layers = [layer.get("layer_id") for asset in placements for layer in asset.get("semantic_layers", []) if layer.get("layer_id")]
        if any(word in combined for word in ("rotate", "rotation", "spin", "orbit")):
            target = next((x for x in layers if any(t in x.lower() for t in ("cloud", "ring", "rotat", "planet"))), placements[0]["asset_id"] if placements else "root")
            tracks.append(MotionTrack(target=target, property="rotate", keyframes=[{"time_s": 0, "value": 0}, {"time_s": scene.duration_s, "value": 120}], easing="linear"))
        if any(word in combined for word in ("flow", "wind", "stream", "momentum", "eastward")):
            target = next((x for x in layers if any(t in x.lower() for t in ("flow", "air", "cloud", "field"))), placements[0]["asset_id"] if placements else "root")
            tracks.append(MotionTrack(target=target, property="translateX", keyframes=[{"time_s": 0, "value": -18}, {"time_s": scene.duration_s, "value": 70}], easing="linear", loop=True))
        return tracks

    def _camera_tracks(self, scene: SceneRequest) -> list[MotionTrack]:
        camera = scene.camera
        if isinstance(camera, str):
            text = camera.lower()
            if "push" in text or "zoom" in text:
                return [MotionTrack(target="camera", property="scale", keyframes=[{"time_s": 0, "value": 1}, {"time_s": scene.duration_s, "value": 1.05}], easing="easeInOut")]
            return []
        if isinstance(camera, dict) and isinstance(camera.get("tracks"), list):
            return [MotionTrack.model_validate(t) for t in camera["tracks"]]
        return []

    @staticmethod
    def _resolve_target(requested: str, available: list[str]) -> str:
        if requested in available or requested == "camera":
            return requested
        tokens = set(re.findall(r"[a-z0-9]+", requested.lower()))
        scored = []
        for target in available:
            target_tokens = set(re.findall(r"[a-z0-9]+", target.lower()))
            scored.append((len(tokens & target_tokens), target))
        return max(scored, default=(0, available[0] if available else "root"))[1]


In [ ]:
%%writefile /content/scientific_motion_studio_v5/src/scistudio_v5/audio.py
# Cell 18: scistudio_v5/audio.py
from __future__ import annotations

import shutil
from pathlib import Path
from typing import Any

from .schemas import ScriptPackage, Storyboard
from .utils import ensure_dir, ffprobe_duration, run_command, save_json


class AudioEngine:
    """Notebook-safe audio engine.

    Edge TTS is invoked through its CLI in a separate process. This avoids the
    `asyncio.run() cannot be called from a running event loop` failure in Colab.
    """

    def __init__(self, config: dict[str, Any], root: str | Path):
        self.config = config
        self.root = ensure_dir(root)

    def synthesize(self, text: str, output: str | Path) -> Path:
        output = Path(output)
        if output.exists() and output.stat().st_size > 1000:
            return output
        text_path = output.with_suffix(".txt")
        text_path.write_text(text.strip(), encoding="utf-8")
        edge_binary = shutil.which("edge-tts")
        if edge_binary:
            try:
                run_command([
                    edge_binary,
                    "--voice", str(self.config.get("voice", "en-US-GuyNeural")),
                    "--rate", str(self.config.get("rate", "+6%")),
                    "--pitch", str(self.config.get("pitch", "+0Hz")),
                    "--file", str(text_path),
                    "--write-media", str(output),
                ], timeout=180)
            except Exception as exc:
                print(f"[audio] Edge TTS failed; using local fallback: {exc}")
        if not output.exists() or output.stat().st_size < 1000:
            self._fallback_tts(text, output)
        text_path.unlink(missing_ok=True)
        return output

    def _fallback_tts(self, text: str, output: Path) -> None:
        wav = output.with_suffix(".wav")
        if not shutil.which("espeak-ng"):
            raise RuntimeError("Both edge-tts and espeak-ng are unavailable")
        run_command(["espeak-ng", "-v", str(self.config.get("fallback_voice", "en-us")), "-s", "170", "-w", str(wav), text])
        run_command(["ffmpeg", "-y", "-i", str(wav), "-c:a", "libmp3lame", "-b:a", "192k", str(output)])
        wav.unlink(missing_ok=True)

    def build(self, script: ScriptPackage, storyboard: Storyboard) -> tuple[ScriptPackage, Storyboard, dict[str, Any]]:
        voice_root = ensure_dir(self.root / "voice")
        voice_files = {}
        updated_beats = []
        updated_scenes = []
        for index, beat in enumerate(script.beats):
            voice = self.synthesize(beat.spoken_line, voice_root / f"{beat.beat_id.lower()}.mp3")
            actual = ffprobe_duration(voice)
            duration = round(max(beat.duration_s, actual + beat.pause_after_ms / 1000 + 0.2), 3)
            updated_beats.append(beat.model_copy(update={"duration_s": duration}))
            scene = storyboard.scenes[index] if index < len(storyboard.scenes) else storyboard.scenes[-1]
            updated_scenes.append(scene.model_copy(update={"duration_s": duration, "narration": beat.spoken_line}))
            voice_files[beat.beat_id] = str(voice)
        total = round(sum(scene.duration_s for scene in updated_scenes), 3)
        script = script.model_copy(update={"beats": updated_beats, "estimated_duration_s": total})
        storyboard = storyboard.model_copy(update={"scenes": updated_scenes, "estimated_duration_s": total})
        scene_starts = []
        _cursor = 0.0
        for scene in updated_scenes:
            scene_starts.append(round(_cursor, 3))
            _cursor += scene.duration_s
        bed = self._support_bed(total, self.root / "support_bed.m4a", scene_starts=scene_starts)
        manifest = {"voice": voice_files, "support_bed": str(bed), "duration_s": total}
        save_json(self.root / "audio_manifest.json", manifest)
        return script, storyboard, manifest

    def _support_bed(self, duration: float, output: Path, scene_starts: list[float] | None = None) -> Path:
        """Designed music bed (not a flat drone): a minor-triad pad with slow
        tremolo movement, a filtered 'air' layer, a soft ~0.9 Hz sub pulse, and
        short filtered-noise whooshes synced to each scene start. All procedural
        via FFmpeg lavfi — no external assets, no licensing."""
        if output.exists() and output.stat().st_size > 1000 and not self.config.get("force_audio", False):
            return output
        fade_out = max(0.0, duration - 2.0)
        starts = [s for s in (scene_starts or []) if 0.05 < s < duration - 0.05]

        inputs = [
            "-f", "lavfi", "-i", f"sine=frequency=110:sample_rate=48000:duration={duration}",      # root A2
            "-f", "lavfi", "-i", f"sine=frequency=130.81:sample_rate=48000:duration={duration}",   # minor third C3
            "-f", "lavfi", "-i", f"sine=frequency=164.81:sample_rate=48000:duration={duration}",   # fifth E3
            "-f", "lavfi", "-i", f"sine=frequency=55:sample_rate=48000:duration={duration}",        # sub A1 (pulsed)
            "-f", "lavfi", "-i", f"anoisesrc=color=pink:sample_rate=48000:duration={duration}",      # air
        ]
        parts = [
            "[0:a]volume=0.020,tremolo=f=0.10:d=0.5,lowpass=f=360[p0]",
            "[1:a]volume=0.013,tremolo=f=0.13:d=0.5,lowpass=f=520[p1]",
            "[2:a]volume=0.012,tremolo=f=0.11:d=0.4,lowpass=f=680[p2]",
            "[3:a]volume=0.030,tremolo=f=0.9:d=0.9,lowpass=f=140[sub]",     # heartbeat pulse
            "[4:a]volume=0.006,highpass=f=120,lowpass=f=1400[air]",
            "[p0][p1][p2][sub][air]amix=inputs=5:normalize=0[bed]",
        ]
        # Whooshes at scene transitions.
        whoosh_labels = []
        wh_input_start = 5
        for i, start in enumerate(starts):
            inputs += ["-f", "lavfi", "-i", "anoisesrc=color=white:sample_rate=48000:duration=0.6"]
            lbl = f"wh{i}"
            delay = int(max(0, (start - 0.18) * 1000))
            parts.append(
                f"[{wh_input_start + i}:a]volume=0.10,highpass=f=300,lowpass=f=5000,"
                f"afade=t=in:st=0:d=0.12,afade=t=out:st=0.22:d=0.35,adelay={delay}|{delay}[{lbl}]"
            )
            whoosh_labels.append(f"[{lbl}]")

        if whoosh_labels:
            parts.append("[bed]" + "".join(whoosh_labels) + f"amix=inputs={1 + len(whoosh_labels)}:normalize=0[mixed]")
            final_in = "[mixed]"
        else:
            final_in = "[bed]"
        parts.append(
            f"{final_in}alimiter=limit=0.72,afade=t=in:st=0:d=1.2,afade=t=out:st={fade_out}:d=2[out]"
        )
        run_command([
            "ffmpeg", "-y", *inputs, "-filter_complex", ";".join(parts),
            "-map", "[out]", "-t", str(duration), "-c:a", "aac", "-b:a", "160k", str(output),
        ])
        return output


In [ ]:
%%writefile /content/scientific_motion_studio_v5/src/scistudio_v5/remotion.py
# Cell 19: scistudio_v5/remotion.py
from __future__ import annotations

import json
import os
import shutil
from pathlib import Path
from typing import Any

from .schemas import CompiledScene
from .utils import ensure_dir, run_command, save_json


class RemotionExporter:
    def __init__(self, config: dict[str, Any], root: str | Path):
        self.config = config
        self.root = ensure_dir(root)
        self.public = ensure_dir(self.root / "public")
        self.src = ensure_dir(self.root / "src")

    def create_project(self, scenes: list[CompiledScene], audio: dict[str, Any]) -> Path:
        audio_root = ensure_dir(self.public / "audio")
        bed_src = Path(audio["support_bed"])
        bed_name = "support_bed" + bed_src.suffix
        shutil.copy2(bed_src, audio_root / bed_name)
        voice_map = {}
        for beat_id, source in audio.get("voice", {}).items():
            source = Path(source)
            name = f"{beat_id.lower()}{source.suffix}"
            shutil.copy2(source, audio_root / name)
            voice_map[beat_id] = f"audio/{name}"

        payload = []
        cursor = 0
        fps = int(self.config.get("fps", 30))
        for scene in scenes:
            assets = []
            for asset in scene.assets:
                item = dict(asset)
                svg_text = Path(item.pop("svg_path")).read_text(encoding="utf-8")
                # Remove asset-local canvas backgrounds before composition.
                import re
                svg_text = re.sub(r'<rect\b(?=[^>]*\bid=["\'][^"\']*background["\'])[^>]*/>', '', svg_text, flags=re.I)
                item["svg"] = svg_text
                assets.append(item)
            frames = max(1, round(scene.duration_s * fps))
            payload.append({
                **scene.model_dump(mode="json", exclude={"assets"}),
                "assets": assets,
                "from_frame": cursor,
                "duration_frames": frames,
                "voice_file": voice_map.get(scene.beat_id, ""),
            })
            cursor += frames
        data = {
            "width": int(self.config.get("width", 1080)),
            "height": int(self.config.get("height", 1920)),
            "fps": fps,
            "duration_frames": cursor,
            "support_bed": f"audio/{bed_name}",
            "scenes": payload,
        }
        save_json(self.public / "data.json", data)
        self._write_package()
        self._write_tsconfig()
        (self.src / "index.tsx").write_text(self._typescript(), encoding="utf-8")
        return self.root

    def install(self) -> None:
        run_command(["npm", "install"], cwd=self.root, timeout=900, capture=False)

    def typecheck(self) -> None:
        run_command(["npx", "tsc", "--noEmit"], cwd=self.root, timeout=300, capture=False)

    def _browser_executable(self) -> str:
        """Resolve an existing Chrome/Chromium so Remotion does not have to
        download its own Chrome Headless Shell. That download hits
        remotion.media, which is blocked on locked-down runtimes (and sometimes
        flaky on Colab). Order: explicit config -> env -> Playwright cache ->
        common system paths."""
        import glob
        explicit = str(self.config.get("browser_executable", "")).strip()
        if explicit and Path(explicit).exists():
            return explicit
        for env in ("REMOTION_BROWSER_EXECUTABLE", "PUPPETEER_EXECUTABLE_PATH", "CHROME_PATH"):
            value = os.environ.get(env, "")
            if value and Path(value).exists():
                return value
        # Remotion needs an old-headless-compatible binary; prefer chrome-headless-shell
        # (full Chrome builds have removed old headless mode).
        pw = os.environ.get("PLAYWRIGHT_BROWSERS_PATH", "/opt/pw-browsers")
        patterns = [
            os.path.join(pw, "chromium_headless_shell-*/chrome-linux/headless_shell"),
            os.path.join(pw, "chromium_headless_shell-*/chrome-linux/chrome-headless-shell"),
            "/opt/pw-browsers/chromium_headless_shell-*/chrome-linux/headless_shell",
            os.path.join(pw, "chromium-*/chrome-linux/chrome"),
            "/usr/bin/chromium", "/usr/bin/chromium-browser", "/usr/bin/google-chrome",
        ]
        for pattern in patterns:
            matches = sorted(glob.glob(pattern))
            if matches:
                return matches[-1]
        return ""

    def ensure_browser(self) -> None:
        # If a usable browser already exists, skip the (often blocked) download.
        if self._browser_executable():
            return
        run_command(["npx", "remotion", "browser", "ensure"], cwd=self.root, timeout=900, capture=False)

    def render(self, output: str | Path) -> Path:
        output = Path(output).resolve()
        ensure_dir(output.parent)
        command = [
            "npx", "remotion", "render", "src/index.tsx", "ScientificMotion",
            str(output), "--codec", "h264", "--crf", str(self.config.get("crf", 18)),
            "--pixel-format", "yuv420p",
        ]
        browser = self._browser_executable()
        if browser:
            command.append(f"--browser-executable={browser}")
        run_command(command, cwd=self.root, timeout=int(self.config.get("render_timeout", 2400)), capture=False)
        return output

    def _write_package(self) -> None:
        package = {
            "name": "scientific-motion-studio-v5-render",
            "version": "5.0.0",
            "private": True,
            "scripts": {"render": "remotion render src/index.tsx ScientificMotion out.mp4", "typecheck": "tsc --noEmit"},
            # Pinned, mutually-compatible versions. "latest" previously pulled
            # TypeScript 7.0.x (the new Go-based compiler with a different API),
            # which crashes Remotion's esbuild loader. Pin to a known-good set.
            "dependencies": {
                "@remotion/cli": self.config.get("remotion_version", "4.0.489"),
                "remotion": self.config.get("remotion_version", "4.0.489"),
                "react": "19.0.0",
                "react-dom": "19.0.0"
            },
            "devDependencies": {
                "typescript": "5.6.3",
                "@types/react": "19.0.0",
                "@types/react-dom": "19.0.0"
            }
        }
        save_json(self.root / "package.json", package)

    def _write_tsconfig(self) -> None:
        save_json(self.root / "tsconfig.json", {
            "compilerOptions": {
                "target": "ES2020", "module": "ESNext", "moduleResolution": "Bundler",
                "jsx": "react-jsx", "strict": True, "esModuleInterop": True,
                "skipLibCheck": True, "resolveJsonModule": True,
                "allowSyntheticDefaultImports": True
            },
            "include": ["src"]
        })

    @staticmethod
    def _typescript() -> str:
        return r'''import React from 'react';
import {
  AbsoluteFill,
  Composition,
  Easing,
  Html5Audio,
  Sequence,
  interpolate,
  registerRoot,
  spring,
  staticFile,
  useCurrentFrame,
  useVideoConfig,
} from 'remotion';
import data from '../public/data.json';

type Keyframe = {time_s?: number; time?: number; value: any};
type Track = {
  target: string;
  property: string;
  keyframes: Keyframe[];
  easing?: any;
  loop?: boolean;
  transform_origin?: any;
};

type Asset = {
  asset_id: string;
  svg: string;
  x: number;
  y: number;
  width: number;
  height: number;
  z_index?: number;
  role?: any;
};

type Scene = {
  scene_id: string;
  duration_s: number;
  duration_frames: number;
  from_frame: number;
  headline: string;
  narration: string;
  background: any;
  assets: Asset[];
  motion_tracks: Track[];
  camera_tracks: Track[];
  voice_file?: string;
};

const easingFor = (name: any) => {
  const key = String(name ?? 'linear').toLowerCase();
  if (key.includes('out')) return Easing.out(Easing.cubic);
  if (key.includes('inout') || key.includes('in_out')) return Easing.inOut(Easing.cubic);
  if (key.includes('in')) return Easing.in(Easing.cubic);
  return Easing.linear;
};

const numeric = (value: any, fallback = 0) => {
  const parsed = Number(value);
  return Number.isFinite(parsed) ? parsed : fallback;
};

const valueAt = (track: Track, seconds: number, fps: number, frame: number) => {
  const keyframes = [...(track.keyframes ?? [])]
    .map((k) => ({time: numeric(k.time_s ?? k.time, 0), value: k.value}))
    .sort((a, b) => a.time - b.time);
  if (!keyframes.length) return 0;
  const end = Math.max(0.0001, keyframes[keyframes.length - 1].time);
  const local = track.loop ? ((seconds % end) + end) % end : seconds;
  if (String(track.easing).toLowerCase().includes('spring') && keyframes.length >= 2) {
    const from = numeric(keyframes[0].value, 0);
    const to = numeric(keyframes[keyframes.length - 1].value, 1);
    const startFrame = Math.round(keyframes[0].time * fps);
    const progress = spring({fps, frame: Math.max(0, frame - startFrame), config: {damping: 18, stiffness: 120, mass: 0.8}});
    return from + (to - from) * progress;
  }
  if (keyframes.length === 1) return keyframes[0].value;
  for (let i = 0; i < keyframes.length - 1; i++) {
    const a = keyframes[i]; const b = keyframes[i + 1];
    if (local <= b.time) {
      if (typeof a.value === 'number' || !Number.isNaN(Number(a.value))) {
        return interpolate(local, [a.time, b.time], [numeric(a.value), numeric(b.value)], {
          extrapolateLeft: 'clamp', extrapolateRight: 'clamp', easing: easingFor(track.easing),
        });
      }
      return local < b.time ? a.value : b.value;
    }
  }
  return keyframes[keyframes.length - 1].value;
};

const cssForTracks = (tracks: Track[], seconds: number, fps: number, frame: number) => {
  const byTarget: Record<string, {transforms: string[]; styles: Record<string,string>; origin: string}> = {};
  for (const track of tracks ?? []) {
    const target = track.target || 'root';
    byTarget[target] ??= {transforms: [], styles: {}, origin: String(track.transform_origin ?? 'center')};
    const bucket = byTarget[target];
    const value = valueAt(track, seconds, fps, frame);
    const prop = String(track.property ?? 'opacity');
    if (prop === 'translateX') bucket.transforms.push(`translateX(${numeric(value)}px)`);
    else if (prop === 'translateY') bucket.transforms.push(`translateY(${numeric(value)}px)`);
    else if (prop === 'rotate') bucket.transforms.push(`rotate(${numeric(value)}deg)`);
    else if (prop === 'scale') bucket.transforms.push(`scale(${numeric(value,1)})`);
    else if (prop === 'blur') bucket.styles.filter = `blur(${numeric(value)}px)`;
    else if (prop === 'brightness') bucket.styles.filter = `brightness(${numeric(value,1)})`;
    else if (prop === 'pathProgress') {
      bucket.styles.strokeDasharray = '1000'; bucket.styles.strokeDashoffset = String(1000 * (1 - numeric(value)));
    } else if (prop === 'clipProgress') {
      bucket.styles.clipPath = `inset(0 ${100 * (1 - numeric(value))}% 0 0)`;
    } else if (prop === 'counter') {
      bucket.styles['--counter-value'] = String(Math.round(numeric(value)));
    } else {
      const cssName = prop.replace(/[A-Z]/g, (m) => '-' + m.toLowerCase());
      bucket.styles[cssName] = typeof value === 'number' ? String(value) : String(value);
    }
  }
  return Object.entries(byTarget).map(([target, bucket]) => {
    const selector = target === 'camera' ? '#camera' : `#${CSS.escape(target)}`;
    const styles = {...bucket.styles};
    if (bucket.transforms.length) styles.transform = bucket.transforms.join(' ');
    styles['transform-origin'] = bucket.origin;
    const body = Object.entries(styles).map(([k,v]) => `${k}:${v} !important`).join(';');
    return `${selector}{${body}}`;
  }).join('\n');
};

const SceneView: React.FC<{scene: Scene}> = ({scene}) => {
  const frame = useCurrentFrame();
  const {fps, width, height} = useVideoConfig();
  const seconds = frame / fps;
  const css = cssForTracks(scene.motion_tracks ?? [], seconds, fps, frame);
  const cameraCss = cssForTracks(scene.camera_tracks ?? [], seconds, fps, frame);
  const background = typeof scene.background === 'string' && scene.background.toLowerCase().includes('dark') ? '#202A33' : '#F7F7F4';
  const integrated = (scene.assets ?? []).some((asset) => String(asset.role ?? '').toLowerCase().includes('integrated'));
  return <AbsoluteFill style={{backgroundColor: background, overflow: 'hidden'}}>
    <style>{css + '\n' + cameraCss}</style>
    <div id="camera" style={{position: 'absolute', inset: 0}}>
      {!integrated ? <div style={{position:'absolute', left: width*0.06, right: width*0.06, top: height*0.055, zIndex: 20}}>
        <div style={{fontFamily:'Arial, sans-serif', fontWeight:900, fontSize:70, letterSpacing:-2, lineHeight:0.96, color: background === '#202A33' ? '#fff' : '#17212B', textTransform:'uppercase'}}>{scene.headline}</div>
        <div style={{height:10, width:'62%', background:'#2D9CDB', borderRadius:10, marginTop:22}} />
      </div> : null}
      {(scene.assets ?? []).map((asset) => <div
        id={asset.asset_id}
        key={asset.asset_id}
        className="svg-asset"
        style={{position:'absolute', left: `${asset.x*100}%`, top:`${asset.y*100}%`, width:`${asset.width*100}%`, height:`${asset.height*100}%`, zIndex: asset.z_index ?? 1}}
        dangerouslySetInnerHTML={{__html: asset.svg}}
      />)}
    </div>
    {!integrated ? <div style={{position:'absolute', left:'6%', right:'6%', bottom:'5%', zIndex:30, background:'rgba(32,42,51,.94)', color:'#fff', border:'5px solid #17212B', borderRadius:24, padding:'22px 28px', fontFamily:'Arial, sans-serif', fontWeight:800, fontSize:35, lineHeight:1.15}}>{scene.narration}</div> : null}
    {scene.voice_file ? <Html5Audio src={staticFile(scene.voice_file)} volume={1} /> : null}
  </AbsoluteFill>;
};

const Film: React.FC = () => <AbsoluteFill>
  <Html5Audio src={staticFile((data as any).support_bed)} volume={0.22} />
  {(data.scenes as Scene[]).map((scene) => <Sequence key={scene.scene_id} from={scene.from_frame} durationInFrames={scene.duration_frames} premountFor={30}>
    <SceneView scene={scene} />
  </Sequence>)}
</AbsoluteFill>;

const Root: React.FC = () => <Composition
  id="ScientificMotion"
  component={Film}
  durationInFrames={(data as any).duration_frames}
  fps={(data as any).fps}
  width={(data as any).width}
  height={(data as any).height}
/>;

registerRoot(Root);
'''


In [ ]:
%%writefile /content/scientific_motion_studio_v5/src/scistudio_v5/pipeline.py
# Cell 20: scistudio_v5/pipeline.py
from __future__ import annotations

import json
import time
import zipfile
from pathlib import Path
from typing import Any

from tqdm.auto import tqdm

from .audio import AudioEngine
from .benchmark import BenchmarkAnalyzer, StyleFingerprint
from .critic import VisualCritic
from .script_critic import ScriptCritic
from .director import CreativeDirector
from .illustration import (
    EditorialBenchmarkDemo,
    IllustrationQualityError,
    OpenSceneIllustrationCompiler,
)
from .llm import LLMRouter
from .motion import MotionCompiler
from .references import ReferenceCollector
from .remotion import RemotionExporter
from .research import PublicResearchSearch, ResearchEngine
from .schemas import AssetRecord, Storyboard, VisualIntent
from .svg_compiler import DynamicSVGEngine
from .utils import ensure_dir, ffprobe_duration, save_json, slugify


class StudioPipeline:
    """Scientific Motion Studio v5 — Illustration-Grade Open Intent Pipeline.

    Key difference from v4: illustration quality is evaluated against a
    reference-style benchmark at both asset and integrated-scene level. A valid
    but icon-like SVG is rejected rather than silently rendered.
    """

    def __init__(self, root: str | Path, config: dict[str, Any], secrets: dict[str, Any]):
        self.root = ensure_dir(root)
        self.config = config
        self.secrets = secrets
        self.cache_root = ensure_dir(self.root / "cache")
        self.llm = LLMRouter(config.get("llm", {}), secrets, self.cache_root / "llm")
        self.search = PublicResearchSearch(config.get("research", {}), self.cache_root / "search")
        self.research = ResearchEngine(self.llm, self.cache_root / "research")
        self.director = CreativeDirector(self.llm, config.get("story", {}), self.cache_root / "story")
        self.script_critic = ScriptCritic(self.llm, config.get("script_critic", {}), self.cache_root / "script_quality")

    def run(
        self,
        *,
        topic_mode: str = "manual",
        topic_value: str = "What if Earth suddenly stopped rotating?",
        force_refresh: bool = False,
        render_video: bool = True,
    ) -> dict[str, Any]:
        topic = self._resolve_topic(topic_mode, topic_value)
        run_id = time.strftime("%Y%m%d_%H%M%S") + "_" + slugify(topic, 64)
        run_dir = ensure_dir(self.root / "runs" / run_id)
        save_json(run_dir / "config_snapshot.json", self.config)

        progress = tqdm(total=13, desc="Scientific Motion Studio v5", unit="stage")

        analyzer, profiles, aggregate_profile, contact_sheet = self._benchmark(run_dir)
        progress.update(1)

        # Regression benchmark proves the gate can distinguish illustration-grade output.
        demo_record = EditorialBenchmarkDemo(run_dir / "benchmark_demo").build()
        demo_gate = VisualCritic(
            self.llm,
            {**self.config.get("critic", {}), "strict_quality_gate": False},
            run_dir / "benchmark_demo" / "quality",
            analyzer, profiles, contact_sheet,
        )
        demo_report = demo_gate.review(
            VisualIntent(intent_id="benchmark-demo", description="original scientific editorial coastal impact scene", role="hero"),
            demo_record,
            force=False,
        )
        save_json(run_dir / "benchmark_demo" / "quality_report.json", demo_report)
        progress.update(1)

        sources = self.search.search(topic, force=force_refresh)
        save_json(run_dir / "sources.json", sources)
        progress.update(1)

        research = self.research.build(topic, sources, force=force_refresh)
        save_json(run_dir / "research.json", research)
        progress.update(1)

        script = self.director.script(research, force=force_refresh)
        save_json(run_dir / "script.json", script)
        # Retention-first script gate: the narration counterpart to the visual critic.
        if self.config.get("script_critic", {}).get("enabled", True):
            script, script_report = self.script_critic.refine(script, research, self.director, force=force_refresh)
            save_json(run_dir / "script_quality.json", script_report)
        progress.update(1)

        storyboard = self.director.storyboard(script, research, force=force_refresh)
        storyboard.width = int(self.config.get("render", {}).get("width", storyboard.width))
        storyboard.height = int(self.config.get("render", {}).get("height", storyboard.height))
        storyboard.fps = int(self.config.get("render", {}).get("fps", storyboard.fps))
        save_json(run_dir / "storyboard_open_intent.json", storyboard)
        progress.update(1)

        audio_engine = AudioEngine(self.config.get("audio", {}), run_dir / "audio")
        script, storyboard, audio = audio_engine.build(script, storyboard)
        save_json(run_dir / "script_timed.json", script)
        save_json(run_dir / "storyboard_timed.json", storyboard)
        progress.update(1)

        references = ReferenceCollector(self.config.get("references", {}), run_dir / "references")
        svg_engine = DynamicSVGEngine(
            self.llm,
            references,
            self.config.get("svg", {}),
            run_dir / "visuals",
            style_reference=aggregate_profile,
        )

        # Build every open-ended LLM visual independently first.
        asset_records: dict[str, AssetRecord] = {}
        for scene in storyboard.scenes:
            for visual in scene.visuals:
                if visual.intent_id in asset_records:
                    continue
                record = svg_engine.build(visual, force=force_refresh)
                asset_records[visual.intent_id] = record
        save_json(run_dir / "asset_records_before_critic.json", asset_records)
        progress.update(1)

        # Asset-level critic is advisory for support assets and strict for hero visuals.
        quality_reports: dict[str, Any] = {}
        asset_critic_config = {
            **self.config.get("critic", {}),
            "strict_quality_gate": False,
            "style_score_threshold": float(self.config.get("critic", {}).get("asset_style_score_threshold", 0.58)),
            "acceptance_threshold": float(self.config.get("critic", {}).get("asset_acceptance_threshold", 0.66)),
        }
        asset_critic = VisualCritic(
            self.llm,
            asset_critic_config,
            run_dir / "quality" / "assets",
            analyzer, profiles, contact_sheet,
        )
        for scene in storyboard.scenes:
            for visual in scene.visuals:
                record = asset_records.get(visual.intent_id)
                if not record:
                    continue
                report = asset_critic.review(visual, record, force=force_refresh)
                # Only refine weak hero/concrete assets here. Integrated scene QC remains strict later.
                if not report.passed and ("hero" in str(visual.role).lower() or len(scene.visuals) <= 2):
                    try:
                        record, report = asset_critic.refine(visual, record, svg_engine, force=force_refresh)
                    except IllustrationQualityError:
                        pass
                asset_records[visual.intent_id] = record
                quality_reports[visual.intent_id] = report.model_dump(mode="json")
        save_json(run_dir / "asset_records.json", asset_records)
        save_json(run_dir / "quality_reports_assets.json", quality_reports)
        progress.update(1)

        # Compose and strictly validate one integrated illustration per scene.
        scene_records, scene_quality, execution_storyboard = self._build_integrated_scenes(
            storyboard=storyboard,
            asset_records=asset_records,
            svg_engine=svg_engine,
            analyzer=analyzer,
            profiles=profiles,
            contact_sheet=contact_sheet,
            run_dir=run_dir,
            force_refresh=force_refresh,
        )
        save_json(run_dir / "scene_records.json", scene_records)
        save_json(run_dir / "quality_reports_scenes.json", scene_quality)
        save_json(run_dir / "storyboard_execution.json", execution_storyboard)
        progress.update(1)

        motion = MotionCompiler(self.llm, self.config.get("motion", {}), run_dir / "motion")
        compiled_scenes = motion.compile(execution_storyboard, scene_records, force=force_refresh)
        progress.update(1)

        remotion_dir = run_dir / "remotion"
        exporter = RemotionExporter(self.config.get("render", {}), remotion_dir)
        exporter.create_project(compiled_scenes, audio)
        progress.update(1)

        video_path = run_dir / "scientific_motion.mp4"
        render_error = ""
        if render_video:
            try:
                if self.config.get("render", {}).get("install_dependencies", True):
                    exporter.install()
                if self.config.get("render", {}).get("typecheck", True):
                    exporter.typecheck()
                if self.config.get("render", {}).get("ensure_browser", True):
                    exporter.ensure_browser()
                exporter.render(video_path)
            except Exception as exc:
                render_error = f"{type(exc).__name__}: {exc}"
                print(f"[render] Video render did not complete: {render_error}")

        qc = self._qc(video_path, scene_records, scene_quality, render_error)
        save_json(run_dir / "run_qc.json", qc)
        metadata = self._metadata(topic, script, research)
        save_json(run_dir / "youtube_metadata.json", metadata)
        archive = self._archive(run_dir)
        progress.update(1)
        progress.close()

        result = {
            "topic": topic,
            "run_dir": str(run_dir),
            "video": str(video_path) if video_path.exists() else "",
            "archive": str(archive),
            "benchmark_contact_sheet": str(contact_sheet) if contact_sheet else "",
            "benchmark_demo": demo_record.preview_path,
            "benchmark_demo_quality": demo_report.model_dump(mode="json"),
            "research": str(run_dir / "research.json"),
            "script": str(run_dir / "script_timed.json"),
            "storyboard": str(run_dir / "storyboard_execution.json"),
            "assets": str(run_dir / "scene_records.json"),
            "quality": str(run_dir / "quality_reports_scenes.json"),
            "render_error": render_error,
            "qc": qc,
        }
        save_json(run_dir / "result.json", result)
        return result

    def _benchmark(self, run_dir: Path):
        benchmark_config = self.config.get("benchmark", {})
        analyzer = BenchmarkAnalyzer(run_dir / "benchmark")
        reference_setting = str(benchmark_config.get("reference_video", "")).strip()
        video = Path(reference_setting).expanduser()
        # Require an actual file: an empty/blank setting resolves to "." (a
        # directory that "exists"), which would crash ffprobe.
        if reference_setting and video.is_file():
            profiles = analyzer.from_video_set(video, sample_count=int(benchmark_config.get("sample_count", 10)))
            aggregate = analyzer.from_video(video, sample_count=int(benchmark_config.get("sample_count", 10)))
            contact_sheet = run_dir / "benchmark" / "reference_contact_sheet_set.png"
        else:
            profiles = self._default_benchmark_profiles()
            aggregate = self._average_profiles(profiles)
            contact_sheet = None
            save_json(run_dir / "benchmark" / "default_profiles.json", [p.to_dict() for p in profiles])
        return analyzer, profiles, aggregate, contact_sheet

    def _build_integrated_scenes(
        self,
        *,
        storyboard: Storyboard,
        asset_records: dict[str, AssetRecord],
        svg_engine: DynamicSVGEngine,
        analyzer: BenchmarkAnalyzer,
        profiles: list[StyleFingerprint],
        contact_sheet: Path | None,
        run_dir: Path,
        force_refresh: bool,
    ):
        compiler = OpenSceneIllustrationCompiler(run_dir / "integrated_scenes", self.config.get("svg", {}).get("style", {}))
        # A scene that cannot reach the benchmark is refined, then the best
        # version is kept and its quality recorded in QC — the run always
        # completes. Set critic.strict_scene_gate=True to instead abort the run
        # on an unacceptable scene (the original v5 behavior).
        strict_scene_gate = bool(self.config.get("critic", {}).get("strict_scene_gate", False))
        critic_config = {
            **self.config.get("critic", {}),
            "strict_quality_gate": strict_scene_gate,
            "style_score_threshold": float(self.config.get("critic", {}).get("scene_style_score_threshold", 0.70)),
            "acceptance_threshold": float(self.config.get("critic", {}).get("scene_acceptance_threshold", 0.76)),
            "minimum_path_count": int(self.config.get("critic", {}).get("scene_minimum_path_count", 18)),
            "minimum_group_count": int(self.config.get("critic", {}).get("scene_minimum_group_count", 8)),
        }
        critic = VisualCritic(
            self.llm, critic_config, run_dir / "quality" / "scenes",
            analyzer, profiles, contact_sheet,
        )
        scene_records: dict[str, AssetRecord] = {}
        reports: dict[str, Any] = {}
        execution = storyboard.model_copy(deep=True)
        max_rounds = int(critic_config.get("max_scene_refinement_rounds", 4))

        for scene_index, scene in enumerate(execution.scenes):
            records = [asset_records[v.intent_id] for v in scene.visuals if v.intent_id in asset_records]
            if not records:
                raise IllustrationQualityError(f"No visual records available for scene {scene.scene_id}")
            scene_intent = VisualIntent(
                intent_id=f"{scene.scene_id or f'SC{scene_index+1:02d}'}--integrated",
                description=self._scene_description(scene),
                kind="complete scientific editorial scene illustration",
                role="integrated_scene",
                attributes={"layout": scene.layout, "dashboard": scene.dashboard, "headline": scene.headline},
                relationships=[v.relationships for v in scene.visuals],
                desired_layers=[{"name": r.asset_id, "role": r.description} for r in records],
                motion_intents=[m.model_dump(mode="json") for m in scene.motions],
                reference_requests=[item for v in scene.visuals for item in v.reference_requests],
                visual_constraints={
                    "match_reference_illustration_level": True,
                    "preserve_all_scene_objects": True,
                    "no_generic_icons": True,
                },
                raw_llm_output={"scene": scene.model_dump(mode="json")},
            )
            current = compiler.compose(
                scene, records,
                width=int(storyboard.width), height=int(storyboard.height),
                layout=scene.layout if isinstance(scene.layout, dict) else {},
            )
            report = critic.review(scene_intent, current, force=force_refresh)

            # If composition alone is insufficient, escalate to a full-scene
            # reference/program reconstruction and keep whichever version scores higher.
            for round_index in range(max_rounds):
                if report.passed:
                    break
                revised = scene_intent.model_copy(deep=True)
                revised.visual_constraints = {
                    **revised.visual_constraints,
                    "critic_round": round_index + 1,
                    "required_repairs": report.repair_plan or report.failures,
                    "rebuild_as_one_coherent_scene": True,
                    "minimum_semantic_parts": 10,
                    "minimum_flat_shading_regions": 4,
                }
                revised.reference_requests = list(revised.reference_requests) + [
                    {"query": revised.description, "purpose": "full scene composition"},
                    {"query": f"{revised.description} scientific editorial illustration", "purpose": "style reference"},
                ]
                rebuilt = svg_engine.build(revised, force=True)
                rebuilt_report = critic.review(revised, rebuilt, force=True)
                # Compare using style score, then minimum vision score.
                old_score = float(report.scores.get("style_consistency", 0.0)) + float(report.scores.get("recognizability", 0.0))
                new_score = float(rebuilt_report.scores.get("style_consistency", 0.0)) + float(rebuilt_report.scores.get("recognizability", 0.0))
                if rebuilt_report.passed or new_score > old_score:
                    current, report = rebuilt, rebuilt_report
                else:
                    # Improve the weakest object assets and recompose.
                    for visual in scene.visuals:
                        record = asset_records.get(visual.intent_id)
                        if not record:
                            continue
                        try:
                            asset_records[visual.intent_id], _ = critic.refine(visual, record, svg_engine, force=True)
                        except IllustrationQualityError:
                            pass
                    records = [asset_records[v.intent_id] for v in scene.visuals if v.intent_id in asset_records]
                    current = compiler.compose(
                        scene, records,
                        width=int(storyboard.width), height=int(storyboard.height),
                        layout=scene.layout if isinstance(scene.layout, dict) else {},
                    )
                    report = critic.review(scene_intent, current, force=True)

            if not report.passed:
                if strict_scene_gate:
                    raise IllustrationQualityError(
                        f"Scene {scene.scene_id} did not reach the illustration benchmark. Failures: {report.failures}"
                    )
                # Non-fatal: keep the best version, record the shortfall in QC.
                print(f"[scene-gate] {scene.scene_id} below benchmark, keeping best. Failures: {report.failures}")
            scene_records[scene_intent.intent_id] = current
            reports[scene_intent.intent_id] = report.model_dump(mode="json")
            scene.visuals = [scene_intent]
            # The integrated scene SVG already includes headline/dashboard/caption.
            scene.layout = {"integrated_scene": True}

        execution.storyboard_hash = execution.storyboard_hash + "-illustration-v5"
        return scene_records, reports, execution

    @staticmethod
    def _scene_description(scene) -> str:
        visual_text = "; ".join(v.description for v in scene.visuals)
        return (
            f"Complete scene for headline '{scene.headline}'. Narration: {scene.narration}. "
            f"Visual event: {scene.visual_event}. Objects and effects: {visual_text}. "
            f"Dashboard: {scene.dashboard}. Compose as one professional scientific editorial illustration."
        )

    def _resolve_topic(self, mode: str, value: str) -> str:
        mode = str(mode).lower().strip()
        value = str(value).strip()
        if mode == "manual":
            if not value:
                raise ValueError("TOPIC_VALUE cannot be empty in manual mode")
            return value
        if mode == "keyword":
            return f"What if {value}?" if not value.lower().startswith("what if") else value
        if mode == "auto":
            topics = self.config.get("auto_topics", [
                "What if Earth suddenly stopped rotating?",
                "What if the Moon disappeared tonight?",
                "What if all ocean currents stopped?",
            ])
            index = int(time.time() // 86400) % len(topics)
            return str(topics[index])
        raise ValueError(f"Unsupported topic mode: {mode}")

    @staticmethod
    def _qc(video: Path, assets: dict[str, AssetRecord], quality: dict[str, Any], render_error: str) -> dict[str, Any]:
        asset_valid = all(Path(record.svg_path).exists() and record.asset_hash for record in assets.values())
        critic_pass_rate = 0.0
        if quality:
            critic_pass_rate = sum(1 for item in quality.values() if item.get("passed")) / len(quality)
        duration = ffprobe_duration(video) if video.exists() else 0.0
        return {
            "scene_illustration_count": len(assets),
            "asset_files_valid": asset_valid,
            "critic_pass_rate": round(critic_pass_rate, 3),
            "video_exists": video.exists(),
            "video_duration_s": duration,
            "render_error": render_error,
            # A run only passes if a valid MP4 actually exists. A render error is
            # never a pass — that previously reported success with no video.
            "passed": bool(
                asset_valid
                and critic_pass_rate >= 0.999
                and video.exists()
                and duration > 1.0
                and not render_error
            ),
        }

    @staticmethod
    def _metadata(topic: str, script, research) -> dict[str, Any]:
        description = script.hook + "\n\n" + " ".join(str(x) for x in research.limitations[:2])
        return {
            "title": script.title[:95] or topic[:95],
            "description": description[:4500],
            "tags": ["science", "what if", "motion graphics", "scientific animation", *[word.lower() for word in topic.split()[:8]]],
            "privacyStatus": "private",
        }

    @staticmethod
    def _archive(run_dir: Path) -> Path:
        archive = run_dir.with_suffix(".zip")
        with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as output:
            for path in run_dir.rglob("*"):
                if not path.is_file() or "node_modules" in path.parts:
                    continue
                output.write(path, arcname=path.relative_to(run_dir))
        return archive

    @staticmethod
    def _average_profiles(profiles: list[StyleFingerprint]) -> StyleFingerprint:
        keys = [
            "foreground_coverage", "edge_density", "contour_count", "connected_components",
            "palette_size", "white_fraction", "dark_fraction", "blue_fraction", "gray_fraction",
            "red_fraction", "mean_saturation", "mean_value",
        ]
        values = {key: sum(float(getattr(p, key)) for p in profiles) / len(profiles) for key in keys}
        palette = []
        for profile in profiles:
            palette.extend(profile.dominant_palette)
        # Keep the first occurrence of visually distinct colors.
        unique = []
        for color in palette:
            if not any(sum((a-b)**2 for a,b in zip(color, existing)) ** 0.5 < 18 for existing in unique):
                unique.append(color)
        return StyleFingerprint(
            width=profiles[0].width, height=profiles[0].height,
            dominant_palette=unique[:10], **values,
        )

    @staticmethod
    def _default_benchmark_profiles() -> list[StyleFingerprint]:
        # Calibrated from multiple archetypes in the supplied benchmark video:
        # globe/negative-space, disaster wave, icon matrix, and cracked-world finale.
        raw = [
            {"width":1280,"height":720,"foreground_coverage":0.1954,"edge_density":0.0574,"contour_count":80.0,"connected_components":39.0,"palette_size":8.0,"white_fraction":0.8599,"dark_fraction":0.0339,"blue_fraction":0.0002,"gray_fraction":0.0515,"red_fraction":0.0003,"mean_saturation":0.0152,"mean_value":0.9073,"dominant_palette":[[233,238,240],[72,79,85],[29,30,30],[152,153,153],[116,113,115],[252,252,252],[55,59,62],[196,198,199]]},
            {"width":1280,"height":720,"foreground_coverage":0.9909,"edge_density":0.0916,"contour_count":22.0,"connected_components":3.0,"palette_size":8.0,"white_fraction":0.3747,"dark_fraction":0.0254,"blue_fraction":0.2851,"gray_fraction":0.1006,"red_fraction":0.0030,"mean_saturation":0.2214,"mean_value":0.7267,"dominant_palette":[[186,187,190],[46,108,151],[34,36,38],[248,249,249],[67,144,196],[73,82,88],[155,201,225],[133,139,144]]},
            {"width":1280,"height":720,"foreground_coverage":0.1704,"edge_density":0.0762,"contour_count":43.0,"connected_components":36.0,"palette_size":8.0,"white_fraction":0.8303,"dark_fraction":0.0367,"blue_fraction":0.0,"gray_fraction":0.1429,"red_fraction":0.0008,"mean_saturation":0.0066,"mean_value":0.8983,"dominant_palette":[[84,85,88],[146,147,148],[30,31,32],[210,210,211],[169,170,172],[61,63,65],[250,250,250],[118,111,112]]},
            {"width":1280,"height":720,"foreground_coverage":0.2283,"edge_density":0.0740,"contour_count":93.0,"connected_components":37.0,"palette_size":8.0,"white_fraction":0.8052,"dark_fraction":0.0573,"blue_fraction":0.0002,"gray_fraction":0.0637,"red_fraction":0.0055,"mean_saturation":0.0244,"mean_value":0.8789,"dominant_palette":[[231,235,238],[70,77,82],[154,152,153],[194,195,197],[194,42,40],[113,114,116],[32,35,38],[250,250,250]]},
        ]
        return [StyleFingerprint(**item) for item in raw]


In [ ]:
%%writefile /content/scientific_motion_studio_v5/src/scistudio_v5/tests.py
# Cell 21: scistudio_v5/tests.py
from __future__ import annotations

import tempfile
from pathlib import Path
from xml.etree import ElementTree as ET

from PIL import Image, ImageDraw

from .benchmark import BenchmarkAnalyzer
from .illustration import AdvancedRasterVectorizer, EditorialBenchmarkDemo, BenchmarkQualityGate
from .schemas import Fact, Storyboard, VisualIntent
from .svg_compiler import ProgramCompiler, SVGValidator


def run_smoke_tests(reference_video: str | Path | None = None) -> dict[str, bool | float]:
    results: dict[str, bool | float] = {}

    fact = Fact.model_validate({
        "claim": "test", "source_ids": "S01",
        "numeric_values": [{"value": 1670, "unit": "km/h", "context": "equatorial speed"}],
    })
    results["structured_numbers_are_preserved"] = isinstance(fact.numeric_values[0], dict)
    results["source_id_string_is_accepted"] = fact.source_ids == ["S01"]

    board = Storyboard.model_validate({
        "topic": "test",
        "storyboard": {
            "scene_0": {
                "duration": 4,
                "voiceover": "Narration",
                "dashboard": "speedometer",
                "assets": ["transparent magnetic field around a levitating train"],
            }
        },
    })
    results["storyboard_mapping_is_accepted"] = len(board.scenes) == 1
    results["dashboard_string_becomes_visual_intent"] = any("speedometer" in v.description for v in board.scenes[0].visuals)
    results["arbitrary_asset_string_is_accepted"] = any("levitating train" in v.description for v in board.scenes[0].visuals)

    compiler = ProgramCompiler({"paper":"#F7F7F4","ink":"#17212B","blue":"#2D9CDB","cyan":"#56CCF2","gray":"#59636D","muted":"#D8DEE3","warning":"#F2994A","danger":"#EB5757"})
    svg, layers = compiler.compile({
        "canvas": [1000, 1000],
        "nodes": [
            {"op":"cutaway","id":"quantum_battery","params":{"layers":["containment shell","electrode lattice","energy core"]}},
            {"op":"vector_field","id":"energy_flow","params":{"rows":5,"cols":7,"direction":"east"}},
            {"op":"gauge","id":"coherence_meter","params":{"value":.72,"label":"COHERENCE"}},
        ],
    }, prefix="smoke")
    SVGValidator.validate(svg)
    ET.fromstring(svg)
    results["open_program_compiles"] = len(layers) >= 3
    results["no_raster_embed"] = "<image" not in svg.lower() and "data:image" not in svg.lower()

    with tempfile.TemporaryDirectory() as td:
        root = Path(td)
        # Synthetic flat reference validates layered color-contour tracing.
        image = Image.new("RGB", (480, 480), "white")
        draw = ImageDraw.Draw(image)
        draw.polygon([(55,360),(145,185),(242,92),(350,175),(420,360)], fill=(55,145,205), outline=(24,32,40), width=7)
        draw.polygon([(125,360),(210,220),(300,360)], fill=(152,210,238), outline=(24,32,40), width=5)
        draw.rectangle((305,250,395,360), fill=(135,145,152), outline=(24,32,40), width=6)
        source = root / "flat_reference.png"
        image.save(source)
        vectorizer = AdvancedRasterVectorizer(root / "trace")
        traced_svg, traced_layers, meta = vectorizer.vectorize(source, asset_id="trace-test")
        SVGValidator.validate(traced_svg)
        results["layered_raster_vectorization"] = len(traced_layers) >= 4 and meta["path_count"] >= 3

        demo = EditorialBenchmarkDemo(root / "demo").build()
        analyzer = BenchmarkAnalyzer(root / "benchmark")
        if reference_video and Path(reference_video).exists():
            profiles = analyzer.from_video_set(reference_video, sample_count=8)
        else:
            # Compare the demo against itself only to regression-test the benchmark mechanics.
            profiles = [analyzer.from_images([demo.preview_path])]
        report = BenchmarkQualityGate(analyzer, profiles, {
            "style_score_threshold": 0.70,
            "minimum_path_count": 16,
            "minimum_group_count": 7,
        }).evaluate(demo)
        results["benchmark_demo_passes"] = report["passed"]
        results["benchmark_demo_score"] = float(report["score"])

    failed = [name for name, value in results.items() if isinstance(value, bool) and not value]
    if failed:
        raise AssertionError(f"Smoke tests failed: {failed}")
    return results


In [ ]:
#@title 22. Import package and run regression tests
import sys
sys.path.insert(0, str(PACKAGE_ROOT))

from scistudio_v5 import StudioPipeline
from scistudio_v5.schemas import VisualIntent
from scistudio_v5.tests import run_smoke_tests

if RUN_SMOKE_TESTS:
    TEST_RESULTS = run_smoke_tests(STYLE_REFERENCE_VIDEO if Path(STYLE_REFERENCE_VIDEO).exists() else None)
    print("All smoke tests passed:")
    for name, value in TEST_RESULTS.items():
        if isinstance(value, bool):
            print(f"  {'✓' if value else '✗'} {name}")
        else:
            print(f"  • {name}: {value:.4f}")
else:
    print("Smoke tests skipped.")

In [ ]:
#@title 23. Studio configuration
CONFIG = {
    "llm": {
        "provider_order": ["openai", "local", "openrouter"],  # OpenAI -> Qwen local -> OpenRouter (free)
        "vision_provider_order": ["openai", "openrouter"],  # vision control: OpenAI, then OpenRouter
        "openrouter_model": "meta-llama/llama-3.3-70b-instruct:free",
        "openrouter_vision_model": "meta-llama/llama-3.2-11b-vision-instruct:free",
        "gemini_model": "gemini-2.5-flash",
        "gemini_vision_model": "gemini-2.5-flash",
        "openai_model": "gpt-5-mini",
        "openai_vision_model": "gpt-5-mini",
        "gemini_image_model": GEMINI_IMAGE_MODEL or None,
        "openai_image_model": OPENAI_IMAGE_MODEL or None,
        "enable_local_fallback": INSTALL_LOCAL_LLM,
        "local_model": "Qwen/Qwen2.5-3B-Instruct",
        "local_4bit": True,
        "local_max_new_tokens": 3200,
        "temperature": 0.72,
        "enable_local_image_generation": ENABLE_LOCAL_IMAGE_GEN,
        "local_image_model": LOCAL_IMAGE_MODEL or None,
        "local_image_steps": 28,
        # Image generation: FLUX.1 Kontext [pro] via the Black Forest Labs API
        # (fast; no multi-GB model download). Set BFL_API_KEY in Colab Secrets.
        "image_provider": "bfl",
        "bfl_model": "flux-kontext-pro",
        "bfl_aspect_ratio": "16:9",
        "bfl_base_url": "https://api.bfl.ai/v1",
        "bfl_timeout": 180,
        # Local FLUX fallback (needs A100); the API path above is primary.
        "flux_steps": 28,
        "flux_guidance": 2.5,
        "flux_init_image": "",
    },
    "research": {
        "timeout": 25,
        "max_sources": 22,
    },
    "story": {
        "words_per_second": 2.65,
    },
    "benchmark": {
        "reference_video": STYLE_REFERENCE_VIDEO,
        "sample_count": 20,
    },
    "references": {
        "timeout": 30,
        "user_reference_dir": str(ROOT / "user_references"),
    },
    "svg": {
        "enable_reference_trace": True,
        "enable_generated_reference": bool(GEMINI_IMAGE_MODEL or OPENAI_IMAGE_MODEL or (ENABLE_LOCAL_IMAGE_GEN and LOCAL_IMAGE_MODEL)),
        "reference_limit": 10,
        "reference_attempts": 5,
        "style": {
            "paper": "#F7F7F4",
            "ink": "#17212B",
            "blue": "#4390C4",
            "cyan": "#9BC9E1",
            "gray": "#858B90",
            "muted": "#B9BDC2",
            "white": "#FFFFFF",
            "warning": "#F3A53B",
            "danger": "#D73232",
            "dark": "#4B555D",
        },
    },
    "critic": {
        "enabled": True,
        "strict_quality_gate": STRICT_QUALITY_GATE,
        "strict_scene_gate": False,  # False = never abort the run; keep best scene and record it in QC
        "max_refinement_rounds": 5,
        "max_scene_refinement_rounds": 4,
        "asset_style_score_threshold": 0.58,
        "asset_acceptance_threshold": 0.66,
        "scene_style_score_threshold": 0.72,
        "scene_acceptance_threshold": 0.76,
        "scene_minimum_path_count": 18,
        "scene_minimum_group_count": 8,
        "minimum_curved_path_fraction": 0.045,
        "minimum_long_path_fraction": 0.16,
    },
    "script_critic": {
        "enabled": True,
        "acceptance_threshold": 0.6,
        "target_duration_min": 40.0,
        "target_duration_max": 60.0,
        "max_beat_words": 26,
        "max_rounds": 2,
        "strict": False,
    },
    "motion": {},
    "audio": {
        "voice": "en-US-GuyNeural",
        "rate": "+6%",
        "pitch": "+0Hz",
        "fallback_voice": "en-us",
    },
    "render": {
        "width": OUTPUT_WIDTH,
        "height": OUTPUT_HEIGHT,
        "fps": OUTPUT_FPS,
        "crf": 18,
        "install_dependencies": True,
        "typecheck": True,
        "ensure_browser": True,
        "render_timeout": 2400,
        # Use an existing Chrome/Chromium instead of downloading one (remotion.media
        # is often blocked). Empty = auto-detect Playwright/system Chrome.
        "browser_executable": "",
        "remotion_version": "4.0.489",
    },
    "auto_topics": [
        "What if Earth suddenly stopped rotating?",
        "What if the Moon disappeared tonight?",
        "What if all ocean currents stopped?",
    ],
}

print("Configuration ready.")

In [ ]:
#@title 24. Illustration benchmark demo and v4-v5 quality check
if RUN_ILLUSTRATION_DEMO:
    from scistudio_v5.benchmark import BenchmarkAnalyzer
    from scistudio_v5.illustration import EditorialBenchmarkDemo, BenchmarkQualityGate
    from IPython.display import display, Image as IPImage, SVG

    demo_root = ROOT / "illustration_demo"
    analyzer = BenchmarkAnalyzer(demo_root / "benchmark")
    if Path(STYLE_REFERENCE_VIDEO).exists():
        profiles = analyzer.from_video_set(STYLE_REFERENCE_VIDEO, sample_count=20)
    else:
        profiles = StudioPipeline._default_benchmark_profiles()
    demo = EditorialBenchmarkDemo(demo_root).build()
    report = BenchmarkQualityGate(analyzer, profiles, CONFIG["critic"]).evaluate(demo)
    print("Illustration benchmark score:", report["score"])
    print("Passed:", report["passed"])
    print("Complexity:", report["complexity"])
    display(IPImage(filename=demo.preview_path))
else:
    print("Illustration demo disabled.")

In [ ]:
#@title 25. Run the full pipeline
import json

PIPELINE = StudioPipeline(ROOT, CONFIG, SECRETS)

RESULT = PIPELINE.run(
    topic_mode=TOPIC_MODE,
    topic_value=TOPIC_VALUE,
    force_refresh=FORCE_REFRESH,
    render_video=RENDER_VIDEO,
)

print(json.dumps(RESULT, indent=2, ensure_ascii=False))

In [ ]:
#@title 26. Show or download results
from pathlib import Path
from IPython.display import Video, display, Image as IPImage

if RESULT.get("benchmark_demo") and Path(RESULT["benchmark_demo"]).exists():
    print("Benchmark-level SVG demo:")
    display(IPImage(filename=RESULT["benchmark_demo"]))

if RESULT.get("video") and Path(RESULT["video"]).exists():
    display(Video(RESULT["video"], embed=True))
else:
    print("MP4 was not produced. Inspect RESULT['render_error'] and the generated Remotion project.")

try:
    from google.colab import files
    print("Archive ready:", RESULT["archive"])
    # files.download(RESULT["archive"])
except Exception:
    pass

## Quality behavior

- Valid XML is not enough. Concrete scene illustrations must pass benchmark style, illustration-detail, recognizability, scientific-accuracy, semantic-layer, and animation-readiness checks.
- The engine compares each generated scene to the **closest** reference-frame archetype, not an averaged frame. This lets a sparse globe scene and a dense disaster scene both pass without copying a specific composition.
- If a scene remains icon-like after refinement, strict mode raises `IllustrationQualityError` and blocks final rendering. It never silently publishes a low-quality generic fallback.
- Add high-quality user references to `/content/scientific_motion_studio_v5/user_references`. Licensed Wikimedia references are searched automatically. Optional generated-reference providers can be enabled by setting an image model identifier.
- Re-run only the full-pipeline cell with `FORCE_REFRESH = False` after fixing a network, TTS, or renderer issue; cached research, scripts, references, SVGs, audio, and critiques are reused.

## v5-final — providers, robustness, capabilities (all tested)

**Providers (this revision):**
- LLM + vision priority: **OpenAI → local Qwen → OpenRouter (free)**. Set `OPENAI_API_KEY`, optionally `OPENROUTER_API_KEY`, in Colab Secrets.
- Image generation: **FLUX.1 Kontext [pro] via the Black Forest Labs API** (`BFL_API_KEY`) — submit→poll→download with a content-addressed PNG cache (identical prompt+seed is never re-requested). No multi-GB model download.

**Robustness — the pipeline no longer stops mid-run on LLM shape mismatches or a single weak scene:**
- Systemic `OpenModel` coercion: a dict/scalar where a list is declared → list; a list/dict where a string is declared → string. `Any` fields stay free. Fixes the repeated `ValidationError`s (research `limitations`, `QualityReport.repair_plan`, etc.).
- `validate_assignment` off + idempotent `normalize_scene`: fixed the storyboard `'dict' has no attribute intent_id` crash (a validator side-effect that re-fired on every attribute set).
- Benchmark: an empty reference-video path no longer resolves to a directory and crashes ffprobe.
- Scene quality gate is **non-fatal by default** (`critic.strict_scene_gate=False`): a scene below the benchmark is refined, then the best version is kept and its score recorded in QC — the run always completes. Set `strict_scene_gate=True` to restore hard-fail.
- Verified: the **entire pipeline runs end-to-end offline** (no keys → fallbacks, render skipped) through all 13 stages without raising.

---
### Earlier fixes carried in this notebook (all tested)

**Reliability fixes** (each reproduced and re-tested):
1. Python 3.11 portability — removed 3.12-only f-strings in `svg_compiler.py`.
2. `text`/`label` op no longer emits duplicate-attribute (invalid) SVG.
3. Honest non-vision quality gate (falls back to structural/style benchmark instead of hard-failing).
4. QC no longer reports success when the render failed (requires a real MP4).
5. Headline no longer collides with the top status panels.
6. `**ns0:` namespace bug** — inlined SVG rendered in CairoSVG but not in the browser (Remotion), so hero art was silently missing from the video. Now emits un-prefixed SVG; heroes render.

**Remotion end-to-end (now verified producing a real 1080×1920 H.264+AAC MP4):**
7. Auto-detect an existing Chrome/Chromium (`--browser-executable`) so the blocked Chrome-Headless-Shell download is skipped; prefer `chrome-headless-shell`.
8. Pinned Remotion/React/TypeScript (`latest` pulled TypeScript 7.0.x which crashes Remotion's bundler).

**New capabilities:**
- **Script retention-critic** (`script_critic.py`) — the narration counterpart to the visual critic: scores hook / information-gain / pacing / escalation / payoff, rewrites weak scripts, and runs in the pipeline after the script stage. Verified to separate weak from strong scripts.
- **Designed sound bed** — minor-triad pad + ~0.9 Hz sub pulse + air layer + scene-synced whooshes (procedural FFmpeg, no assets).
- **FLUX.1 [Kontext-dev] image generation** wired into the router (text-to-image, or Kontext instruction-edit of a seed render) and **OpenAI vision control** via `vision_provider_order`.

> Live paths still need runtime resources: OpenAI key (text + vision), an A100-class GPU for FLUX, and internet for research/Edge-TTS. The deterministic media path (SVG → scene → Remotion MP4) and the script critic were tested offline in this build.
